# Ứng dụng 1 · Phân tích giỏ hàng với dữ liệu Retail

Ứng dụng phân tích 88.162 giỏ hàng để tìm những mặt hàng có thể gợi ý mua kèm. Apriori tìm các nhóm thường xuất hiện trong cùng một giỏ, rồi từ mỗi nhóm ta lập các luật như `X ⇒ Y`. Với từng luật, ta xem bao nhiêu phần trăm giỏ có X cũng chứa Y, rồi so với tỷ lệ mua Y trong toàn bộ dữ liệu. Bảng kết quả xếp hạng 10 luật có lift cao nhất.

Trên Colab, chọn **Python 3 (CPU)** rồi chạy các ô từ trên xuống. Mã Apriori và dữ liệu đã có sẵn trong notebook.

## Chuẩn bị mã nguồn và dữ liệu

Các ô dưới chuẩn bị Julia, ghi mã nguồn và giải nén dữ liệu đã kiểm tra SHA-256 vào thư mục riêng của phiên chạy.

In [ ]:
# @title Chuẩn bị Julia và thư mục làm việc
from pathlib import Path
import csv
import hashlib
import platform
import shutil
import subprocess
import tempfile
import urllib.request

base = Path("/content") if Path("/content").is_dir() else Path.cwd()
cache = base / "apriori-colab-cache"
cache.mkdir(exist_ok=True)
project = Path(tempfile.mkdtemp(prefix="apriori-retail-", dir=base))

def verified_download(url, dest, expected_sha256):
    def digest(path):
        h = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    if dest.is_file() and digest(dest) == expected_sha256:
        return dest
    with tempfile.NamedTemporaryFile(dir=dest.parent, delete=False) as temp:
        staging = Path(temp.name)
    try:
        urllib.request.urlretrieve(url, staging)
        if digest(staging) != expected_sha256:
            raise ValueError(f"Tệp tải về không khớp SHA-256: {dest.name}")
        staging.replace(dest)
    finally:
        staging.unlink(missing_ok=True)
    return dest

julia = shutil.which("julia")
if not julia:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("Hãy chọn Python 3 trên Colab hoặc thêm Julia 1.9+ vào PATH.")
    julia = str(cache / "julia-1.11.4" / "bin" / "julia")
    if not Path(julia).is_file():
        archive = verified_download(
            "https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz",
            cache / "julia-1.11.4-linux-x86_64.tar.gz",
            "fb3d3c5fccef82158a70677c0044ac5ae40410eceb0604cdc8e643eeff21df8d",
        )
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(cache)], check=True)
print(subprocess.check_output([julia, "--version"], text=True).strip())
print("Thư mục kết quả của phiên:", project)

def run_julia(*arguments):
    result = subprocess.run(
        [julia, "--startup-file=no", "--project=.", *map(str, arguments)],
        cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    )
    print(result.stdout, end="")
    result.check_returncode()

In [ ]:
# @title Mã nguồn Julia và bộ kiểm thử
# Phiên bản mã nguồn trong repository: e48110f6978f226a80329dbcfeef07e1c519c530
source_files = {
    "Project.toml": r'''name = "FrequentItemsetMining"
uuid = "284f3ddb-453b-4c46-9ce6-1e85fd28ca9d"
version = "0.1.0"

[deps]
Dates = "ade2ca70-3891-5945-98fb-dc099432e06a"
DelimitedFiles = "8bb1440f-4735-579b-a4ab-409b98df4dab"
Printf = "de0858da-6303-5e67-8744-51eddeeeb8d7"
SHA = "ea8e919c-243c-51af-8825-aaa63cd721ce"
TOML = "fa267f1f-6049-4f14-aa54-33bafae1ed76"
UUIDs = "cf7118a7-6976-5b1a-9a39-7adc72f591a4"

[compat]
julia = "1.9"

[extras]
Test = "8dfed614-e22c-5e08-85e1-65c5234f0b40"
Random = "9a3f8284-a2c9-5f02-9a11-845980a1fd5c"

[targets]
test = ["Test", "Random"]
''',
    "src/apriori.jl": r'''"""Statistics for a level actually counted (Figure 1 of the paper)."""
struct LevelStats
    size::Int
    candidates::Int
    frequent::Int
end

"""Measured monotonic duration for one mining phase; level totals are nested."""
struct PhaseTiming
    phase::Symbol
    size::Int
    elapsed_ns::UInt64
    candidates::Int
    frequent::Int
end

"""All frequent itemsets and integer transaction support counts."""
struct MiningResult
    # Variable-length keys occur only at the result boundary. Counting uses
    # concrete NTuple{K,Int} vectors and Int counters inside each level.
    itemsets::Dict{Tuple{Vararg{Int}},Int}
    transaction_count::Int
    min_support_count::Int
    levels::Vector{LevelStats}
    timings::Vector{PhaseTiming}
    mining_ns::UInt64
end

MiningResult(itemsets, transaction_count, min_support_count, levels) =
    MiningResult(itemsets, transaction_count, min_support_count, levels, PhaseTiming[], UInt64(0))

record_phase!(::Nothing, phase, size, elapsed_ns, candidates, frequent) = nothing
function record_phase!(timings::Vector{PhaseTiming}, phase, size, elapsed_ns, candidates, frequent)
    push!(timings, PhaseTiming(phase, size, elapsed_ns, candidates, frequent))
    return nothing
end

function normalize_transactions(transactions)
    database = Vector{Int}[]
    for (row_number, row) in enumerate(transactions)
        items = Int[]
        for item in row
            if item isa Bool || !(item isa Integer) || item < 0 || item > typemax(Int)
                throw(ArgumentError("transaction $row_number: items must be nonnegative Int values"))
            end
            push!(items, Int(item))
        end
        sort!(items)
        unique!(items)
        push!(database, items)
    end
    return database
end

"""Parse decimal/scientific notation without an intermediate binary float."""
function decimal_ratio(text::AbstractString)
    parsed = match(r"^([+-]?)(\d*)(?:\.(\d*))?(?:[eE]([+-]?\d+))?$", strip(text))
    parsed === nothing && throw(ArgumentError("min_support must be a finite decimal ratio"))
    sign, whole, fraction, exponent = parsed.captures
    fraction = isnothing(fraction) ? "" : fraction
    isempty(whole) && isempty(fraction) && throw(ArgumentError("min_support has no digits"))
    power = isnothing(exponent) ? 0 : tryparse(Int, exponent)
    (isnothing(power) || abs(big(power)) > 10000) &&
        throw(ArgumentError("min_support exponent is outside the supported range"))
    numerator = parse(BigInt, (isempty(whole) ? "0" : whole) * fraction)
    sign == "-" && (numerator = -numerator)
    scale = length(fraction) - power
    return scale >= 0 ? numerator // big(10)^scale : (numerator * big(10)^(-scale)) // big(1)
end

function support_ratio(value::Rational)
    return big(numerator(value)) // big(denominator(value))
end
support_ratio(value::AbstractString) = decimal_ratio(value)
support_ratio(value::Integer) = big(value) // big(1)
function support_ratio(value::AbstractFloat)
    isfinite(value) || throw(ArgumentError("min_support must be finite"))
    return decimal_ratio(string(value))
end
support_ratio(value) = throw(ArgumentError("min_support must be a decimal string, Integer, Rational or AbstractFloat"))

function support_threshold(n::Int, min_support, min_count)
    isnothing(min_support) == isnothing(min_count) &&
        throw(ArgumentError("provide exactly one of min_support or min_count"))
    if !isnothing(min_count)
        if min_count isa Bool || !(min_count isa Integer) || !(1 <= min_count <= typemax(Int))
            throw(ArgumentError("min_count must be an integer in 1:typemax(Int)"))
        end
        return Int(min_count)
    end
    min_support isa Bool && throw(ArgumentError("min_support cannot be Bool"))
    ratio = support_ratio(min_support)
    0 < ratio <= 1 || throw(ArgumentError("min_support must be in (0, 1]"))
    return Int(cld(numerator(ratio) * n, denominator(ratio)))
end

"""§2.1.1: prefix join followed by membership pruning of every K-subset."""
function apriori_gen(previous::Vector{NTuple{K,Int}}, trace::AbstractAprioriTrace=NoTrace()) where K
    K >= 1 || throw(ArgumentError("previous itemsets must not be empty"))
    ordered = sort!(unique(previous))
    known = Set(ordered)
    candidates = NTuple{K+1,Int}[]
    for i in eachindex(ordered)
        left = ordered[i]
        # Index iteration avoids copying every remaining tail of the list.
        for j in (i + 1):length(ordered)
            right = ordered[j]
            all(left[p] == right[p] for p in 1:(K - 1)) || break
            candidate = (left..., right[K])
            record!(trace, :join, left, right, candidate)
            valid = true
            for removed in 1:(K + 1)
                subset = ntuple(p -> candidate[p + (p >= removed)], Val(K))
                if !(subset in known)
                    record!(trace, :prune, candidate, subset, K)
                    valid = false
                    break
                end
            end
            if valid
                push!(candidates, candidate)
                record!(trace, :accepted, candidate)
            end
        end
    end
    return candidates
end


"""
    foreach_candidate_batch(callback, previous, batch_size, trace)

Generate Apriori candidates with the same prefix-join and all-subsets pruning
rules as `apriori_gen`, but deliver bounded batches to `callback`. The callback
must finish counting/filtering a batch before returning; candidate tuples are
not retained by this function after the callback.
"""
function foreach_candidate_batch(callback::F, previous::Vector{NTuple{K,Int}},
                                 batch_size::Int, trace::AbstractAprioriTrace) where {K,F}
    K >= 1 || throw(ArgumentError("previous itemsets must not be empty"))
    batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    generation_started = time_ns()
    ordered = issorted(previous) && allunique(previous) ? previous : sort!(unique(previous))
    known = Set(ordered)
    batch = NTuple{K+1,Int}[]
    total = 0
    generation_ns = UInt64(0)

    for i in eachindex(ordered)
        left = ordered[i]
        for j in (i + 1):length(ordered)
            right = ordered[j]
            all(left[p] == right[p] for p in 1:(K - 1)) || break
            candidate = (left..., right[K])
            record!(trace, :join, left, right, candidate)
            valid = true
            for removed in 1:(K + 1)
                subset = ntuple(p -> candidate[p + (p >= removed)], Val(K))
                if !(subset in known)
                    record!(trace, :prune, candidate, subset, K)
                    valid = false
                    break
                end
            end
            if valid
                push!(batch, candidate)
                total += 1
                record!(trace, :accepted, candidate)
                if length(batch) == batch_size
                    generation_ns += time_ns() - generation_started
                    callback(batch)
                    empty!(batch)
                    generation_started = time_ns()
                end
            end
        end
    end

    generation_ns += time_ns() - generation_started
    isempty(batch) || callback(batch)
    return (; total, generation_ns)
end

function count_candidates(candidates::Vector{NTuple{K,Int}}, database::Vector{Vector{Int}},
                          ::Val{:scan}, trace::AbstractAprioriTrace=NoTrace(),
                          timings::Union{Nothing,Vector{PhaseTiming}}=nothing) where K
    # Scan has no reusable counter structure to build.
    record_phase!(timings, :build_counter, K, UInt64(0), length(candidates), 0)
    started = time_ns()
    counts = zeros(Int, length(candidates))
    for (tid, transaction) in enumerate(database)
        record!(trace, :transaction, K, tid, transaction)
        present = Set(transaction)
        for id in eachindex(candidates)
            if all(item in present for item in candidates[id])
                counts[id] += 1
                record!(trace, :increment, tid, candidates[id], counts[id])
            end
        end
    end
    record_phase!(timings, :count_support, K, time_ns() - started, length(candidates), 0)
    return counts
end

function count_candidates(candidates::Vector{NTuple{K,Int}}, database::Vector{Vector{Int}},
                          ::Val{:hash_tree}, trace::AbstractAprioriTrace=NoTrace(),
                          timings::Union{Nothing,Vector{PhaseTiming}}=nothing) where K
    started = time_ns()
    tree = HashTree(candidates)
    workspace = HashQueryWorkspace(tree)
    counts = zeros(Int, length(candidates))
    record_phase!(timings, :build_counter, K, time_ns() - started, length(candidates), 0)
    started = time_ns()
    for (tid, transaction) in enumerate(database)
        record!(trace, :transaction, K, tid, transaction)
        for id in ordered_matches!(trace, matching_ids!(workspace, tree, transaction))
            counts[id] += 1
            record!(trace, :increment, tid, candidates[id], counts[id])
        end
    end
    record_phase!(timings, :count_support, K, time_ns() - started, length(candidates), 0)
    return counts
end

function next_level(previous::Vector{NTuple{K,Int}}, database, threshold, mode,
                    trace::AbstractAprioriTrace=NoTrace(),
                    timings::Union{Nothing,Vector{PhaseTiming}}=nothing;
                    candidate_batch_size::Int=10_000) where K
    record!(trace, :level, K+1)
    frequent = NTuple{K+1,Int}[]
    supports = Int[]
    generation_phase_index = Ref(0)

    candidate_stats = foreach_candidate_batch(previous, candidate_batch_size, trace) do candidates
        if !isnothing(timings) && iszero(generation_phase_index[])
            push!(timings, PhaseTiming(:generate_candidates, K+1, UInt64(0), 0, 0))
            generation_phase_index[] = length(timings)
        end
        counts = count_candidates(candidates, database, mode, trace, timings)
        filter_started = time_ns()
        frequent_in_batch = 0
        for id in eachindex(candidates)
            record!(trace, :decision, candidates[id], counts[id], threshold)
            if counts[id] >= threshold
                push!(frequent, candidates[id])
                push!(supports, counts[id])
                frequent_in_batch += 1
            end
        end
        record_phase!(timings, :filter, K+1, time_ns() - filter_started,
                      length(candidates), frequent_in_batch)
    end

    candidate_total = candidate_stats.total
    explain_frequent!(trace, K+1, frequent, supports)
    if !isnothing(timings)
        if iszero(generation_phase_index[])
            record_phase!(timings, :generate_candidates, K+1,
                          candidate_stats.generation_ns, candidate_total, 0)
        else
            timings[generation_phase_index[]] =
                PhaseTiming(:generate_candidates, K+1, candidate_stats.generation_ns,
                            candidate_total, 0)
        end
    end
    record!(trace, :candidates, K+1, candidate_total)
    return frequent, supports, candidate_total
end

counter_mode(counter) = counter in (:scan, "scan") ? Val(:scan) :
                        counter in (:hash_tree, "hash-tree") ? Val(:hash_tree) :
                        throw(ArgumentError("counter must be :scan or :hash_tree"))

"""
    mine(transactions; min_support=nothing, min_count=nothing, counter=:hash_tree, explain=nothing)

Plain Apriori (Figure 1): join/prune, scan original transactions, filter L_k.
Pass exactly one support threshold. Empty API rows still contribute to N.
All data is materialized in RAM; §2.1.3 external-memory handling is deferred.
"""
function mine(transactions; min_support=nothing, min_count=nothing, counter=:hash_tree,
              explain=nothing, candidate_batch_size::Int=10_000)
    mining_started = time_ns()
    candidate_batch_size >= 1 ||
        throw(ArgumentError("candidate_batch_size must be >= 1"))
    mode = counter_mode(counter)
    trace = isnothing(explain) ? NoTrace() : explain
    trace isa AbstractAprioriTrace || throw(ArgumentError("explain must be ExplainTrace or nothing"))
    timings = PhaseTiming[]
    started = time_ns()
    database = normalize_transactions(transactions)
    record_phase!(timings, :normalize, 0, time_ns() - started, 0, 0)
    started = time_ns()
    threshold = support_threshold(length(database), min_support, min_count)
    record_phase!(timings, :threshold, 0, time_ns() - started, 0, 0)
    answer = Dict{Tuple{Vararg{Int}},Int}()
    levels = LevelStats[]
    mode_name = mode isa Val{:scan} ? "scan" : "hash-tree"
    record!(trace, :begin, length(database), threshold, mode_name)
    explain_input!(trace, database, min_support, threshold)
    if isempty(database)
        record!(trace, :stop, "CSDL rỗng.")
        finish_explanation!(trace, 0)
        return MiningResult(answer, 0, threshold, levels, timings, time_ns() - mining_started)
    end

    # Figure 1, line 1: count singletons, then select L_1.
    level_started = time_ns()
    started = time_ns()
    singletons = Dict{Int,Int}()
    record!(trace, :level, 1)
    for (tid, transaction) in enumerate(database)
        record!(trace, :transaction, 1, tid, transaction)
        for item in transaction
            singletons[item] = get(singletons, item, 0) + 1
            record!(trace, :increment, tid, (item,), singletons[item])
        end
    end
    record_phase!(timings, :singleton_count, 1, time_ns() - started, length(singletons), 0)
    started = time_ns()
    explain_singletons!(trace, singletons, threshold)
    previous = NTuple{1,Int}[(item,) for (item, count) in singletons if count >= threshold]
    sort!(previous)
    record_phase!(timings, :filter, 1, time_ns() - started, length(singletons), length(previous))
    for itemset in previous
        answer[itemset] = singletons[itemset[1]]
    end
    push!(levels, LevelStats(1, length(singletons), length(previous)))
    # Support values are already in the result; no extra collection when off.
    explain_frequent!(trace, 1, previous, (singletons[itemset[1]] for itemset in previous))
    isempty(previous) && record!(trace, :stop, "L_1 rỗng, không có item frequent.")
    record_phase!(timings, :level_total, 1, time_ns() - level_started, length(singletons), length(previous))

    # next_level is a function barrier: its loops specialize for each K.
    size = 2
    while !isempty(previous)
        level_started = time_ns()
        current, supports, candidate_count = next_level(previous, database, threshold, mode, trace, timings; candidate_batch_size)
        if iszero(candidate_count)
            record!(trace, :stop, "C_$size rỗng, không còn ứng viên để đếm.")
            record_phase!(timings, :level_total, size, time_ns() - level_started, 0, 0)
            break
        end
        push!(levels, LevelStats(size, candidate_count, length(current)))
        for id in eachindex(current)
            answer[current[id]] = supports[id]
        end
        previous = current
        isempty(previous) && record!(trace, :stop, "L_$size rỗng, mọi ứng viên đều dưới ngưỡng.")
        record_phase!(timings, :level_total, size, time_ns() - level_started, candidate_count, length(current))
        size += 1
    end
    finish_explanation!(trace, length(answer))
    return MiningResult(answer, length(database), threshold, levels, timings, time_ns() - mining_started)
end
''',
    "src/AssociationRules.jl": r'''"""
    AssociationRules

Sinh luật kết hợp `X => Y` từ các frequent itemset đã khai phá bằng Apriori.

Đầu vào là file itemsets theo định dạng SPMF (`1 3 5 #SUP: 42`) và tổng số
transaction `N`. Với mỗi itemset phổ biến `Z` và mỗi cách tách `Z = X ∪ Y`
(X, Y khác rỗng, rời nhau):

    support(X ⇒ Y)    = sup(Z) / N
    confidence(X ⇒ Y) = sup(Z) / sup(X)
    lift(X ⇒ Y)       = confidence / (sup(Y) / N) = sup(Z)·N / (sup(X)·sup(Y))

Do tính chất Apriori, mọi tập con của `Z` đều phổ biến nên `sup(X)` và `sup(Y)`
luôn có sẵn trong file itemsets; không cần quét lại dữ liệu.

Module chỉ dùng thư viện chuẩn của Julia và không có trạng thái toàn cục.
Kết quả được sắp theo thứ tự toàn phần (lift ↓, confidence ↓, count ↓, rồi
antecedent/consequent tăng dần) nên chạy lại luôn cho cùng một đầu ra.
"""
module AssociationRules

export Rule, read_itemsets, count_transactions, generate_rules, write_rules_csv

"Số item tối đa của một itemset được tách luật (2^k phép tách)."
const MAX_ITEMSET_SIZE = 24
const SUP_MARK = "#SUP:"
const EPS = 1e-12

"Một luật kết hợp `antecedent ⇒ consequent` cùng các chỉ số đo."
struct Rule
    antecedent::Vector{Int}
    consequent::Vector{Int}
    count::Int          # sup(X ∪ Y), số transaction chứa cả hai vế
    support::Float64
    confidence::Float64
    lift::Float64
end

"""
    read_itemsets(path) -> Dict{Vector{Int},Int}

Đọc file itemsets SPMF. Khóa là vector item đã sắp tăng, giá trị là support count.
Bỏ qua dòng trống và dòng metadata bắt đầu bằng `@`, `%` hoặc `#`.
"""
function read_itemsets(path::AbstractString)
    counts = Dict{Vector{Int},Int}()
    open(path, "r") do source
        for (ln, line) in enumerate(eachline(source))
            s = strip(line)
            (isempty(s) || s[1] in ('@', '%', '#')) && continue
            pos = findfirst(SUP_MARK, s)
            pos === nothing && throw(ArgumentError("$path:$ln: thiếu '$SUP_MARK'"))
            left = strip(s[1:prevind(s, first(pos))])
            right = strip(s[(last(pos) + 1):end])
            items = try
                sort!(parse.(Int, split(left)))
            catch
                throw(ArgumentError("$path:$ln: item không phải số nguyên"))
            end
            isempty(items) && continue
            counts[items] = try
                parse(Int, right)
            catch
                throw(ArgumentError("$path:$ln: support count không hợp lệ"))
            end
        end
    end
    return counts
end

"""
    count_transactions(path) -> Int

Đếm số transaction hợp lệ của file dữ liệu SPMF (bỏ dòng trống và metadata).
"""
function count_transactions(path::AbstractString)
    n = 0
    for line in eachline(path)
        s = strip(line)
        (isempty(s) || s[1] in ('#', '%', '@')) && continue
        n += 1
    end
    return n
end

function _rule_isless(a::Rule, b::Rule)
    a.lift != b.lift && return a.lift > b.lift
    a.confidence != b.confidence && return a.confidence > b.confidence
    a.count != b.count && return a.count > b.count
    a.antecedent != b.antecedent && return a.antecedent < b.antecedent
    return a.consequent < b.consequent
end

function _rules_from_itemset!(rules::Vector{Rule}, counts::Dict{Vector{Int},Int},
                              items::Vector{Int}, supxy::Int, ntrans::Int,
                              minconf::Float64, minlift::Float64)
    k = length(items)
    full = (1 << k) - 1
    @inbounds for mask in 1:(full - 1)      # mọi tập con khác rỗng và khác toàn bộ
        ant = Int[]
        con = Int[]
        for j in 1:k
            if (mask >> (j - 1)) & 1 == 1
                push!(ant, items[j])
            else
                push!(con, items[j])
            end
        end
        supx = get(counts, ant, 0)
        supy = get(counts, con, 0)
        if supx == 0 || supy == 0
            throw(ArgumentError("thiếu support của tập con $(supx == 0 ? ant : con) " *
                                "trong file itemsets (file không đầy đủ?)"))
        end
        conf = supxy / supx
        conf + EPS < minconf && continue
        lift = Float64(supxy * ntrans) / Float64(supx * supy)
        lift + EPS < minlift && continue
        push!(rules, Rule(ant, con, supxy, supxy / ntrans, conf, lift))
    end
    return rules
end

"""
    generate_rules(counts, ntrans; minconf=0.5, minlift=0.0) -> Vector{Rule}

Sinh mọi luật có confidence ≥ `minconf` và lift ≥ `minlift`, đã sắp xếp
(lift giảm dần, hòa thì confidence, count, rồi thứ tự từ điển).
"""
function generate_rules(counts::Dict{Vector{Int},Int}, ntrans::Integer;
                        minconf::Real=0.5, minlift::Real=0.0)
    ntrans > 0 || throw(ArgumentError("ntrans phải > 0"))
    0 <= minconf <= 1 || throw(ArgumentError("minconf phải nằm trong [0, 1]"))
    minlift >= 0 || throw(ArgumentError("minlift phải >= 0"))
    rules = Rule[]
    for (items, supxy) in counts
        length(items) < 2 && continue
        if length(items) > MAX_ITEMSET_SIZE
            throw(ArgumentError("itemset có $(length(items)) item, vượt giới hạn " *
                                "$MAX_ITEMSET_SIZE; hãy tăng minsup"))
        end
        _rules_from_itemset!(rules, counts, items, supxy, Int(ntrans),
                             Float64(minconf), Float64(minlift))
    end
    sort!(rules; lt=_rule_isless)
    return rules
end

"Ghi luật ra CSV: antecedent, consequent (item cách nhau bằng khoảng trắng), count, support, confidence, lift."
function write_rules_csv(path::AbstractString, rules::Vector{Rule})
    dir = dirname(path)
    isempty(dir) || mkpath(dir)
    open(path, "w") do io
        println(io, "antecedent,consequent,count,support,confidence,lift")
        for r in rules
            println(io, join(r.antecedent, ' '), ',', join(r.consequent, ' '), ',',
                    r.count, ',', r.support, ',', r.confidence, ',', r.lift)
        end
    end
    return path
end

end # module
''',
    "src/cli.jl": r'''const CLI_HELP = """
Plain Apriori from the original paper (Julia, no FIM library).
Usage: julia --project=. bin/apriori.jl --input INPUT [--output OUTPUT]
       (--minsup RATIO | --min-count COUNT) [--counter scan|hash-tree] [--trace]
       [--explain | --explain-file PATH] [--explain-limit N]
       [--report-dir PATH] [--warmup N] [--quiet]

--minsup     Minimum support ratio in (0, 1], e.g. 0.5
--min-count  Minimum absolute transaction count >= 1
--counter    scan or hash-tree (default: hash-tree)
--trace      Print per-level C_k / L_k statistics to stderr
--explain    Explain the actual Apriori steps in Vietnamese on stderr
--explain-file PATH  Write the explanation as UTF-8 to a separate file
--explain-limit N    Maximum detail lines (positive integer; default: 5000)
--report-dir PATH   New directory for this run's immutable report bundle
--warmup N          Untraced mining runs before the measured run (default: 0)
--quiet             Suppress success messages; reports are still written
--help       Show this help
Reports default to output/runs/<dataset>-N/ or <output-parent>/runs/<dataset>-N/.
Without --output, itemsets.txt is written directly inside the report directory.
Output parent must exist unless it is inside the new report directory.
"""

function parse_cli(args)
    values = Dict{String,String}()
    trace = false
    explain = false
    quiet = false
    index = 1
    while index <= length(args)
        option = args[index]
        if option == "--trace"
            trace && throw(ArgumentError("duplicate --trace"))
            trace = true
            index += 1
        elseif option == "--explain"
            explain && throw(ArgumentError("duplicate --explain"))
            explain = true
            index += 1
        elseif option == "--quiet"
            quiet && throw(ArgumentError("duplicate --quiet"))
            quiet = true
            index += 1
        elseif option in ("--input", "--output", "--minsup", "--min-count", "--counter",
                          "--explain-file", "--explain-limit", "--report-dir", "--warmup")
            haskey(values, option) && throw(ArgumentError("duplicate $option"))
            index < length(args) || throw(ArgumentError("missing value for $option"))
            values[option] = args[index + 1]
            index += 2
        else
            throw(ArgumentError("unknown option: $option"))
        end
    end
    haskey(values, "--input") || throw(ArgumentError("required option: --input"))
    haskey(values, "--minsup") == haskey(values, "--min-count") &&
        throw(ArgumentError("provide exactly one of --minsup or --min-count"))
    return values, trace, explain, quiet
end

function cli_same_path(left, right)
    a, b = cli_resolved_path(left), cli_resolved_path(right)
    Sys.iswindows() && ((a, b) = (lowercase(a), lowercase(b)))
    return a == b || (ispath(left) && ispath(right) && samefile(left, right))
end

function cli_inside(path, directory)
    candidate, parent = cli_resolved_path(path), cli_resolved_path(directory)
    Sys.iswindows() && ((candidate, parent) = (lowercase(candidate), lowercase(parent)))
    relative = relpath(candidate, parent)
    return relative != "." && !isabspath(relative) &&
           first(splitpath(relative)) != ".."
end

function check_cli_report_paths(report_dir, input, output, explanation_path)
    destinations = isnothing(explanation_path) ? [output] : [output, explanation_path]
    for destination in destinations
        islink(destination) && !ispath(destination) &&
            throw(ArgumentError("destination is a dangling symlink: $destination"))
        cli_same_path(destination, report_dir) &&
            throw(ArgumentError("destination must be a file, not the report directory"))
        cli_inside(report_dir,destination) &&
            throw(ArgumentError("report directory cannot be inside a destination file path"))
        isdir(destination) && throw(ArgumentError("destination is a directory: $destination"))
        if !isdir(dirname(abspath(destination))) && !cli_inside(destination, report_dir)
            throw(ArgumentError("destination parent directory must already exist: $destination"))
        end
    end
    paths = isnothing(explanation_path) ? [input, output] : [input, output, explanation_path]
    for name in ("report.txt", "timings.csv", "metadata.toml", "itemsets.txt", "trace.txt")
        artifact = joinpath(report_dir, name)
        for (index, path) in enumerate(paths)
            allowed = name == "itemsets.txt" && index == 2 ||
                      name == "trace.txt" && index == 3
            if (cli_same_path(path, artifact) && !allowed) || cli_inside(path, artifact)
                throw(ArgumentError("path conflicts with reserved report artifact $name"))
            end
        end
    end
    # Check ancestors without creating them, including a file where a directory
    # would be needed. The final directory itself is claimed with mkdir below.
    ancestor = dirname(abspath(report_dir))
    while !ispath(ancestor)
        ancestor = dirname(ancestor)
    end
    isdir(ancestor) || throw(ArgumentError("report parent is not a directory: $ancestor"))
    return nothing
end

# Resolve existing ancestors too, so an output that does not yet exist is
# still compared correctly when its parent is reached through a symlink.
function cli_resolved_path(path)
    absolute = normpath(abspath(path))
    ispath(absolute) && return realpath(absolute)
    parent = dirname(absolute)
    parent == absolute && return absolute
    return joinpath(cli_resolved_path(parent), basename(absolute))
end

function check_cli_paths(paths)
    resolved = cli_resolved_path.(paths)
    Sys.iswindows() && (resolved = lowercase.(resolved))
    for left in eachindex(paths), right in (left + 1):length(paths)
        same_path = resolved[left] == resolved[right]
        same_existing = ispath(paths[left]) && ispath(paths[right]) &&
                        samefile(paths[left], paths[right])
        (same_path || same_existing) &&
            throw(ArgumentError("input, output and explanation must not refer to the same file"))
        (cli_inside(paths[left], paths[right]) || cli_inside(paths[right], paths[left])) &&
            throw(ArgumentError("a file destination cannot also be another path's parent directory"))
    end
    return nothing
end

function main(args=ARGS; out=stdout, err=stderr)
    if args == ["--help"]
        print(out, CLI_HELP)
        return 0
    end
    try
        values, trace, explain, quiet = parse_cli(args)
        cli_start = time_ns()
        started = Dates.now(Dates.UTC)
        run_id = Dates.format(started, dateformat"yyyymmddTHHMMSSsss") * "_" * string(uuid4())
        input = abspath(values["--input"])
        requested_output = haskey(values, "--output") ? abspath(values["--output"]) : nothing
        explanation_path = haskey(values, "--explain-file") ? abspath(values["--explain-file"]) : nothing
        output_for(dir) = something(requested_output, joinpath(dir, "itemsets.txt"))
        function validate_paths(dir)
            output = output_for(dir)
            paths = [input, output]
            !isnothing(explanation_path) && push!(paths, explanation_path)
            check_cli_paths(paths)
            check_cli_report_paths(dir, input, output, explanation_path)
        end
        min_count = nothing
        if haskey(values, "--min-count")
            min_count = tryparse(Int, values["--min-count"])
            isnothing(min_count) && throw(ArgumentError("--min-count must be an integer"))
        end
        limit = tryparse(Int, get(values, "--explain-limit", "5000"))
        isnothing(limit) && throw(ArgumentError("--explain-limit must be a positive integer"))
        explanation = ExplainTrace(err; limit)
        warmup = tryparse(Int, get(values, "--warmup", "0"))
        (isnothing(warmup) || warmup < 0) &&
            throw(ArgumentError("--warmup must be a nonnegative integer"))
        min_support = get(values, "--minsup", nothing)
        counter = get(values, "--counter", "hash-tree")
        # Validate all input and mining options before opening either destination.
        read_start = time_ns()
        # Parse and fingerprint the same immutable byte snapshot. A file
        # changed after this read cannot be relabelled as the mined input.
        input_bytes=read(input)
        input_sha256=bytes2hex(sha256(input_bytes))
        rows = read_spmf(IOBuffer(input_bytes))
        read_ns = time_ns() - read_start
        support_threshold(length(rows), min_support, min_count)
        counter_mode(counter)
        source_sha256=implementation_fingerprint()
        # All validation has finished before any destination is created/opened.
        if haskey(values, "--report-dir")
            report_dir = abspath(values["--report-dir"])
            (ispath(report_dir) || islink(report_dir)) &&
                throw(ArgumentError("report directory must not already exist: $report_dir"))
            validate_paths(report_dir)
            mkpath(dirname(report_dir))
            mkdir(report_dir)
        else
            parent = isnothing(requested_output) ? abspath("output") : dirname(requested_output)
            stem = splitext(basename(input))[1]
            report_dir = numbered_run_directory(joinpath(parent, "runs"), stem; validate=validate_paths)
        end
        output = output_for(report_dir)
        for destination in (output, explanation_path)
            if !isnothing(destination) && cli_inside(destination, report_dir)
                mkpath(dirname(destination))
            end
        end
        warmup_ns = UInt64(0)
        if warmup > 0
            warmup_start = time_ns()
            for _ in 1:warmup
                mine(rows; min_support, min_count, counter)
            end
            warmup_ns = time_ns() - warmup_start
        end
        result = if !isnothing(explanation_path)
            open(explanation_path, "w") do stream
                mine(rows; min_support, min_count, counter,
                     explain=ExplainTrace(stream; limit))
            end
        else
            mine(rows; min_support, min_count, counter,
                 explain=explain ? explanation : nothing)
        end
        write_start = time_ns()
        write_spmf(result.itemsets, output)
        write_ns = time_ns() - write_start
        if trace
            println(err, "transactions=", result.transaction_count, " min_count=",
                    result.min_support_count, " itemsets=", length(result.itemsets))
            for level in result.levels
                println(err, "k=", level.size, " candidates=", level.candidates,
                        " frequent=", level.frequent)
            end
        end
        cli_ns = time_ns() - cli_start
        metadata = Dict{String,Any}(
            "run_id" => run_id,
            "started_utc" => Dates.format(started, dateformat"yyyy-mm-ddTHH:MM:SS.sssZ"),
            "input_path" => input, "minsup_argument" => get(values, "--minsup", ""),
            "input_sha256" => input_sha256, "input_bytes" => length(input_bytes),
            "input_hash_scope" => "exact_bytes_parsed", "implementation_sha256" => source_sha256,
            "counter" => counter, "warmup_runs" => warmup,
            "explain_enabled" => explain || !isnothing(explanation_path), "explain_limit" => limit)
        write_run_report(report_dir, result; metadata, output_path=output, explanation_path,
                         read_ns, write_ns, warmup_ns, cli_ns)
        if !quiet
            println(out, "Report: ", report_dir)
            println(out, "MiningMs: ", result.mining_ns / 1_000_000)
        end
        return 0
    catch exc
        if exc isa ArgumentError || exc isa SystemError || exc isa Base.IOError
            print(err, "apriori: error: ")
            showerror(err, exc)
            println(err)
            return 2
        end
        rethrow()
    end
end
''',
    "src/explain.jl": r'''abstract type AbstractAprioriTrace end
struct NoTrace <: AbstractAprioriTrace end

"""Stream real Apriori events, limiting detail lines without limiting mining.

    ExplainTrace(io=stderr; limit=5000)

Create a fresh trace for each mine() call. The final summary is always printed,
even when the detail limit was reached. No event history is kept in memory.
"""
mutable struct ExplainTrace{T<:IO} <: AbstractAprioriTrace
    io::T
    limit::Int
    emitted::Int
    truncated::Bool
end

function ExplainTrace(io::IO=stderr; limit=5000)
    if limit isa Bool || !(limit isa Integer) || !(1 <= limit <= typemax(Int))
        throw(ArgumentError("explain limit must be a positive integer fitting Int"))
    end
    return ExplainTrace(io, Int(limit), 0, false)
end

item_text(items) = "{" * join(items, ',') * "}"
record!(::NoTrace, event, args...) = nothing

function record!(trace::ExplainTrace, event::Symbol, args...)
    if trace.emitted >= trace.limit
        if !trace.truncated
            println(trace.io, "[Log đã đạt giới hạn ", trace.limit,
                    " dòng chi tiết; thuật toán vẫn tiếp tục tính đầy đủ.]")
            trace.truncated = true
        end
        return nothing
    end
    text = if event === :begin
        n, threshold, mode = args
        "BẮT ĐẦU Apriori: N=$n; min_count=$threshold; counter=$mode"
    elseif event === :threshold
        ratio, n, threshold = args
        "Ngưỡng support: ceil($ratio × $n) = $threshold"
    elseif event === :input
        tid, row = args
        "Dữ liệu chuẩn hóa: T$tid = $(item_text(row))"
    elseif event === :level
        "MỨC k=$(args[1])"
    elseif event === :transaction
        k, tid, row = args
        "Quét T$tid = $(item_text(row)) (k=$k)"
    elseif event === :increment
        tid, candidate, count = args
        "  T$tid chứa $(item_text(candidate)): count $(count-1) -> $count"
    elseif event === :join
        left, right, candidate = args
        "JOIN $(item_text(left)) + $(item_text(right)) -> $(item_text(candidate))"
    elseif event === :prune
        candidate, subset, k = args
        "PRUNE $(item_text(candidate)): thiếu tập con $(item_text(subset)) trong L_$k -> loại"
    elseif event === :accepted
        "Ứng viên $(item_text(args[1])): qua prune (mọi tập con đều frequent)"
    elseif event === :candidates
        "C_$(args[1]) sau prune: $(args[2]) ứng viên"
    elseif event === :decision
        candidate, count, threshold = args
        count >= threshold ? "GIỮ $(item_text(candidate)): support=$count >= $threshold" :
                             "LOẠI $(item_text(candidate)): support=$count < $threshold"
    elseif event === :frequent
        "L_$(args[1]): $(args[2]) tập phổ biến"
    elseif event === :frequent_item
        "  $(item_text(args[1])) #SUP: $(args[2])"
    elseif event === :stop
        "DỪNG: $(args[1])"
    else
        throw(ArgumentError("unknown Apriori trace event: $event"))
    end
    println(trace.io, text)
    trace.emitted += 1
    return nothing
end

explain_input!(::NoTrace, database, ratio, threshold) = nothing
function explain_input!(trace::ExplainTrace, database, ratio, threshold)
    !isnothing(ratio) && record!(trace, :threshold, ratio, length(database), threshold)
    for (tid, row) in enumerate(database)
        record!(trace, :input, tid, row)
    end
end

explain_singletons!(::NoTrace, counts, threshold) = nothing
function explain_singletons!(trace::ExplainTrace, counts, threshold)
    for item in sort!(collect(keys(counts)))
        record!(trace, :decision, (item,), counts[item], threshold)
    end
end

explain_frequent!(::NoTrace, k, candidates, counts) = nothing
function explain_frequent!(trace::ExplainTrace, k, candidates, counts)
    record!(trace, :frequent, k, length(candidates))
    for (candidate, count) in zip(candidates, counts)
        record!(trace, :frequent_item, candidate, count)
    end
end

ordered_matches!(::NoTrace, ids) = ids
ordered_matches!(::ExplainTrace, ids) = sort!(ids)

finish_explanation!(::NoTrace, total) = nothing
function finish_explanation!(trace::ExplainTrace, total)
    println(trace.io, "KẾT THÚC: ", total,
            " frequent itemsets; support là số giao dịch chứa itemset.")
end
''',
    "src/FrequentItemsetMining.jl": r'''module FrequentItemsetMining

using Dates
using SHA
using TOML
using UUIDs

export mine, mine_bitset, mine_streaming, apriori_gen, MiningResult, LevelStats, read_spmf, write_spmf, main
export HashTree, HashQueryWorkspace, matching_ids, matching_ids!
export ExplainTrace
export PhaseTiming, ExternalMiningSummary, mine_external, mine_bitset_external, mine_streaming_external

include("explain.jl")
include("hash_tree.jl")
include("apriori.jl")
include("io.jl")
include("apriori_optimized.jl")
include("apriori_external.jl")
include("reports.jl")
include("run_directories.jl")
include("cli.jl")

end
''',
    "src/hash_tree.jl": r'''"""A hash-tree node: candidate IDs in a leaf, bucket-to-node indices otherwise."""
mutable struct HashTreeNode
    candidate_ids::Vector{Int}
    children::Dict{Int,Int}
end

"""A same-width candidate hash tree whose IDs are input candidate positions."""
struct HashTree{K}
    candidates::Vector{NTuple{K,Int}}
    compressed_candidates::Vector{NTuple{K,Int}}
    nodes::Vector{HashTreeNode}
    item_positions::Dict{Int,Int}
    bucket_count::Int
    leaf_capacity::Int
end

function _hash_positive_int(value, name)
    if !(value isa Integer) || value isa Bool || value < 1 || value > typemax(Int)
        throw(ArgumentError("$name must be a positive integer fitting Int"))
    end
    return Int(value)
end

function _hash_candidate_item(item)
    if !(item isa Integer) || item isa Bool || item < 0 || item > typemax(Int)
        throw(ArgumentError("candidate items must be nonnegative integers fitting Int"))
    end
    return Int(item)
end

function HashTree(candidates::AbstractVector; bucket_count=17, leaf_capacity=8)
    buckets = _hash_positive_int(bucket_count, "bucket_count")
    capacity = _hash_positive_int(leaf_capacity, "leaf_capacity")
    if isempty(candidates)
        candidate_type = eltype(candidates)
        if !(candidate_type <: Tuple) || !isconcretetype(candidate_type)
            throw(ArgumentError("empty candidates must have a concrete tuple element type"))
        end
        width = fieldcount(candidate_type)
    else
        first(candidates) isa Tuple || throw(ArgumentError("candidates must be tuples"))
        width = length(first(candidates))
    end
    width > 0 || throw(ArgumentError("candidates must be nonempty tuples"))
    canonical = NTuple{width,Int}[]
    sizehint!(canonical, length(candidates))
    for candidate in candidates
        candidate isa Tuple && length(candidate) == width ||
            throw(ArgumentError("candidates must be tuples of the same width"))
        converted = ntuple(index -> _hash_candidate_item(candidate[index]), width)
        for index in 2:width
            converted[index - 1] < converted[index] ||
                throw(ArgumentError("candidate items must be strictly increasing"))
        end
        push!(canonical, converted)
    end
    return _build_hash_tree(canonical, buckets, capacity)
end

function _build_hash_tree(candidates::Vector{NTuple{K,Int}}, buckets::Int,
                          capacity::Int) where {K}
    positions = Dict{Int,Int}()
    for candidate in candidates, item in candidate
        if !haskey(positions, item)
            positions[item] = length(positions) + 1
        end
    end
    compressed = NTuple{K,Int}[ntuple(i -> positions[candidate[i]], K)
                               for candidate in candidates]
    nodes = HashTreeNode[HashTreeNode(collect(eachindex(candidates)), Dict{Int,Int}())]
    tree = HashTree{K}(candidates, compressed, nodes, positions, buckets, capacity)
    _split_hash_leaf!(tree, 1, 1)
    return tree
end

function _split_hash_leaf!(tree::HashTree{K}, node_id::Int, depth::Int) where {K}
    node = tree.nodes[node_id]
    if length(node.candidate_ids) <= tree.leaf_capacity || depth > K
        return
    end
    for candidate_id in node.candidate_ids
        bucket = mod(tree.candidates[candidate_id][depth], tree.bucket_count)
        child_id = get(node.children, bucket, 0)
        if child_id == 0
            push!(tree.nodes, HashTreeNode(Int[], Dict{Int,Int}()))
            child_id = length(tree.nodes)
            node.children[bucket] = child_id
        end
        push!(tree.nodes[child_id].candidate_ids, candidate_id)
    end
    empty!(node.candidate_ids)
    for child_id in values(node.children)
        _split_hash_leaf!(tree, child_id, depth + 1)
    end
    return
end

"""Reusable query buffers; results remain valid only until the next query."""
struct HashQueryWorkspace
    membership::BitVector
    touched_items::Vector{Int}
    visited_leaves::BitVector
    touched_leaves::Vector{Int}
    result_ids::Vector{Int}
end

HashQueryWorkspace(tree::HashTree) = HashQueryWorkspace(
    falses(length(tree.item_positions)), Int[], falses(length(tree.nodes)), Int[], Int[])

function _query_hash_node!(workspace::HashQueryWorkspace, tree::HashTree{K},
                           node_id::Int, transaction::Vector{Int}, start::Int,
                           depth::Int) where {K}
    node = tree.nodes[node_id]
    if isempty(node.children)
        workspace.visited_leaves[node_id] && return
        workspace.visited_leaves[node_id] = true
        push!(workspace.touched_leaves, node_id)
        for candidate_id in node.candidate_ids
            candidate = tree.compressed_candidates[candidate_id]
            if all(index -> workspace.membership[index], candidate)
                push!(workspace.result_ids, candidate_id)
            end
        end
        return
    end
    # A node may be reached through several transaction prefixes. Only leaves
    # can be skipped globally, after their complete candidates were checked.
    for index in start:(length(transaction) - (K - depth))
        bucket = mod(transaction[index], tree.bucket_count)
        child_id = get(node.children, bucket, 0)
        if child_id != 0
            _query_hash_node!(workspace, tree, child_id, transaction, index + 1, depth + 1)
        end
    end
    return
end

"""Return matching candidate IDs in workspace-owned storage.

Transactions are sorted, unique vectors of nonnegative Int item IDs, as produced
by the miner's normalization. Create the workspace for the tree being queried.
"""
function matching_ids!(workspace::HashQueryWorkspace, tree::HashTree{K},
                       transaction::Vector{Int}) where {K}
    for index in workspace.touched_items
        workspace.membership[index] = false
    end
    for node_id in workspace.touched_leaves
        workspace.visited_leaves[node_id] = false
    end
    empty!(workspace.touched_items)
    empty!(workspace.touched_leaves)
    empty!(workspace.result_ids)
    for item in transaction
        index = get(tree.item_positions, item, 0)
        if index != 0 && !workspace.membership[index]
            workspace.membership[index] = true
            push!(workspace.touched_items, index)
        end
    end
    if length(transaction) >= K
        _query_hash_node!(workspace, tree, 1, transaction, 1, 1)
    end
    return workspace.result_ids
end

"""Return an independently owned vector of matching candidate IDs."""
function matching_ids(tree::HashTree, transaction::Vector{Int})
    return matching_ids!(HashQueryWorkspace(tree), tree, transaction)
end
''',
    "src/io.jl": r'''"""Read SPMF integer transactions; skip metadata and blank lines."""
function read_spmf(source::IO)
    database = Vector{Int}[]
    for (line_number, line) in enumerate(eachline(source))
            if line_number == 1 && startswith(line, '\ufeff')
                line = chop(line; head=1, tail=0)
            end
            text = strip(line)
            (isempty(text) || startswith(text, ('#', '%', '@'))) && continue
            items = Int[]
            for token in split(text)
                item = tryparse(Int, token)
                (isnothing(item) || item < 0) &&
                    throw(ArgumentError("line $line_number: expected nonnegative integer items"))
                push!(items, item)
            end
            push!(database, unique!(sort!(items)))
    end
    return database
end

read_spmf(path::AbstractString)=open(read_spmf,path,"r")

"""Write all itemsets in deterministic size/lexicographic order."""
function write_spmf(itemsets::AbstractDict, path::AbstractString)
    ordered = sort!(collect(keys(itemsets)); by=items -> (length(items), items))
    open(path, "w") do target
        for itemset in ordered
            println(target, join(itemset, ' '), " #SUP: ", itemsets[itemset])
        end
    end
    return nothing
end
''',
    "src/reports.jl": r'''"""SHA-256 of the implementation files, independent of output or test data."""
function implementation_fingerprint()
    root=dirname(@__DIR__)
    files=[joinpath(root,"Project.toml")]
    append!(files,sort!(filter(path -> endswith(path,".jl"), readdir(@__DIR__;join=true))))
    stream=IOBuffer()
    for path in files
        write(stream,replace(relpath(path,root),'\\'=>'/'),'\0',read(path),'\0')
    end
    return bytes2hex(sha256(take!(stream)))
end

const PHASE_NAMES = Dict(
    :normalize=>"Chuẩn hóa giao dịch", :threshold=>"Tính ngưỡng support",
    :singleton_count=>"Đếm item đơn", :generate_candidates=>"Sinh ứng viên (join + prune)",
    :build_counter=>"Dựng bộ đếm / hash-tree", :count_support=>"Quét giao dịch và đếm support",
    :filter=>"Lọc tập phổ biến", :level_total=>"Tổng thời gian mức k",
    :read_input=>"Đọc và parse dữ liệu", :write_output=>"Ghi kết quả SPMF",
    :warmup_total=>"Các lượt warmup", :mining_total=>"Toàn bộ mine()",
    :cli_total=>"Xử lý CLI đến trước ghi bundle")

elapsed_ms(ns::UInt64)=Float64(ns)/1_000_000
csv_cell(value)="\"" * replace(string(value),'"'=>"\"\"") * "\""
report_path_key(path)=Sys.iswindows() ? lowercase(normpath(abspath(path))) : normpath(abspath(path))

function report_timings(result, read_ns, write_ns, warmup_ns, cli_ns)
    rows=PhaseTiming[PhaseTiming(:read_input,0,read_ns,0,0)]
    push!(rows,PhaseTiming(:warmup_total,0,warmup_ns,0,0))
    append!(rows,result.timings)
    push!(rows,PhaseTiming(:mining_total,0,result.mining_ns,0,length(result.itemsets)))
    push!(rows,PhaseTiming(:write_output,0,write_ns,0,length(result.itemsets)))
    push!(rows,PhaseTiming(:cli_total,0,cli_ns,0,length(result.itemsets)))
    return rows
end

"""Persist a successful run as human-readable and machine-readable artifacts.

The caller owns a freshly created report_dir. Existing report artifacts are
never overwritten. itemsets.txt / trace.txt may already be the explicit CLI
destinations within that directory. Report serialization and input/source
fingerprinting are outside cli_ns and mining_ns; process wall time includes them.
"""
function write_run_report(report_dir::AbstractString,result::MiningResult;
                          metadata::AbstractDict,output_path::AbstractString,
                          explanation_path=nothing,read_ns::UInt64=UInt64(0),
                          write_ns::UInt64=UInt64(0),warmup_ns::UInt64=UInt64(0),
                          cli_ns::UInt64=UInt64(0))
    isdir(report_dir) || throw(ArgumentError("report directory must already exist"))
    report=joinpath(report_dir,"report.txt")
    timings=joinpath(report_dir,"timings.csv")
    metapath=joinpath(report_dir,"metadata.toml")
    snapshot=joinpath(report_dir,"itemsets.txt")
    tracepath=joinpath(report_dir,"trace.txt")
    for path in (report,timings,metapath)
        ispath(path) && throw(ArgumentError("report artifact already exists: $path"))
    end
    same_output=report_path_key(snapshot)==report_path_key(output_path)
    ispath(snapshot) && !same_output && throw(ArgumentError("itemset snapshot already exists"))
    same_trace=!isnothing(explanation_path) && report_path_key(tracepath)==report_path_key(explanation_path)
    !isnothing(explanation_path) && ispath(tracepath) && !same_trace &&
        throw(ArgumentError("trace snapshot already exists"))

    meta=Dict{String,Any}(string(k)=>v for (k,v) in metadata)
    input=abspath(meta["input_path"])
    meta["schema_version"]=1
    meta["input_path"]=input
    input_final_hash=isfile(input) ? open(io -> bytes2hex(sha256(io)),input) : ""
    meta["input_final_sha256"]=input_final_hash
    meta["input_sha256"]=get(meta,"input_sha256",input_final_hash)
    meta["input_bytes"]=get(meta,"input_bytes",isfile(input) ? filesize(input) : -1)
    meta["input_hash_scope"]=get(meta,"input_hash_scope","file_at_report_time")
    meta["input_changed_after_read"]=meta["input_sha256"]!=input_final_hash
    source_final_hash=implementation_fingerprint()
    meta["implementation_final_sha256"]=source_final_hash
    meta["implementation_sha256"]=get(meta,"implementation_sha256",source_final_hash)
    meta["implementation_changed_during_run"]=meta["implementation_sha256"]!=source_final_hash
    meta["report_written_utc"]=Dates.format(now(UTC),dateformat"yyyy-mm-ddTHH:MM:SS.sss") * "Z"
    meta["julia_version"]=string(VERSION)
    meta["os"]=string(Sys.KERNEL)
    meta["architecture"]=string(Sys.ARCH)
    meta["cpu"]=Sys.CPU_NAME
    meta["julia_threads"]=Threads.nthreads()
    meta["cpu_threads"]=Sys.CPU_THREADS
    meta["word_size"]=Sys.WORD_SIZE
    meta["transactions"]=result.transaction_count
    meta["min_support_count"]=result.min_support_count
    meta["itemset_count"]=length(result.itemsets)
    meta["mining_ns"]=Int(result.mining_ns)
    meta["read_ns"]=Int(read_ns)
    meta["write_ns"]=Int(write_ns)
    meta["warmup_ns"]=Int(warmup_ns)
    meta["cli_ns"]=Int(cli_ns)
    meta["timing_policy"]=meta["warmup_runs"]==0 ? "cold_call_may_include_jit" :
                          "warmup_requested_jit_may_remain"
    meta["timing_unit"]="nanoseconds; CSV also includes elapsed_ms"
    meta["timing_scope"]="mining excludes input/output; cli excludes startup and report serialization; nested level/phase rows must not be summed"
    meta["output_path"]=abspath(output_path)
    meta["itemsets_path"]=abspath(snapshot)
    meta["report_path"]=abspath(report)
    meta["timings_path"]=abspath(timings)
    meta["trace_path"]=isnothing(explanation_path) ? "" : abspath(tracepath)
    rows=report_timings(result,read_ns,write_ns,warmup_ns,cli_ns)

    !same_output && write_spmf(result.itemsets,snapshot)
    !isnothing(explanation_path) && !same_trace && cp(explanation_path,tracepath)
    open(timings,"w") do stream
        println(stream,"run_id,k,phase,elapsed_ns,elapsed_ms,candidates,frequent")
        for row in rows
            println(stream,join(csv_cell.(
                (meta["run_id"],row.size,row.phase,row.elapsed_ns,elapsed_ms(row.elapsed_ns),
                 row.candidates,row.frequent)),','))
        end
    end
    open(metapath,"w") do stream
        TOML.print(stream,meta;sorted=true)
    end
    open(report,"w") do stream
        println(stream,"BÁO CÁO MỘT LẦN CHẠY APRIORI")
        for key in ("run_id","started_utc","input_path","input_sha256","input_hash_scope",
                    "input_changed_after_read","implementation_sha256","implementation_changed_during_run",
                    "transactions","minsup_argument","min_support_count","counter",
                    "itemset_count","julia_version","os","architecture","cpu","julia_threads",
                    "warmup_runs","explain_enabled","explain_limit","timing_policy")
            println(stream,key," = ",meta[key])
        end
        println(stream,"\nPHẠM VI ĐO THỜI GIAN")
        println(stream,"Đơn vị: ms để đọc; CSV giữ elapsed_ns gốc. time_ns() là đồng hồ monotonic.")
        println(stream,"read_input gồm đọc snapshot, SHA256 và parse cùng nội dung; mining không gồm bước này.")
        println(stream,"Mining đo mine(); không gồm ghi output hoặc serialize báo cáo.")
        println(stream,"CLI đo xử lý đến trước ghi bundle; không gồm khởi động Julia hoặc ghi bundle.")
        println(stream,"level_total và các phase con lồng nhau: không cộng mọi dòng thành tổng.")
        println(stream,"Thời gian có thể chứa JIT/GC/dispatch. Warmup không bảo đảm bỏ hết JIT.")
        println(stream,"Bật explain có chi phí format/IO trong các phase; chỉ so cùng chế độ.")
        println(stream,"Scan có build_counter = 0; generate_candidates gồm join và prune.")
        println(stream,"Mức cuối C_k rỗng vẫn có timing, nhưng không thêm LevelStats đã đếm giả.")
        println(stream,"Đối chiếu hash cuối và serialize báo cáo nằm ngoài mining/CLI; hash snapshot nằm trong read_input.")
        if meta["input_changed_after_read"] || meta["implementation_changed_during_run"]
            println(stream,"CẢNH BÁO: file input hoặc source đã đổi sau snapshot; metadata giữ hash ban đầu và hash cuối riêng.")
        end
        println(stream,"\nk | phase | mô tả | elapsed_ms | candidates | frequent")
        for row in rows
            println(stream,row.size," | ",row.phase," | ",get(PHASE_NAMES,row.phase,string(row.phase)),
                    " | ",round(elapsed_ms(row.elapsed_ns);digits=6)," | ",row.candidates," | ",row.frequent)
        end
        println(stream,"\nTOÀN BỘ FREQUENT ITEMSETS")
        println(stream,"itemset | support_count | support_relative")
        for itemset in sort!(collect(keys(result.itemsets));by=items -> (length(items),items))
            count=result.itemsets[itemset]
            ratio=iszero(result.transaction_count) ? 0.0 : count/result.transaction_count
            println(stream,item_text(itemset)," | ",count," | ",round(ratio;digits=10))
        end
        isempty(result.itemsets) && println(stream,"(Không có itemset đạt ngưỡng.)")
        println(stream,"\nFile chuẩn SPMF: ",abspath(snapshot))
        println(stream,"Dữ liệu thời gian: ",abspath(timings))
        println(stream,"Metadata: ",abspath(metapath))
        !isnothing(explanation_path) && println(stream,"Luồng chi tiết: ",abspath(tracepath))
        println(stream,"Đây là lần chạy thực tế; chưa phải benchmark lặp lại hoặc đối chiếu SPMF/Java.")
    end
    return report
end
''',
    "src/run_directories.jl": r'''"""Claim a fresh stem-N directory, starting after the largest existing suffix."""
function numbered_run_directory(parent, stem; validate=identity)
    prefix = stem * "-"
    Sys.iswindows() && (prefix = lowercase(prefix))
    last_number = big(0)
    if isdir(parent)
        for entry in readdir(parent; sort=false)
            name = Sys.iswindows() ? lowercase(entry) : entry
            startswith(name, prefix) || continue
            suffix = chopprefix(name, prefix)
            occursin(r"^[0-9]+\z", suffix) || continue
            last_number = max(last_number, parse(BigInt, suffix))
        end
    end
    number = last_number + 1
    while true
        directory = joinpath(parent, "$stem-$number")
        validate(directory)
        mkpath(parent)
        try
            return mkdir(directory)
        catch exc
            # Another process may have claimed this number since readdir.
            exc isa Base.IOError && exc.code == Base.UV_EEXIST || rethrow()
            number += 1
        end
    end
end
''',
    "bin/apriori.jl": r'''using FrequentItemsetMining

exit(main())
''',
    "bin/benchmark-worker.jl": r'''using FrequentItemsetMining

input, target, count, counter = ARGS
rows = read_spmf(input)
result = mine(rows; min_count=parse(Int, count), counter)
write_spmf(result.itemsets, target)
println("mining_ms=", result.mining_ns / 1e6)
println("k,phase,elapsed_ns,candidates,frequent")
for phase in result.timings
    println(join((phase.size, phase.phase, phase.elapsed_ns, phase.candidates, phase.frequent), ','))
end
''',
    "bin/benchmark.jl": r'''using SHA, Dates, Printf, FrequentItemsetMining

const ROOT = dirname(@__DIR__)
const SIZES = (10, 25, 50, 75, 100)
const ENGINES = ("scan", "hash-tree", "spmf")
const SEED = 42
const PROTOCOL = "apriori-cold-v1"
filehash(path) = open(io -> bytes2hex(sha256(io)), path)
percent(bp) = string(bp ÷ 100, bp % 100 == 0 ? "" : "." * rstrip(lpad(string(bp % 100), 2, '0'), '0'))

function write_samples(input, parent)
    rows = read_spmf(input)
    length(rows) >= 10 || error("At least 10 transactions required")
    ranks = sortperm([bytes2hex(sha256("apriori-v1|$SEED|$i")) for i in eachindex(rows)])
    samples = Dict{Int,NamedTuple}()
    for pct in SIZES
        indices = sort(ranks[1:fld(length(rows) * pct, 100)])
        path = joinpath(parent, "sample-$pct.txt")
        items = Set{Int}()
        entries = 0
        open(path, "w") do io
            for i in indices
                println(io, join(rows[i], ' '))
                union!(items, rows[i])
                entries += length(rows[i])
            end
        end
        samples[pct] = (;path, n=length(indices), sha=filehash(path),
            idsha=bytes2hex(sha256(join(indices, "\n") * "\n")),
            distinct=length(items), mean_width=entries / length(indices))
    end
    return samples
end

function patterns(path, minimum, n)
    result = Dict{String,Int}()
    open(path) do io
      for line in eachline(io)
        (isempty(strip(line)) || startswith(line, '@')) && continue
        parts = split(line, "#SUP:")
        length(parts) == 2 || error("Invalid itemset in $path")
        items = sort(parse.(Int, split(parts[1])))
        count = parse(Int, strip(parts[2]))
        key = join(items, ' ')
        isempty(items) && error("Empty itemset in $path")
        allunique(items) && !haskey(result, key) || error("Duplicate item or itemset in $path")
        minimum <= count <= n || error("Invalid support in $path: $line")
        result[key] = count
      end
    end
    return result
end

function signature(result)
    io = IOBuffer()
    for key in sort!(collect(keys(result)))
        println(io, key, " #SUP: ", result[key])
    end
    return bytes2hex(sha256(take!(io)))
end

function compare(ref, got)
    missing = count(k -> !haskey(got, k), keys(ref))
    extra = count(k -> !haskey(ref, k), keys(got))
    wrong = count(k -> haskey(got, k) && got[k] != ref[k], keys(ref))
    total = length(ref) + extra
    return (;missing, extra, wrong, match_pct=total == 0 ? 100.0 : 100 * (total-missing-extra-wrong)/total)
end

struct WinMemory
    size::UInt32
    faults::UInt32
    peak_ws::UInt
    remaining::NTuple{7,UInt}
end

function peak_working_set(handle)
    info = Ref(WinMemory(sizeof(WinMemory), 0, 0, ntuple(_ -> UInt(0), 7)))
    ok = ccall((:GetProcessMemoryInfo, "psapi"), stdcall, Int32,
        (Ptr{Cvoid}, Ref{WinMemory}, UInt32), handle, info, sizeof(WinMemory))
    return ok == 0 ? 0 : Int(info[].peak_ws)
end

function timed_process(command, logpath)
    timerfile = logpath * ".rss"
    measured = if Sys.iswindows()
        command
    elseif Sys.islinux() && isfile("/usr/bin/time")
        `/usr/bin/time -o $timerfile -f %M $command`
    else
        error("Use Windows or Linux with GNU /usr/bin/time")
    end
    open(logpath, "w") do log
        start_ns = time_ns()
        proc = run(pipeline(measured; stdout=log, stderr=log); wait=false)
        peak = 0
        try
            if Sys.iswindows()
                handle = ccall((:OpenProcess, "kernel32"), stdcall, Ptr{Cvoid},
                    (UInt32, Int32, UInt32), 0x0410, 0, getpid(proc))
                handle == C_NULL && error("Cannot open process for memory measurement")
                try
                    while process_running(proc)
                        peak = max(peak, peak_working_set(handle))
                        sleep(0.01)
                    end
                    peak = max(peak, peak_working_set(handle))
                finally
                    ccall((:CloseHandle, "kernel32"), stdcall, Int32, (Ptr{Cvoid},), handle)
                end
            end
            wait(proc)
        finally
            if process_running(proc)
                kill(proc)
                wait(proc)
            end
        end
        wall = (time_ns() - start_ns) / 1e6
        success(proc) || error("Child failed ($(proc.exitcode)); see $logpath")
        Sys.islinux() && (peak = 1024 * parse(Int, strip(read(timerfile, String))))
        peak > 0 || error("Peak RAM unavailable: $logpath")
        return (;wall, mib=peak/2^20)
    end
end

function capture(command)
    io = IOBuffer()
    run(pipeline(command; stdout=io, stderr=io))
    return strip(String(take!(io)))
end

function write_csv(path, rows)
    names = keys(first(rows))
    escaped(x) = "\"" * replace(string(x), "\"" => "\"\"") * "\""
    open(path, "w") do io
        println(io, join(names, ','))
        for row in rows
            println(io, join(escaped.(values(row)), ','))
        end
    end
end

function bench(args)
    length(args) == 5 || error("Usage: benchmark.jl DATASET INPUT SPMF.jar BASIS_POINTS_LIST MIDDLE_BP")
    dataset, input_arg, jar_arg, grid_arg, middle_arg = args
    dataset in ("Retail", "Chess", "Mushrooms", "Pumsb") || error("Unknown dataset")
    input, jar = abspath(input_arg), abspath(jar_arg)
    grid = parse.(Int, split(grid_arg, ','))
    middle = parse(Int, middle_arg)
    5 <= length(grid) <= 7 && allunique(grid) && issorted(grid; rev=true) &&
        all(bp -> 0 < bp <= 10000, grid) && middle in grid || error("Invalid minsup grid")
    launcher = something(Sys.which("java"), "java")
    settings = capture(`$launcher -XshowSettings:properties -version`)
    home = match(r"(?m)^\s*java\.home\s*=\s*([^\r\n]+)", settings)
    isnothing(home) && error("Cannot locate the JVM from java.home")
    # Oracle's PATH shim on Windows spawns the JVM as another process.
    # Measuring the executable in java.home gives the JVM's working set.
    java = joinpath(strip(home[1]), "bin", Sys.iswindows() ? "java.exe" : "java")
    isfile(java) || error("Missing Java executable: $java")
    java_version = first(split(capture(`$java -version`), '\n')) |> strip
    version_output = capture(`$java -jar $jar version`)
    spmf_version = match(r"(?i)SPMF version\s+([\d.]+)", version_output)
    isnothing(spmf_version) && error("Cannot identify SPMF version: $version_output")
    base = joinpath(ROOT, "benchmarks", dataset)
    bundle = FrequentItemsetMining.numbered_run_directory(base, "run")
    raw_parent = mkpath(joinpath(ROOT, "output", "benchmarks", dataset))
    evidence = mkdir(joinpath(raw_parent, basename(bundle)))
    samples = write_samples(input, evidence)
    samples[100].n > 0 || error("Empty sample")
    worker = joinpath(@__DIR__, "benchmark-worker.jl")
    julia = joinpath(Sys.BINDIR, Base.julia_exename())
    common = (;protocol=PROTOCOL, dataset, run=basename(evidence), seed=SEED, raw_sha256=filehash(input),
        source_sha256=FrequentItemsetMining.implementation_fingerprint(), harness_sha256=bytes2hex(sha256(vcat(read(@__FILE__),read(worker)))),
        spmf_version=spmf_version.captures[1], jar_sha256=filehash(jar), julia_version=string(VERSION), java_version,
        os=string(Sys.KERNEL), arch=string(Sys.ARCH), cpu=Sys.cpu_info()[1].model, host_ram_bytes=Sys.total_memory(),
        os_threads=Sys.CPU_THREADS, julia_threads=1, java_active_processors=1, started_utc=string(now(UTC)))
    cases = vcat([(100,bp) for bp in grid], [(pct,middle) for pct in SIZES if pct != 100])
    rows = NamedTuple[]
    println("Evidence: ", evidence)
    flush(stdout)
    for (pct,bp) in cases
        sample = samples[pct]
        minimum = cld(sample.n * bp, 10000)
        for rep in 1:3
            order = circshift(collect(ENGINES), 1-rep)
            trial = mkpath(joinpath(evidence, "p$pct-bp$bp", "rep-$rep"))
            metrics = Dict{String,NamedTuple}()
            for engine in order
                output = joinpath(trial, "$engine.txt")
                cmd = if engine == "spmf"
                    # Relative data paths also work with Windows Java launchers
                    # whose argument encoding cannot represent the project path.
                    Cmd(`$java -XX:ActiveProcessorCount=1 -jar $jar run Apriori $(relpath(sample.path,trial)) $(basename(output)) $(percent(bp)*"%")`; dir=trial)
                else
                    `$julia --startup-file=no --threads=1 --project=$ROOT $worker $(sample.path) $output $minimum $engine`
                end
                timing = timed_process(cmd, joinpath(trial, "$engine.log"))
                isfile(output) || error("Engine produced no itemsets file: $trial/$engine")
                metrics[engine] = (;timing..., output)
            end
            ref = patterns(metrics["spmf"].output, minimum, sample.n)
            spmf_log = read(joinpath(trial, "spmf.log"), String)
            stated = match(r"Frequent itemsets count\s*:\s*(\d+)", spmf_log)
            !isnothing(stated) && parse(Int, stated[1]) == length(ref) || error("Incomplete SPMF output at $trial")
            matched = true
            fields = (;)
            for engine in ENGINES
                m = metrics[engine]
                got = engine == "spmf" ? ref : patterns(m.output, minimum, sample.n)
                diff = compare(ref, got)
                matched &= (diff.missing + diff.extra + diff.wrong == 0)
                native_time = engine == "spmf" ? match(r"Total time\s*~\s*([\d.]+) ms", spmf_log) :
                    match(r"mining_ms=([\d.eE+-]+)", read(joinpath(trial,"$engine.log"), String))
                vals = (;itemsets=length(got), output_sha256=signature(got), wall_ms=m.wall, peak_mib=m.mib,
                    native_ms=parse(Float64, only(native_time.captures)),
                    missing=diff.missing, extra=diff.extra, wrong_support=diff.wrong, match_pct=diff.match_pct)
                prefix = replace(engine, '-'=>'_')
                fields = merge(fields, (;(Symbol(prefix,"_",k)=>v for (k,v) in pairs(vals))...))
            end
            row = (;common..., sample_pct=pct, transactions=sample.n, distinct_items=sample.distinct,
                mean_width=sample.mean_width, sample_sha256=sample.sha, selected_tid_sha256=sample.idsha,
                minsup_bp=bp, min_count=minimum, rep, engine_order=join(order,"/"),
                evidence=relpath(trial,ROOT), fields..., status=matched ? "PASS" : "MISMATCH")
            push!(rows, row)
            write_csv(joinpath(evidence, "progress.csv"), rows)
            println("$(length(rows))/$(3*length(cases)) | sample=$pct% minsup=$(percent(bp))% rep=$rep | $(row.status) | $(length(ref)) itemsets")
            flush(stdout)
        end
    end
    csv = joinpath(bundle, "results.csv")
    cp(joinpath(evidence,"progress.csv"), csv; force=false)
    println("CSV: ", csv)
    all(row -> row.status == "PASS", rows) || error("Support mismatch; see CSV and raw outputs")
end

if abspath(PROGRAM_FILE) == @__FILE__
    bench(ARGS)
end
''',
    "bin/new-batch.jl": r'''include(joinpath(@__DIR__, "..", "src", "run_directories.jl"))

length(ARGS) == 1 || error("Usage: julia bin/new-batch.jl PARENT")
println(basename(numbered_run_directory(abspath(only(ARGS)), "batch")))
''',
    "bin/rules.jl": r'''#!/usr/bin/env julia
# Sinh luật kết hợp từ file frequent itemsets (đầu ra của bin/apriori.jl).
#
#   julia --project=. bin/rules.jl --itemsets output/retail.txt \
#         --data data/benchmarks/retail.txt --minconf 0.5 --top 10 \
#         --out output/retail-rules.csv

include(joinpath(@__DIR__, "..", "src", "AssociationRules.jl"))
using .AssociationRules

const USAGE = """
Cách dùng: julia --project=. bin/rules.jl --itemsets FILE (--data FILE | --ntrans N) [tùy chọn]

  --itemsets FILE   file itemsets SPMF do apriori.jl tạo ra (dòng "1 2 #SUP: 7")
  --data FILE       file transaction gốc, dùng để đếm tổng số transaction N
  --ntrans N        cho trực tiếp N thay cho --data
  --minconf X       ngưỡng confidence trong [0,1] (mặc định 0.5)
  --minlift X       ngưỡng lift tối thiểu (mặc định 0)
  --top K           in K luật đầu theo lift (mặc định 10)
  --out FILE        ghi toàn bộ luật ra CSV
"""

function parse_options(args::Vector{String})
    opts = Dict{String,String}()
    i = 1
    while i <= length(args)
        a = args[i]
        if a in ("-h", "--help")
            print(USAGE)
            exit(0)
        end
        startswith(a, "--") || error("tham số không hợp lệ: $a")
        i < length(args) || error("thiếu giá trị cho $a")
        opts[a] = args[i + 1]
        i += 2
    end
    return opts
end

fmt(v::Vector{Int}) = "{" * join(v, ", ") * "}"

function main(args::Vector{String})
    opts = parse_options(args)
    haskey(opts, "--itemsets") || error("cần --itemsets FILE\n\n" * USAGE)
    n = if haskey(opts, "--ntrans")
        parse(Int, opts["--ntrans"])
    elseif haskey(opts, "--data")
        count_transactions(opts["--data"])
    else
        error("cần --data FILE hoặc --ntrans N\n\n" * USAGE)
    end
    minconf = parse(Float64, get(opts, "--minconf", "0.5"))
    minlift = parse(Float64, get(opts, "--minlift", "0"))
    top = parse(Int, get(opts, "--top", "10"))

    counts = read_itemsets(opts["--itemsets"])
    rules = generate_rules(counts, n; minconf=minconf, minlift=minlift)

    println("Transaction N = $n | itemsets = $(length(counts)) | " *
            "luật (minconf=$minconf, minlift=$minlift) = $(length(rules))")
    for (i, r) in enumerate(first(rules, top))
        println(lpad(i, 3), ". ", fmt(r.antecedent), " => ", fmt(r.consequent),
                " | count=", r.count,
                " sup=", round(r.support; digits=4),
                " conf=", round(r.confidence; digits=4),
                " lift=", round(r.lift; digits=3))
    end
    if haskey(opts, "--out")
        write_rules_csv(opts["--out"], rules)
        println("Đã ghi ", length(rules), " luật vào ", opts["--out"])
    end
    return nothing
end

if abspath(PROGRAM_FILE) == @__FILE__
    main(ARGS)
end
''',
    "test/benchmark_protocol.jl": r'''using Test
include(joinpath(@__DIR__, "..", "bin", "benchmark.jl"))

@testset "Nested transaction samples and support comparisons" begin
    mktempdir() do scratch
        source = joinpath(scratch, "transactions.txt")
        write(source, "@ITEM=9=name\n\n" * join(1:29, "\n") * "\n")
        samples = write_samples(source, scratch)
        @test [samples[p].n for p in SIZES] == [2, 7, 14, 21, 29]
        previous = Int[]
        for p in SIZES
            indices = parse.(Int, readlines(samples[p].path))
            @test issorted(indices) && allunique(indices)
            @test issubset(previous, indices)
            @test samples[p].idsha == filehash(samples[p].path)
            previous = indices
        end
        other = write_samples(source, mkpath(joinpath(scratch, "again")))
        @test all(samples[p].sha == other[p].sha for p in SIZES)

        output = joinpath(scratch, "spmf.txt")
        write(output, "@FILETYPE=\"Frequent itemsets\"\n@SOURCE=SPMF\n2 1 #SUP: 4\n9 #SUP: 2\n")
        got = patterns(output, 2, 10)
        @test got == Dict("1 2" => 4, "9" => 2)
        @test compare(got, got).match_pct == 100
        @test compare(Dict(), Dict()).match_pct == 100
        diff = compare(got, Dict("1 2" => 3, "4" => 2))
        @test (diff.missing, diff.extra, diff.wrong, diff.match_pct) == (1, 1, 1, 0)
        @test signature(got) == signature(Dict("9" => 2, "1 2" => 4))
        write(output, "1 2 #SUP: 4\n2 1 #SUP: 4\n")
        @test_throws ErrorException patterns(output, 2, 10)
        @test percent.(Int[1000,750,500,25,1]) == ["10","7.5","5","0.25","0.01"]
    end
end
''',
    "test/cli.jl": r'''const EXPECTED_SPMF = "1 #SUP: 2\n2 #SUP: 3\n3 #SUP: 3\n5 #SUP: 3\n" *
    "1 3 #SUP: 2\n2 3 #SUP: 2\n2 5 #SUP: 3\n3 5 #SUP: 2\n2 3 5 #SUP: 2\n"

function invoke_cli(input, output, options...)
    out, err = IOBuffer(), IOBuffer()
    status = main(["--input", input, "--output", output, options..., "--quiet"]; out=out, err=err)
    return status, String(take!(out)), String(take!(err))
end

@testset "SPMF I/O and CLI" begin
    mktempdir() do scratch
        input, output = joinpath(scratch, "input.txt"), joinpath(scratch, "output.txt")
        write(input, "\ufeff# comment\n% metadata\n@ITEM=1=a\n\n3 1 1\n0 7\n")
        @test read_spmf(input) == [[1, 3], [0, 7]]
        # A BOM is permitted only at the start of the file. Embedded BOMs
        # must not silently turn two numeric fragments into another item ID.
        write(input, "1\ufeff2 3\n")
        @test_throws ArgumentError read_spmf(input)
        for token in ("bad", "-1", "1.5")
            write(input, "# comment\n1 2\n2 $token\n")
            error = try read_spmf(input); nothing catch exc; exc end
            @test error isa ArgumentError
            @test occursin("line 3", sprint(showerror, error))
        end
        write_spmf(Dict((2, 3) => 4, (3,) => 5, (1,) => 6), output)
        @test read(output, String) == "1 #SUP: 6\n3 #SUP: 5\n2 3 #SUP: 4\n"

        write(input, "1 3 4\n2 3 5\n1 2 3 5\n2 5\n")
        for counter in ("scan", "hash-tree")
            status, out, err = invoke_cli(input, output, "--minsup", "0.5", "--counter", counter, "--trace")
            @test status == 0
            @test isempty(out)
            @test occursin("k=3 candidates=1 frequent=1", err)
            @test read(output, String) == EXPECTED_SPMF
        end
        status, out, err = invoke_cli(input, output, "--min-count", "2")
        @test status == 0
        @test isempty(err)
        @test read(output, String) == EXPECTED_SPMF

        original = read(input, String)
        for alias in (input, joinpath(scratch, "unused", "..", "input.txt"))
            status, _, err = invoke_cli(input, alias, "--min-count", "2")
            @test status != 0
            @test occursin("same file", err)
            @test read(input, String) == original
        end
        link = joinpath(scratch, "hardlink.txt")
        hardlink(input, link)
        status, _, err = invoke_cli(input, link, "--min-count", "2")
        @test status != 0
        @test occursin("same file", err)
        @test read(input, String) == original

        write(output, "keep output")
        write(input, "1 2\nbad\n")
        status, _, err = invoke_cli(input, output, "--min-count", "2")
        @test status != 0
        @test occursin("line 2", err)
        @test read(output, String) == "keep output"
        # Valid input isolates option/threshold validation from parser errors.
        write(input, original)
        for options in (("--minsup", "NaN"), ("--minsup", "0"), ("--min-count", "0"),
                        ("--minsup", "0.5", "--min-count", "2"), (), ("--other", "1"))
            status, _, err = invoke_cli(input, output, options...)
            @test status != 0
            @test !occursin("line 2", err)
            @test read(output, String) == "keep output"
        end
    end
    out, err = IOBuffer(), IOBuffer()
    @test main(["--help"]; out=out, err=err) == 0
    @test occursin("--minsup", String(take!(out)))
    @test isempty(String(take!(err)))
end
''',
    "test/core.jl": r'''const PAPER_ROWS = [[1, 3, 4], [2, 3, 5], [1, 2, 3, 5], [2, 5]]
const PAPER_ANSWER = Dict{Tuple{Vararg{Int}},Int}(
    (1,) => 2, (2,) => 3, (3,) => 3, (5,) => 3,
    (1, 3) => 2, (2, 3) => 2, (2, 5) => 3, (3, 5) => 2,
    (2, 3, 5) => 2,
)

"""Independent oracle enumerates every subset; no production helpers."""
function exhaustive(rows, threshold)
    sets = Set{Int}[Set(row) for row in rows]
    universe = sort!(collect(union(Set{Int}(), sets...)))
    answer = Dict{Tuple{Vararg{Int}},Int}()
    for mask in 1:((1 << length(universe)) - 1)
        candidate = Tuple(universe[i] for i in eachindex(universe) if !iszero(mask & (1 << (i - 1))))
        count = sum(all(item in row for item in candidate) for row in sets; init=0)
        count >= threshold && (answer[candidate] = count)
    end
    return answer
end

@testset "Figure 3: all 9 itemsets and support" begin
    for counter in (:scan, :hash_tree)
        result = mine(PAPER_ROWS; min_count=2, counter=counter)
        @test result.itemsets == PAPER_ANSWER
        @test result.transaction_count == 4
        @test result.min_support_count == 2
        @test [(s.size, s.candidates, s.frequent) for s in result.levels] ==
              [(1, 5, 4), (2, 6, 4), (3, 1, 1)]
    end
end

@testset "Paper join/prune" begin
    previous = [(1, 2, 3), (1, 2, 4), (1, 3, 4), (1, 3, 5), (2, 3, 4)]
    @test apriori_gen(previous) == [(1, 2, 3, 4)]
    @test apriori_gen([(1, 2), (2, 3)]) == NTuple{3,Int}[]
    @test apriori_gen(NTuple{2,Int}[]) == NTuple{3,Int}[]
    @test eltype(apriori_gen([(1,), (2,), (3,)])) == NTuple{2,Int}
end

@testset "Five databases vs independent oracle" begin
    databases = [PAPER_ROWS,
        [[3, 1, 1], [1, 3], [2], Int[]],
        [[1], [2], [3], [4]],
        [[0, 1, 2], [0, 1, 2], [0, 1], [1, 2]],
        [[0, 10^12], [10^12], [0, 9], [9, 10^12]],
    ]
    for rows in databases, threshold in (1, 2, length(rows), length(rows) + 1)
        expected = exhaustive(rows, threshold)
        for counter in (:scan, :hash_tree)
            @test mine(rows; min_count=threshold, counter=counter).itemsets == expected
        end
    end
end

@testset "One-shot generators, duplicates and empty rows" begin
    rows = ((item for item in row) for row in [[3, 1, 1], [1, 3], Int[]])
    result = mine(rows; min_support="0.5")
    @test result.transaction_count == 3
    @test result.itemsets == Dict((1,) => 2, (3,) => 2, (1, 3) => 2)
    @test isempty(mine(Vector{Int}[]; min_count=1).itemsets)
    @test mine(Vector{Int}[]; min_support="0.5").min_support_count == 0
    @test isempty(mine([Int[]]; min_count=1).itemsets)
    @test isempty(mine([[1]]; min_count=2).itemsets)
end

@testset "Exact thresholds and inclusive support" begin
    rows = vcat(fill([1], 7), fill([2], 93))
    for value in ("0.07", "7e-2", 0.07, 7 // 100)
        result = mine(rows; min_support=value)
        @test result.min_support_count == 7
        @test result.itemsets == Dict((1,) => 7, (2,) => 93)
    end
    for value in ("0.34", 0.34, 34 // 100)
        result = mine([[1], [1], [2]]; min_support=value)
        @test result.min_support_count == 2
        @test result.itemsets == Dict((1,) => 2)
    end
end

@testset "Invalid values are not silently coerced" begin
    invalid = [NamedTuple(), (; min_count=true), (; min_count=0), (; min_count=-1),
        (; min_count=1.5), (; min_count="2"), (; min_support=true),
        (; min_support=0), (; min_support=-0.1), (; min_support=1.1),
        (; min_support="NaN"), (; min_support=Inf), (; min_support="abc"),
        (; min_support=""), (; min_support="0.5", min_count=1),
        (; min_count=big(typemax(Int)) + 1)]
    for kwargs in invalid
        @test_throws ArgumentError mine(Vector{Int}[]; kwargs...)
    end
    for row in ([true], [-1], [1.5], ["1"], [big(typemax(Int)) + 1])
        @test_throws ArgumentError mine([row]; min_count=1)
    end
    @test_throws ArgumentError mine(Vector{Int}[]; min_count=1, counter=:unknown)
end

@testset "Seeded randomized databases" begin
    rng = MersenneTwister(20261116)
    for trial in 1:30
        rows = [[i for i in 0:6 if rand(rng) < 0.6] for _ in 1:8]
        threshold = rand(rng, 1:8)
        expected = exhaustive(rows, threshold)
        for counter in (:scan, :hash_tree)
            @test mine(rows; min_count=threshold, counter=counter).itemsets == expected
        end
    end
end
''',
    "test/explain.jl": r'''function explain_run(rows; counter=:hash_tree, limit=5000, kwargs...)
    stream = IOBuffer()
    result = mine(rows; counter, explain=ExplainTrace(stream; limit), kwargs...)
    return result, String(take!(stream))
end

@testset "Actual Apriori explanation" begin
    for counter in (:scan, :hash_tree)
        result, log = explain_run(PAPER_ROWS; min_count=2, counter)
        @test result.itemsets == PAPER_ANSWER
        @test occursin("N=4; min_count=2", log)
        @test occursin("JOIN {2,3} + {2,5} -> {2,3,5}", log)
        @test occursin("T1 chứa {1,3}: count 0 -> 1", log)
        @test occursin("T3 chứa {1,3}: count 1 -> 2", log)
        @test occursin("LOẠI {1,2}: support=1 < 2", log)
        @test occursin("GIỮ {2,3,5}: support=2 >= 2", log)
        @test occursin("C_4 rỗng", log)
        @test occursin("KẾT THÚC: 9 frequent itemsets", log)
        expected_levels = mine(PAPER_ROWS; min_count=2, counter).levels
        @test [(s.size,s.candidates,s.frequent) for s in result.levels] ==
              [(s.size,s.candidates,s.frequent) for s in expected_levels]
    end

    fixture = [[1,2],[1,2],[1,3],[1,3]]
    result, log = explain_run(fixture; min_count=2)
    @test result.itemsets == exhaustive(fixture, 2)
    @test occursin("JOIN {1,2} + {1,3} -> {1,2,3}", log)
    @test occursin("PRUNE {1,2,3}: thiếu tập con {2,3} trong L_2", log)
    @test occursin("C_3 rỗng", log)

    kit = read_spmf(joinpath(@__DIR__, "..", "data", "test-kit", "datasets", "01_spmf_example.txt"))
    for counter in (:scan, :hash_tree)
        result, log = explain_run(kit; min_support="0.40", counter)
        @test length(result.itemsets) == 15
        @test occursin("ceil(0.40 × 5) = 2", log)
        @test occursin("T5 chứa {1,2,3,5}: count 1 -> 2", log)
        @test occursin("C_5 rỗng", log)
    end
    _, first = explain_run(PAPER_ROWS; min_count=2)
    _, second = explain_run(PAPER_ROWS; min_count=2)
    @test first == second

    # Ten colliding pairs force the default hash-tree to split. Trace only
    # records candidates actually counted, once per transaction.
    ids = [0,17,34,51,68]
    candidates = [(ids[i],ids[j]) for i in 1:4 for j in (i+1):5]
    stream = IOBuffer()
    trace = ExplainTrace(stream)
    counts = FrequentItemsetMining.count_candidates(candidates,[ids,ids],Val(:hash_tree),trace)
    @test counts == fill(2,10)
    log = String(take!(stream))
    for candidate in candidates
        text = "{" * join(candidate,',') * "}"
        @test length(findall("T1 chứa $text: count 0 -> 1",log)) == 1
        @test length(findall("T2 chứa $text: count 1 -> 2",log)) == 1
    end
end

@testset "Explanation limits and stopping conditions" begin
    result, log = explain_run(PAPER_ROWS; min_count=2, limit=3)
    @test result.itemsets == PAPER_ANSWER
    @test occursin("đã đạt giới hạn 3", log)
    @test occursin("vẫn tiếp tục tính đầy đủ", log)
    @test occursin("KẾT THÚC: 9 frequent itemsets", log)
    for invalid in (0, -1, true, 1.5)
        @test_throws ArgumentError ExplainTrace(IOBuffer(); limit=invalid)
    end
    for (rows, threshold, reason) in ((Vector{Int}[],1,"CSDL rỗng"),
                                      ([[1],[2]],2,"L_1 rỗng"),
                                      ([[1],[1],[2],[2]],2,"L_2 rỗng"))
        _, log = explain_run(rows; min_count=threshold)
        @test occursin(reason, log)
        @test occursin("KẾT THÚC", log)
    end
end

@testset "Explanation CLI and file preservation" begin
    mktempdir() do scratch
        input, output, logpath = joinpath.(scratch, ("input.txt", "output.txt", "explanation.txt"))
        write(input, "1 3 4\n2 3 5\n1 2 3 5\n2 5\n")
        status, out, err = invoke_cli(input, output, "--min-count", "2", "--explain")
        @test status == 0
        @test isempty(out)
        @test occursin("JOIN", err)
        @test read(output,String) == EXPECTED_SPMF
        status, _, err = invoke_cli(input, output, "--min-count", "2", "--explain-file", logpath)
        @test status == 0
        @test isempty(err)
        @test occursin("KẾT THÚC", read(logpath,String))
        @test read(output,String) == EXPECTED_SPMF
        status, _, _ = invoke_cli(input, output, "--min-count", "2", "--explain-file", logpath, "--explain-limit", "2")
        @test status == 0
        @test occursin("đã đạt giới hạn 2", read(logpath,String))

        original_input, original_output = read(input,String), read(output,String)
        for alias in (input, output, joinpath(scratch,"unused","..","input.txt"))
            status, _, err = invoke_cli(input, output, "--min-count", "2", "--explain-file", alias)
            @test status != 0
            @test occursin("same file", err)
            @test read(input,String) == original_input
            @test read(output,String) == original_output
        end
        link = joinpath(scratch,"linked-input.txt")
        hardlink(input,link)
        status, _, err = invoke_cli(input, output, "--min-count", "2", "--explain-file", link)
        @test status != 0
        @test occursin("same file", err)
        @test read(input,String) == original_input

        write(logpath,"keep log")
        for options in (("--minsup","NaN"), ("--min-count","0"),
                        ("--min-count","2","--counter","unknown"),
                        ("--min-count","2","--explain-limit","0"))
            status, _, _ = invoke_cli(input, output, options..., "--explain-file", logpath)
            @test status != 0
            @test read(logpath,String) == "keep log"
            @test read(output,String) == original_output
        end
        write(input,"bad\n")
        status, _, _ = invoke_cli(input, output, "--min-count","2","--explain-file",logpath)
        @test status != 0
        @test read(logpath,String) == "keep log"
    end
end
''',
    "test/hash_tree.jl": r'''@testset "Hash-tree candidate IDs" begin
    @testset "terminal collisions require complete candidate checks" begin
        candidates = [(0, 2), (0, 4), (2, 4), (2, 6)]
        tree = HashTree(candidates; bucket_count=1, leaf_capacity=1)
        @test sort(matching_ids(tree, [0, 2, 6])) == [1, 4]
        @test sort(matching_ids(tree, [0, 2, 4, 6])) == [1, 2, 3, 4]
        @test isempty(matching_ids(tree, [0]))
        @test isempty(matching_ids(tree, Int[]))
    end

    @testset "internal nodes may be revisited for different prefixes" begin
        candidates = [(0, 1, 2), (0, 1, 4), (0, 2, 3), (1, 3, 5),
                      (2, 3, 4), (2, 4, 6), (3, 5, 7)]
        tree = HashTree(candidates; bucket_count=2, leaf_capacity=1)
        @test sort(matching_ids(tree, [0, 1, 2, 3, 4, 5, 6, 7])) == collect(1:7)
        @test sort(matching_ids(tree, [0, 2, 3, 4, 6])) == [3, 5, 6]
        @test sort(matching_ids(tree, [1, 3, 5, 7])) == [4, 7]
    end

    @testset "workspace reset and independent convenience results" begin
        tree = HashTree([(0, 2), (0, 4), (2, 4)]; leaf_capacity=1)
        workspace = HashQueryWorkspace(tree)
        first_result = matching_ids!(workspace, tree, [0, 2, 4])
        @test sort(first_result) == [1, 2, 3]
        second_result = matching_ids!(workspace, tree, [0, 4])
        @test second_result === first_result
        @test second_result == [2]
        @test isempty(matching_ids!(workspace, tree, [2]))
        @test matching_ids!(workspace, tree, [2, 4]) == [3]
        owned_result = matching_ids(tree, [0, 2])
        @test owned_result == [1]
        @test matching_ids(tree, [0, 4]) == [2]
        @test owned_result == [1]
        @test matching_ids!(workspace, tree, [0, 2]) == [1]
    end

    @testset "sparse IDs use bounded membership storage" begin
        huge = typemax(Int)
        tree = HashTree([(0, huge - 2), (huge - 2, huge)]; bucket_count=2,
                        leaf_capacity=1)
        workspace = HashQueryWorkspace(tree)
        @test matching_ids!(workspace, tree, [0, huge - 2]) == [1]
        @test matching_ids!(workspace, tree, [huge - 2, huge]) == [2]
        @test isempty(matching_ids!(workspace, tree, [0, huge - 1, huge]))
        @test Base.summarysize(workspace) < 100_000
    end

    @testset "empty typed candidate collections" begin
        tree = HashTree(NTuple{3,Int}[]; bucket_count=1, leaf_capacity=1)
        workspace = HashQueryWorkspace(tree)
        @test isempty(matching_ids(tree, [0, 1, 2]))
        @test isempty(matching_ids!(workspace, tree, Int[]))
    end

    @testset "constructor validates candidates and configuration" begin
        for invalid in (0, -1, true, 1.5, big(typemax(Int)) + 1)
            @test_throws ArgumentError HashTree([(0, 1)]; bucket_count=invalid)
            @test_throws ArgumentError HashTree([(0, 1)]; leaf_capacity=invalid)
        end
        @test_throws ArgumentError HashTree([(-1, 2)])
        @test_throws ArgumentError HashTree([(2, 1)])
        @test_throws ArgumentError HashTree([(1, 1)])
        @test_throws ArgumentError HashTree([()])
        @test_throws ArgumentError HashTree([(true, 2)])
        @test_throws ArgumentError HashTree([(0, 1), (0, 1, 2)])
        @test_throws ArgumentError HashTree([(0, big(typemax(Int)) + 1)])
    end

    @testset "subset oracle across collisions and leaf sizes" begin
        candidates = NTuple{3,Int}[(a, b, c) for a in 0:3 for b in a+1:4 for c in b+1:5]
        for bucket_count in (1, 2, 17), leaf_capacity in (1, 3, 100)
            tree = HashTree(candidates; bucket_count, leaf_capacity)
            workspace = HashQueryWorkspace(tree)
            for mask in 0:127
                transaction = Int[item for item in 0:6 if !iszero(mask & (1 << item))]
                expected = Int[id for (id, candidate) in enumerate(candidates)
                               if all(item in transaction for item in candidate)]
                actual = matching_ids!(workspace, tree, transaction)
                @test sort(actual) == expected
                @test length(actual) == length(unique(actual))
            end
        end
    end
end
''',
    "test/report_cli.jl": r'''using TOML

function report_cli(input, output, options...)
    out, err = IOBuffer(), IOBuffer()
    status = main(["--input", input, "--output", output, options...]; out, err)
    return status, String(take!(out)), String(take!(err))
end

@testset "CLI run history and timing context" begin
    mktempdir() do scratch
        input, output = joinpath.(scratch, ("input.txt", "current.txt"))
        write(input, "1 2\n1\n")
        status, out, err = report_cli(input, output, "--minsup", "0.5", "--warmup", "1")
        @test status == 0
        @test isempty(err)
        @test occursin("Report:", out)
        @test occursin("MiningMs:", out)
        runs = joinpath(scratch, "runs")
        @test isdir(runs)
        if isdir(runs)
            first_run = only(readdir(runs; join=true))
            @test basename(first_run) == "input-1"
            @test all(isfile(joinpath(first_run, file)) for file in
                      ("itemsets.txt", "report.txt", "timings.csv", "metadata.toml"))
            @test read(joinpath(first_run, "itemsets.txt"), String) == "1 #SUP: 2\n2 #SUP: 1\n1 2 #SUP: 1\n"
            metadata = TOML.parsefile(joinpath(first_run, "metadata.toml"))
            @test metadata["warmup_runs"] == 1
            @test metadata["explain_enabled"] == false
            @test metadata["minsup_argument"] == "0.5"
            @test metadata["input_hash_scope"] == "exact_bytes_parsed"
            @test metadata["input_sha256"] == bytes2hex(FrequentItemsetMining.SHA.sha256(read(input)))
            @test metadata["input_changed_after_read"] == false
            original = Dict(file => read(joinpath(first_run, file)) for file in readdir(first_run))
            status, out, err = report_cli(input, output, "--min-count", "2", "--quiet")
            @test status == 0
            @test isempty(out) && isempty(err)
            @test length(readdir(runs)) == 2
            @test isdir(joinpath(runs, "input-2"))
            @test read(output, String) == "1 #SUP: 2\n"
            @test all(read(joinpath(first_run, file)) == bytes for (file, bytes) in original)
        end
    end
end

@testset "Report numbering follows each dataset" begin
    mktempdir() do scratch
        cd(scratch) do
            runs = joinpath("output", "runs")
            mkpath(joinpath(runs, "retail-5pct-old-uuid"))
            for dataset in ("retail", "other")
                write("$dataset.txt", "1 2\n1\n")
            end
            run_dataset(name, count) = main(["--input", "$name.txt", "--min-count", count, "--quiet"]; err=devnull)
            @test run_dataset("retail", "3") == 0
            @test run_dataset("other", "2") == 0
            @test run_dataset("retail", "1") == 0
            @test isfile(joinpath(runs, "retail-1", "itemsets.txt"))
            @test isfile(joinpath(runs, "retail-2", "report.txt"))
            @test isfile(joinpath(runs, "other-1", "report.txt"))
            mkdir(joinpath(runs, "retail-10"))
            @test run_dataset("retail", "0") != 0
            @test !ispath(joinpath(runs, "retail-11"))
            @test run_dataset("retail", "2") == 0
            @test isfile(joinpath(runs, "retail-11", "report.txt"))
        end
    end
end

@testset "A concurrent claim keeps its directory" begin
    mktempdir() do parent
        first_run = joinpath(parent, "batch-1")
        function race_with_first_claim(directory)
            if directory == first_run
                mkdir(first_run)
                write(joinpath(first_run, "owner.txt"), "first")
            end
        end
        run = FrequentItemsetMining.numbered_run_directory(parent, "batch"; validate=race_with_first_claim)
        @test run == joinpath(parent, "batch-2")
        @test isdir(run)
        @test read(joinpath(first_run, "owner.txt"), String) == "first"
    end
end

@testset "CLI explicit report destination and prevalidation" begin
    mktempdir() do scratch
        input = joinpath(scratch, "input.txt")
        write(input, "1 2\n1\n")
        bundle = joinpath(scratch, "nested", "one-run")
        output, trace = joinpath.(bundle, ("itemsets.txt", "trace.txt"))
        status, out, err = report_cli(input, output, "--min-count", "1",
            "--report-dir", bundle, "--explain-file", trace)
        @test status == 0
        @test isempty(err)
        @test occursin(abspath(bundle), out)
        if isdir(bundle)
            @test occursin("JOIN", read(trace, String))
            metadata = TOML.parsefile(joinpath(bundle, "metadata.toml"))
            @test metadata["explain_enabled"] == true
            @test metadata["minsup_argument"] == ""
            original = Dict(file => read(joinpath(bundle, file)) for file in readdir(bundle))
            status, out, err = report_cli(input, output, "--min-count", "2", "--report-dir", bundle)
            @test status != 0
            @test isempty(out)
            @test occursin("exist", err)
            @test all(read(joinpath(bundle, file)) == bytes for (file, bytes) in original)
        end

        existing_output = joinpath(scratch, "current.txt")
        write(existing_output, "preserve me")
        for options in (("--warmup", "-1"), ("--warmup", "bad"),
                        ("--warmup", "999999999999999999999999999"),
                        ("--counter", "unknown"), ("--explain-limit", "0"))
            target = joinpath(scratch, "invalid-run")
            status, out, err = report_cli(input, existing_output, "--min-count", "1",
                "--report-dir", target, "--quiet", options...)
            @test status != 0
            @test isempty(out) && !isempty(err)
            @test !ispath(target)
            @test read(existing_output, String) == "preserve me"
        end
        for reserved in ("report.txt", "timings.csv", "metadata.toml", "trace.txt")
            target = joinpath(scratch, "reserved-run")
            status, out, err = report_cli(input, joinpath(target, reserved), "--min-count", "1",
                "--report-dir", target)
            @test status != 0
            @test isempty(out) && !isempty(err)
            @test !ispath(target)
        end
        target = joinpath(scratch, "reserved-log")
        status, _, _ = report_cli(input, existing_output, "--min-count", "1",
            "--report-dir", target, "--explain-file", joinpath(target, "itemsets.txt"))
        @test status != 0
        @test !ispath(target)
        @test read(existing_output, String) == "preserve me"

        target = joinpath(scratch, "file-parent-run")
        status, out, err = report_cli(input, joinpath(target, "other.txt"), "--min-count", "1",
            "--report-dir", target, "--explain-file", joinpath(target, "other.txt", "trace.txt"))
        @test status != 0
        @test isempty(out) && !isempty(err)
        @test !ispath(target)

        blocked_output=joinpath(scratch,"future-output")
        kept_log=joinpath(scratch,"kept-log.txt")
        write(kept_log,"keep explanation")
        status, _, _ = report_cli(input,blocked_output,"--min-count","1",
            "--report-dir",joinpath(blocked_output,"run"),"--explain-file",kept_log)
        @test status != 0
        @test !ispath(blocked_output)
        @test read(kept_log,String)=="keep explanation"

        # Symlink creation may require privileges on Windows. If available,
        # verify a dangling destination cannot become a reserved report file.
        link=joinpath(scratch,"dangling-output")
        dangling_run=joinpath(scratch,"dangling-run")
        link_created=false
        try
            symlink(joinpath(dangling_run,"report.txt"),link)
            link_created=true
        catch exc
            exc isa Base.IOError || exc isa SystemError || rethrow()
        end
        if link_created
            try
                write(kept_log,"keep explanation")
                status, _, _ = report_cli(input,link,"--min-count","1",
                    "--report-dir",dangling_run,"--explain-file",kept_log)
                @test status != 0
                @test !ispath(dangling_run)
                @test read(kept_log,String)=="keep explanation"
            finally
                # Remove only the known link itself before temporary cleanup.
                islink(link) && rm(link)
            end
        end

        target = joinpath(scratch, "malformed-run")
        write(input, "invalid\n")
        status, _, _ = report_cli(input, existing_output, "--min-count", "1", "--report-dir", target)
        @test status != 0
        @test !ispath(target)
        @test read(existing_output, String) == "preserve me"
    end
end
''',
    "test/reports.jl": r'''using TOML

function report_metadata(input; id="writer-test", explain=false)
    Dict{String,Any}("run_id"=>id, "started_utc"=>"2026-11-17T00:00:00.000Z",
        "input_path"=>input, "minsup_argument"=>"0.5", "counter"=>"hash-tree",
        "warmup_runs"=>0, "explain_enabled"=>explain, "explain_limit"=>5000)
end

@testset "Persistent measured run reports" begin
    mktempdir() do scratch
        input, output = joinpath.(scratch,("input.txt","output.txt"))
        write(input,"1 3 4\n2 3 5\n1 2 3 5\n2 5\n")
        result=mine(read_spmf(input);min_support="0.5")
        write_spmf(result.itemsets,output)
        bundle=mkdir(joinpath(scratch,"run1"))
        FrequentItemsetMining.write_run_report(bundle,result; metadata=report_metadata(input),
            output_path=output, read_ns=UInt64(100),write_ns=UInt64(200),cli_ns=UInt64(300))
        @test read(joinpath(bundle,"itemsets.txt"),String)==EXPECTED_SPMF
        report=read(joinpath(bundle,"report.txt"),String)
        @test occursin("writer-test",report)
        @test occursin("{2,3,5}",report)
        @test occursin("support_count",report)
        @test occursin("normalize",report)
        @test occursin("generate_candidates",report)
        @test occursin("JIT",report)
        @test occursin("không cộng",report)
        meta=TOML.parsefile(joinpath(bundle,"metadata.toml"))
        @test meta["schema_version"]==1
        @test meta["run_id"]=="writer-test"
        @test meta["transactions"]==4
        @test meta["min_support_count"]==2
        @test meta["itemset_count"]==9
        @test meta["input_sha256"]==bytes2hex(FrequentItemsetMining.SHA.sha256(read(input)))
        @test length(meta["implementation_sha256"])==64
        @test meta["mining_ns"]==Int(result.mining_ns)
        @test meta["read_ns"]==100
        @test meta["write_ns"]==200
        @test meta["cli_ns"]==300
        @test meta["warmup_runs"]==0
        @test meta["explain_enabled"]==false
        @test meta["timing_policy"]=="cold_call_may_include_jit"
        lines=readlines(joinpath(bundle,"timings.csv"))
        @test occursin("elapsed_ns",lines[1])
        @test occursin("elapsed_ms",lines[1])
        @test occursin("phase",lines[1])
        @test any(line -> occursin("mining_total",line),lines)
        @test any(line -> occursin("read_input",line),lines)
        @test any(line -> occursin("level_total",line),lines)
        original=read(joinpath(bundle,"report.txt"))
        @test_throws ArgumentError FrequentItemsetMining.write_run_report(bundle,result;
            metadata=report_metadata(input),output_path=output)
        @test read(joinpath(bundle,"report.txt"))==original
        # The snapshot remains independent when the latest output is replaced.
        write(output,"a later run\n")
        @test read(joinpath(bundle,"itemsets.txt"),String)==EXPECTED_SPMF

        bundle2=mkdir(joinpath(scratch,"run2"))
        tracepath=joinpath(bundle2,"trace.txt")
        explain_result=open(tracepath,"w") do stream
            mine(PAPER_ROWS;min_count=2,explain=ExplainTrace(stream))
        end
        itempath=joinpath(bundle2,"itemsets.txt")
        write_spmf(explain_result.itemsets,itempath)
        meta2=report_metadata(input;id="writer-test-2",explain=true)
        meta2["warmup_runs"]=1
        FrequentItemsetMining.write_run_report(bundle2,explain_result;metadata=meta2,
            output_path=itempath,explanation_path=tracepath,warmup_ns=UInt64(50))
        @test occursin("KẾT THÚC",read(tracepath,String))
        saved=TOML.parsefile(joinpath(bundle2,"metadata.toml"))
        @test saved["explain_enabled"]==true
        @test saved["warmup_runs"]==1
        @test saved["timing_policy"]=="warmup_requested_jit_may_remain"
        @test saved["trace_path"]==abspath(tracepath)
        @test isfile(joinpath(bundle,"report.txt"))

        original_bytes=read(input)
        captured=report_metadata(input;id="snapshot-test")
        captured["input_sha256"]=bytes2hex(FrequentItemsetMining.SHA.sha256(original_bytes))
        captured["input_bytes"]=length(original_bytes)
        captured["input_hash_scope"]="exact_bytes_parsed"
        captured["implementation_sha256"]=repeat("0",64)
        write(input,"99\n")
        bundle3=mkdir(joinpath(scratch,"run3"))
        FrequentItemsetMining.write_run_report(bundle3,result;metadata=captured,output_path=output)
        snapshot_meta=TOML.parsefile(joinpath(bundle3,"metadata.toml"))
        @test snapshot_meta["input_sha256"]==captured["input_sha256"]
        @test snapshot_meta["input_bytes"]==length(original_bytes)
        @test get(snapshot_meta,"input_changed_after_read",false)
        @test snapshot_meta["implementation_sha256"]==repeat("0",64)
        @test get(snapshot_meta,"implementation_changed_during_run",false)
        @test read(joinpath(bundle3,"itemsets.txt"),String)==EXPECTED_SPMF
    end
end
''',
    "test/rules_tests.jl": r'''using Test
using Random
include(joinpath(@__DIR__, "..", "src", "AssociationRules.jl"))
using .AssociationRules

# Ví dụ của bài báo Agrawal & Srikant (min count = 2, N = 4)
const PAPER_COUNTS = Dict{Vector{Int},Int}(
    [1] => 2, [2] => 3, [3] => 3, [5] => 3,
    [1, 3] => 2, [2, 3] => 2, [2, 5] => 3, [3, 5] => 2,
    [2, 3, 5] => 2,
)

key(r::Rule) = (r.antecedent, r.consequent)

@testset "AssociationRules" begin
    @testset "đọc file itemsets và đếm transaction" begin
        mktempdir() do d
            f = joinpath(d, "itemsets.txt")
            write(f, "@CONVERTED_FROM_TEXT\n\n5 2 #SUP: 3\n1 #SUP: 2\n2 3 5 #SUP: 2\n")
            c = read_itemsets(f)
            @test c == Dict([2, 5] => 3, [1] => 2, [2, 3, 5] => 2)

            bad = joinpath(d, "bad.txt")
            write(bad, "1 2 3\n")
            @test_throws ArgumentError read_itemsets(bad)

            g = joinpath(d, "data.txt")
            write(g, "# ghi chú\n1 3 4\n\n2 3 5\n@meta\n1 2 3 5\n2 5\n")
            @test count_transactions(g) == 4
        end
    end

    @testset "ví dụ bài báo" begin
        all_rules = generate_rules(PAPER_COUNTS, 4; minconf=0.0)
        @test length(all_rules) == 14             # 4 cặp × 2 + 1 bộ ba × 6

        r = generate_rules(PAPER_COUNTS, 4; minconf=1.0)
        @test Set(key.(r)) == Set([([1], [3]), ([2], [5]), ([5], [2]),
                                   ([2, 3], [5]), ([3, 5], [2])])
        @test all(x -> x.confidence == 1.0, r)
        @test all(x -> isapprox(x.lift, 4 / 3), r)
        # support cao nhất đứng trước, hòa thì antecedent tăng dần
        @test key(r[1]) == ([2], [5])
        @test key(r[2]) == ([5], [2])
        @test r[1].support == 0.75
        @test r[1].count == 3

        # {5} ⇒ {3}: conf = 2/3, lift = (2/3) / (3/4) = 8/9 < 1
        x = only(filter(y -> key(y) == ([5], [3]), all_rules))
        @test isapprox(x.confidence, 2 / 3)
        @test isapprox(x.lift, 8 / 9)
        @test x.support == 0.5

        # lọc theo lift
        pos = generate_rules(PAPER_COUNTS, 4; minconf=0.0, minlift=1.0)
        @test all(x -> x.lift >= 1.0, pos)
        @test length(pos) < length(all_rules)
    end

    @testset "đối chiếu brute-force trên dữ liệu ngẫu nhiên (seed cố định)" begin
        rng = MersenneTwister(42)
        nitems, ntrans = 8, 60
        trans = [sort!(collect(Iterators.take(randperm(rng, nitems), rand(rng, 2:5))))
                 for _ in 1:ntrans]
        supcount(set) = count(t -> all(in(t), set), trans)

        counts = Dict{Vector{Int},Int}()
        for mask in 1:(1 << nitems) - 1
            set = [i for i in 1:nitems if (mask >> (i - 1)) & 1 == 1]
            c = supcount(set)
            c >= 6 && (counts[set] = c)
        end

        minconf = 0.4
        rules = generate_rules(counts, ntrans; minconf=minconf)

        # đếm luật kỳ vọng bằng cách duyệt mọi cặp (X, Y) rời nhau
        expected = 0
        for (z, cz) in counts, mask in 1:(1 << length(z)) - 2
            x = [z[j] for j in eachindex(z) if (mask >> (j - 1)) & 1 == 1]
            cz / supcount(x) >= minconf - 1e-12 && (expected += 1)
        end
        @test length(rules) == expected
        @test !isempty(rules)

        for r in rules
            z = sort!(vcat(r.antecedent, r.consequent))
            @test isempty(intersect(r.antecedent, r.consequent))
            @test r.count == supcount(z)
            @test r.confidence ≈ supcount(z) / supcount(r.antecedent)
            @test r.lift ≈ (supcount(z) / ntrans) /
                           ((supcount(r.antecedent) / ntrans) * (supcount(r.consequent) / ntrans))
        end
        @test issorted(rules; by=r -> -r.lift)

        # chạy lại cho đúng cùng kết quả (tái lập)
        again = generate_rules(counts, ntrans; minconf=minconf)
        @test key.(again) == key.(rules)
        @test [r.lift for r in again] == [r.lift for r in rules]
    end

    @testset "ghi CSV" begin
        mktempdir() do d
            p = write_rules_csv(joinpath(d, "sub", "rules.csv"),
                                generate_rules(PAPER_COUNTS, 4; minconf=1.0))
            lines = readlines(p)
            @test lines[1] == "antecedent,consequent,count,support,confidence,lift"
            @test length(lines) == 6
            @test startswith(lines[2], "2,5,3,0.75,1.0,")
        end
    end

    @testset "kiểm tra lỗi đầu vào" begin
        @test_throws ArgumentError generate_rules(PAPER_COUNTS, 0)
        @test_throws ArgumentError generate_rules(PAPER_COUNTS, 4; minconf=1.5)
        @test_throws ArgumentError generate_rules(PAPER_COUNTS, 4; minlift=-1)
        thieu = Dict{Vector{Int},Int}([1, 2] => 3, [1] => 3)   # thiếu support của {2}
        @test_throws ArgumentError generate_rules(thieu, 5)
    end
end
''',
    "test/runtests.jl": r'''using Test
using Random

include(joinpath(@__DIR__, "..", "src", "FrequentItemsetMining.jl"))
using .FrequentItemsetMining

@testset "FrequentItemsetMining" begin
    include("core.jl")
    include("hash_tree.jl")
    include("cli.jl")
    include("explain.jl")
    include("timing.jl")
    include("reports.jl")
    include("report_cli.jl")
    include("market_basket.jl")
    include("splice_junction.jl")
end

include("spmf_test_kit.jl")
include("rules_tests.jl")
include("apriori_optimized.jl")
include("benchmark_optimized_sampling.jl")
''',
    "test/splice_junction.jl": r'''using Test
include(joinpath(@__DIR__, "..", "apps", "splice-junction", "main.jl"))

@testset "Splice junctions: coordinates, unknown bases and group support" begin
    sequence(window) = "A"^26 * window * "T"^26
    @test SpliceJunction.item_label.(SpliceJunction.encode(sequence("ACGTACGT"))) ==
          ["-4:A","-3:C","-2:G","-1:T","+1:A","+2:C","+3:G","+4:T"]
    @test isempty(SpliceJunction.encode(sequence("RSDNNSDR")))
    @test_throws ArgumentError SpliceJunction.encode("ACGT")
    @test_throws ArgumentError SpliceJunction.encode(sequence("AXGTACGT"))
    mktempdir() do dir
        path = joinpath(dir, "splice.data")
        open(path, "w") do f
            for (group, windows) in [("EI", ["NNNACNNN","NNNACNNN"]),
                                     ("IE", ["NNNACNNN","NNNNNNNN","NNNNCNNN"]),
                                     ("N", ["NNNATNNN"])]
                for (i, window) in enumerate(windows)
                    println(f, group, ", sample", i, ", ", sequence(window))
                end
            end
        end
        root = joinpath(dir, "runs")
        @test SpliceJunction.main(["--input", path, "--minsup", "0.75"];
                                  output_root=root, out=devnull, err=devnull) == 0
        run = only(readdir(root; join=true))
        rows = readlines(joinpath(run, "patterns.csv"))
        fields = split(first(rows), ',')
        parsed = [Dict(zip(fields, split(row, ','))) for row in rows[2:end]]
        pair = only(filter(row -> row["pattern"] == "-1:A +1:C", parsed))
        @test [parse(Int,pair["count_" * g]) for g in ("EI","IE","N")] == [2,1,0]
        @test [parse(Float64,pair["support_" * g]) for g in ("EI","IE","N")] ≈ [1,1/3,0]
        @test occursin("IE=3", read(joinpath(run,"report.txt"),String))
        @test SpliceJunction.main(["--input",path,"--minsup","0"]; output_root=root,
                                  out=devnull, err=devnull) == 2
        write(path, "EI, id, " * sequence("ACGTACGT"))
        @test SpliceJunction.main(["--input",path]; output_root=root,out=devnull,err=devnull) == 2
        @test length(readdir(root)) == 1
    end
end
''',
    "test/timing.jl": r'''@testset "Measured mining phases preserve results and counted levels" begin
    @test isdefined(FrequentItemsetMining, :PhaseTiming)
    result = mine(PAPER_ROWS; min_count=2, counter=:scan)
    @test hasproperty(result, :timings)
    @test hasproperty(result, :mining_ns)
    if hasproperty(result, :timings) && hasproperty(result, :mining_ns)
        expected = [(0, :normalize, 0, 0), (0, :threshold, 0, 0),
                    (1, :singleton_count, 5, 0), (1, :filter, 5, 4), (1, :level_total, 5, 4),
                    (2, :generate_candidates, 6, 0), (2, :build_counter, 6, 0),
                    (2, :count_support, 6, 0), (2, :filter, 6, 4), (2, :level_total, 6, 4),
                    (3, :generate_candidates, 1, 0), (3, :build_counter, 1, 0),
                    (3, :count_support, 1, 0), (3, :filter, 1, 1), (3, :level_total, 1, 1),
                    (4, :generate_candidates, 0, 0), (4, :level_total, 0, 0)]
        for counter in (:scan, :hash_tree)
            actual = mine(PAPER_ROWS; min_support="0.5", counter)
            @test actual.itemsets == PAPER_ANSWER
            @test [(s.size, s.candidates, s.frequent) for s in actual.levels] ==
                  [(1, 5, 4), (2, 6, 4), (3, 1, 1)]
            @test [(t.size, t.phase, t.candidates, t.frequent) for t in actual.timings] == expected
            @test actual.timings isa Vector{FrequentItemsetMining.PhaseTiming}
            @test actual.mining_ns isa UInt64
            @test all(t.elapsed_ns isa UInt64 && t.elapsed_ns <= actual.mining_ns for t in actual.timings)
            for size in 1:4
                rows = filter(t -> t.size == size, actual.timings)
                total = only(filter(t -> t.phase == :level_total, rows)).elapsed_ns
                @test all(t.elapsed_ns <= total for t in rows)
            end
            builds = filter(t -> t.phase == :build_counter, actual.timings)
            if counter == :scan
                @test all(iszero(t.elapsed_ns) for t in builds)
            else
                # A measured tree/workspace build may round to zero on coarse clocks.
                @test all(t.elapsed_ns >= 0 for t in builds)
            end
        end
    end
end

@testset "Timing edge cases and compatibility" begin
    result = mine(Vector{Int}[]; min_count=1)
    @test hasproperty(result, :timings)
    if hasproperty(result, :timings)
        @test [(t.size, t.phase) for t in result.timings] == [(0, :normalize), (0, :threshold)]
        @test isempty(result.levels)
        @test all(t.elapsed_ns <= result.mining_ns for t in result.timings)
        for rows in ([Int[]], [[1]])
            no_frequent = mine(rows; min_count=2)
            @test [(t.size, t.phase) for t in no_frequent.timings] ==
                  [(0, :normalize), (0, :threshold), (1, :singleton_count), (1, :filter), (1, :level_total)]
            @test length(no_frequent.levels) == 1
        end
        # A counted but entirely rejected L2 remains a LevelStats entry.
        rejected = mine([[1], [2]]; min_count=1)
        @test [(s.size, s.candidates, s.frequent) for s in rejected.levels] == [(1, 2, 2), (2, 1, 0)]
        @test [(t.size, t.phase) for t in rejected.timings if t.phase == :level_total] ==
              [(1, :level_total), (2, :level_total)]
        old = MiningResult(copy(PAPER_ANSWER), 4, 2, LevelStats[])
        @test old.itemsets == PAPER_ANSWER
        @test isempty(old.timings)
        @test old.mining_ns === UInt64(0)
        independent = mine([[1]]; min_count=1)
        empty!(independent.timings)
        @test !isempty(mine([[1]]; min_count=1).timings)
    end
    candidates = [(1, 3), (2, 5)]
    for mode in (Val(:scan), Val(:hash_tree))
        @test (@inferred FrequentItemsetMining.count_candidates(candidates, PAPER_ROWS, mode)) == [2, 3]
    end
end
''',
    "apps/splice-junction/main.jl": r'''module SpliceJunction

import FrequentItemsetMining as FIM
using Printf: @printf

const ROOT = normpath(joinpath(@__DIR__, "..", ".."))
const GROUPS = ("EI", "IE", "N")
const BASES = "ACGT"
const POSITIONS = (-4, -3, -2, -1, 1, 2, 3, 4)
const HELP = """
    julia --project=. apps/splice-junction/main.jl [--minsup 0.20] [--input splice.data]
Apriori xét 4 nucleotide mỗi bên tâm đoạn. Minsup thuộc (0, 1] và áp dụng riêng cho EI, IE, N.
Kết quả mỗi lần chạy được lưu vào thư mục mới bên trong output/splice-junction/.
"""

function encode(sequence)
    (ncodeunits(sequence) == 60 && all(c -> c in "ACGTDRSN", sequence)) ||
        throw(ArgumentError("cần đúng 60 nucleotide theo bảng ký hiệu của bộ UCI"))
    items = Int[]
    # The boundary is between columns 30 and 31. No position is numbered zero.
    for (slot, base) in enumerate(sequence[27:34])
        index = findfirst(==(base), BASES)
        isnothing(index) || push!(items, 4 * (slot - 1) + index)
    end
    return items
end

function item_label(id)
    slot, base = divrem(id - 1, 4)
    pos = POSITIONS[slot + 1]
    return string(pos > 0 ? "+" : "", pos, ":", BASES[base + 1])
end

function read_groups(path)
    groups = Dict(g => Vector{Int}[] for g in GROUPS)
    open(path) do io
        for (line_num, line) in enumerate(eachline(io))
            isempty(strip(line)) && continue
            fields = strip.(split(line, ','))
            (length(fields) == 3 && fields[1] in GROUPS && !isempty(fields[2])) ||
                throw(ArgumentError("dòng $line_num cần có nhãn EI/IE/N, tên và chuỗi DNA"))
            push!(groups[fields[1]], encode(uppercase(fields[3])))
        end
    end
    all(g -> !isempty(groups[g]), GROUPS) || throw(ArgumentError("dữ liệu cần có đủ ba nhóm EI, IE, N"))
    return groups
end

"""The 32 position/base items fit in one mask. Used only to recount across groups."""
function item_mask(items)
    mask = zero(UInt32)
    for id in items
        mask |= UInt32(1) << (id - 1)
    end
    return mask
end

function compare_groups(groups, minsup)
    results = [FIM.mine(groups[g]; min_support=minsup) for g in GROUPS]
    patterns = union((keys(r.itemsets) for r in results)...)
    masks = [item_mask.(groups[g]) for g in GROUPS]
    ordered = sort!(collect(patterns); by=p -> (length(p), p))
    return map(ordered) do pattern
        wanted = item_mask(pattern)
        counts = ntuple(3) do i
            # Missing from a group's frequent sets means below cutoff, possibly nonzero.
            get(results[i].itemsets, pattern) do
                count(mask -> (mask & wanted) == wanted, masks[i])
            end
        end
        support = ntuple(i -> counts[i] // results[i].transaction_count, 3)
        (; pattern, counts, support)
    end
end

pattern_label(p) = join(item_label.(p), " ")

function write_patterns(path, rows)
    open(path, "w") do io
        println(io, "pattern,size,count_EI,count_IE,count_N,support_EI,support_IE,support_N")
        for row in rows
            print(io, pattern_label(row.pattern), ',', length(row.pattern), ',', join(row.counts, ','))
            for value in row.support
                @printf(io, ",%.8f", value)
            end
            println(io)
        end
    end
end

function write_report(path, groups, minsup, rows)
    cutoff = FIM.support_ratio(minsup)
    open(path, "w") do io
        println(io, "Các mẫu nucleotide quanh điểm nối exon – intron")
        println(io, "Số đoạn: ", join(("$g=$(length(groups[g]))" for g in GROUPS), ", "))
        println(io, "Vị trí: -4..-1, +1..+4. Minsup mỗi nhóm: $minsup. Tổng số mẫu khác nhau: ", length(rows))
        for (i, boundary) in enumerate(("exon → intron", "intron → exon"))
            group = GROUPS[i]
            pairs = filter(r -> length(r.pattern) == 2 && r.support[i] >= cutoff &&
                                r.support[i] > r.support[3], rows)
            println(io, "\n$group ($boundary)")
            if isempty(pairs)
                println(io, "Không có cặp đạt ngưỡng trong $group và phổ biến hơn ở nhóm N.")
                continue
            end
            # The CSV contains all patterns. Choose one pair per boundary for interpretation.
            row = first(sort!(pairs; by=r -> (r.support[3] - r.support[i], -r.support[i], r.pattern)))
            println(io, "Cặp có chênh lệch support lớn nhất so với N: ", pattern_label(row.pattern))
            for (j, g) in enumerate(GROUPS)
                @printf(io, "  %s: %d/%d (%.2f%%)\n", g, row.counts[j], length(groups[g]), 100 * row.support[j])
            end
            println(io, "Trong bộ UCI, hai nucleotide ở đúng hai vị trí này thường đi cùng nhau gần ranh giới $boundary hơn ở tâm các đoạn đối chứng.")
        end
        println(io, "\nCác tỷ lệ mô tả tập dữ liệu đã gán nhãn. Đánh giá khả năng dự đoán điểm nối cần dữ liệu kiểm chứng độc lập.")
    end
end

function main(args=ARGS; output_root=joinpath(ROOT,"output","splice-junction"), out=stdout, err=stderr)
    if args == ["--help"]
        print(out, HELP)
        return 0
    end
    try
        flags = Dict{String,String}()
        for i in 1:2:length(args)
            key = args[i]
            (key in ("--minsup", "--input") && !haskey(flags, key) && i < length(args)) ||
                throw(ArgumentError("tham số không hợp lệ: $key. Xem --help"))
            flags[key] = args[i+1]
        end
        minsup = get(flags, "--minsup", "0.20")
        FIM.support_threshold(1, minsup, nothing)
        path = get(flags, "--input", joinpath(ROOT,"data","splice-junction","splice.data"))
        groups = read_groups(path)
        rows = compare_groups(groups, minsup)
        dir = FIM.numbered_run_directory(output_root, "splice")
        csv, report = joinpath(dir, "patterns.csv"), joinpath(dir, "report.txt")
        try
            write_patterns(csv, rows)
            write_report(report, groups, minsup, rows)
        catch
            rm(csv; force=true)
            rm(report; force=true)
            isempty(readdir(dir)) && rm(dir)
            rethrow()
        end
        println(out, join(("$g=$(length(groups[g]))" for g in GROUPS), ", "), "; mẫu: ", length(rows))
        println(out, "CSV: ", csv)
        println(out, "Báo cáo: ", report)
        return 0
    catch exc
        if exc isa ArgumentError || exc isa SystemError || exc isa Base.IOError
            println(err, "splice-junction: ", sprint(showerror, exc))
            return 2
        end
        rethrow()
    end
end

end # module

if abspath(PROGRAM_FILE) == @__FILE__
    exit(SpliceJunction.main())
end
''',
    "src/apriori_external.jl": r'''# Disk-backed Apriori path used by the benchmark runner. Itemsets in each level
# are stored as fixed-width Int64 records in lexicographic order.

struct DiskLevel
    path::String
    width::Int
    count::Int
end

struct ExternalMiningSummary
    transaction_count::Int
    min_count::Int
    itemsets::Int
    support_sum::UInt128
    candidate_total::Int
    level_count::Int
    fingerprint_sum::UInt64
    fingerprint_xor::UInt64
end

@inline function external_mix64(value::UInt64)
    value ⊻= value >> 30
    value *= 0xbf58476d1ce4e5b9
    value ⊻= value >> 27
    value *= 0x94d049bb133111eb
    value ⊻= value >> 31
    return value
end

@inline function external_itemset_hash(itemset::NTuple{K,Int}, support::Int) where K
    h = external_mix64(UInt64(K) ⊻ (UInt64(support) << 1))
    for item in itemset
        h = external_mix64(h ⊻ UInt64(item))
    end
    return external_mix64(h ⊻ UInt64(support))
end

@inline function write_itemset(io::IO, itemset::NTuple{K,Int}) where K
    for item in itemset
        write(io, Int64(item))
    end
end

@inline function read_itemset_at(io::IO, level::DiskLevel, index::Int)
    1 <= index <= level.count || throw(BoundsError(1:level.count, index))
    seek(io, (index - 1) * level.width * sizeof(Int64))
    return ntuple(_ -> Int(read(io, Int64)), level.width)
end

@inline function tuple_less(left::Tuple, right::Tuple)
    for i in eachindex(left)
        left[i] == right[i] || return left[i] < right[i]
    end
    return false
end

function disk_level_contains(io::IO, level::DiskLevel, target::NTuple{K,Int}) where K
    low, high = 1, level.count
    while low <= high
        middle = (low + high) >>> 1
        value = read_itemset_at(io, level, middle)
        if value == target
            return true
        elseif tuple_less(value, target)
            low = middle + 1
        else
            high = middle - 1
        end
    end
    return false
end

# Bounded-memory Bloom filter: most absent subsets are rejected without random
# disk seeks. False positives are resolved by the exact binary search above.
struct DiskMembershipBloom
    bits::Vector{UInt64}
    bit_count::UInt64
end

@inline function bloom_hash_pair(itemset::Tuple)
    h = UInt64(0xcbf29ce484222325)
    for item in itemset
        h = (h ⊻ UInt64(item)) * UInt64(0x100000001b3)
    end
    first = external_mix64(h)
    second = external_mix64(first ⊻ UInt64(0x9e3779b97f4a7c15)) | UInt64(1)
    return first, second
end

function build_disk_membership_bloom(level::DiskLevel; max_bytes::Int=32 * 1024^2)
    max_bytes >= 8 || throw(ArgumentError("Bloom filter memory limit must be >= 8 bytes"))
    requested_bits = max(UInt64(64), UInt64(level.count) * UInt64(10))
    maximum_bits = UInt64(max_bytes) * UInt64(8)
    bit_count = min(requested_bits, maximum_bits)
    bit_count = max(UInt64(64), bit_count - (bit_count % UInt64(64)))
    filter = DiskMembershipBloom(zeros(UInt64, Int(bit_count ÷ UInt64(64))), bit_count)
    open(level.path, "r") do source
        for _ in 1:level.count
            itemset = ntuple(_ -> Int(read(source, Int64)), level.width)
            h1, h2 = bloom_hash_pair(itemset)
            index = rem(h1, bit_count)
            step = rem(h2, bit_count)
            for _ in 1:7
                word = Int((index >>> 6) + 1)
                bit = UInt64(1) << (index & UInt64(63))
                filter.bits[word] |= bit
                index += step
                index >= bit_count && (index -= bit_count)
            end
        end
    end
    return filter
end

@inline function bloom_might_contain(filter::DiskMembershipBloom, itemset::Tuple)
    h1, h2 = bloom_hash_pair(itemset)
    index = rem(h1, filter.bit_count)
    step = rem(h2, filter.bit_count)
    for _ in 1:7
        word = Int((index >>> 6) + 1)
        bit = UInt64(1) << (index & UInt64(63))
        (filter.bits[word] & bit) == 0 && return false
        index += step
        index >= filter.bit_count && (index -= filter.bit_count)
    end
    return true
end

# Exact compact membership index for item IDs that fit in a 64- or 128-bit
# packed key. This avoids one disk binary search for every positive Bloom query.
struct PackedDiskMembership{T<:Unsigned}
    keys::Set{T}
    bits_per_item::Int
end

@inline function packed_itemset_key(itemset::Tuple, bits::Int, ::Type{T}) where {T<:Unsigned}
    key = zero(T)
    for item in itemset
        key = (key << bits) | T(item)
    end
    return key
end

@inline function disk_membership_contains(index::PackedDiskMembership{T},
                                          itemset::Tuple) where {T<:Unsigned}
    return packed_itemset_key(itemset, index.bits_per_item, T) in index.keys
end

function build_packed_disk_membership(previous::DiskLevel;
                                      max_bytes::Int=128 * 1024^2)
    previous.count == 0 && return nothing
    max_bytes >= 0 || throw(ArgumentError("packed membership memory limit must be nonnegative"))

    maximum_item = UInt64(0)
    open(previous.path, "r") do source
        for _ in 1:previous.count, _ in 1:previous.width
            value = read(source, Int64)
            value >= 0 || throw(ArgumentError("disk level contains a negative item"))
            maximum_item = max(maximum_item, UInt64(value))
        end
    end
    bits = max(1, 64 - leading_zeros(maximum_item))
    total_bits = previous.width * bits
    key_type, estimated_bytes_per_key = if total_bits <= 64
        (UInt64, 32)
    elseif total_bits <= 128
        (UInt128, 40)
    else
        return nothing
    end
    Int128(previous.count) * estimated_bytes_per_key <= max_bytes || return nothing

    keys = Set{key_type}()
    sizehint!(keys, previous.count)
    open(previous.path, "r") do source
        for _ in 1:previous.count
            itemset = ntuple(_ -> Int(read(source, Int64)), previous.width)
            push!(keys, packed_itemset_key(itemset, bits, key_type))
        end
    end
    return PackedDiskMembership(keys, bits)
end

"""Generate sorted candidates from a disk-backed sorted L_k, with bounded RAM."""
function foreach_disk_candidate_batch(callback::F, previous::DiskLevel,
                                      batch_size::Int; progress::Bool=false,
                                      level::Int=previous.width + 1) where F
    batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    K = previous.width
    K >= 1 || throw(ArgumentError("invalid disk level width"))
    batch = NTuple{K+1,Int}[]
    total = 0
    generation_started = time_ns()

    # Small levels use a tuple Set. Larger levels try an exact packed-key Set
    # (capped at 128 MiB); only levels that cannot be packed/fitted use Bloom +
    # exact on-disk binary search.
    cache_limit = 64 * 1024^2
    estimated_cache_bytes = previous.count * (previous.width * sizeof(Int) + 24)
    known = nothing
    packed_known = nothing
    if estimated_cache_bytes <= cache_limit
        cached = Set{NTuple{K,Int}}()
        sizehint!(cached, previous.count)
        open(previous.path, "r") do source
            for _ in 1:previous.count
                push!(cached, ntuple(_ -> Int(read(source, Int64)), K))
            end
        end
        known = cached
    else
        packed_known = build_packed_disk_membership(previous)
    end
    bloom = isnothing(known) && isnothing(packed_known) ?
            build_disk_membership_bloom(previous) : nothing

    progress_step = max(1, cld(previous.count, 20))
    open(previous.path, "r") do sequential_source
        open(previous.path, "r") do membership_source
            for i in 1:previous.count
                left = read_itemset_at(sequential_source, previous, i)
                seek(sequential_source, i * previous.width * sizeof(Int64))
                for j in (i + 1):previous.count
                    right = ntuple(_ -> Int(read(sequential_source, Int64)), K)
                    all(left[p] == right[p] for p in 1:(K - 1)) || break
                    candidate = (left..., right[K])

                    # The two subsets already known from the join are obtained
                    # by removing positions K and K+1. Check every other K-subset.
                    # In particular, removing position 1 must be checked too;
                    # skipping it admits candidates that Apriori should prune.
                    valid = true
                    for removed in 1:(K - 1)
                        subset = ntuple(p -> candidate[p + (p >= removed)], Val(K))
                        present = if !isnothing(known)
                            subset in known
                        elseif !isnothing(packed_known)
                            disk_membership_contains(packed_known, subset)
                        elseif bloom_might_contain(bloom, subset)
                            disk_level_contains(membership_source, previous, subset)
                        else
                            false
                        end
                        if !present
                            valid = false
                            break
                        end
                    end
                    if valid
                        push!(batch, candidate)
                        total += 1
                        if length(batch) == batch_size
                            callback(batch)
                            empty!(batch)
                        end
                    end
                end
                if progress && (i % progress_step == 0 || i == previous.count)
                    pct = round(100 * i / previous.count; digits=1)
                    println(stderr, "[external] L$level join scan: $i/$(previous.count) ($pct%) | candidates=$total")
                    flush(stderr)
                end
            end
        end
    end
    isempty(batch) || callback(batch)
    return (; total, generation_ns=UInt64(time_ns() - generation_started))
end

function create_level_file(directory::String, width::Int, name::String)
    path = joinpath(directory, name * ".bin")
    return path
end

function build_external_summary(transaction_count, threshold, levels, itemsets,
                                support_sum, fingerprint_sum, fingerprint_xor)
    candidate_total = sum(level.candidates for level in levels; init=0)
    return ExternalMiningSummary(transaction_count, threshold, itemsets, support_sum,
        candidate_total, length(levels), fingerprint_sum, fingerprint_xor)
end

"""Shared disk-output loop. `count_batch` returns supports aligned with candidates."""
function external_mine_levels!(directory::String, transaction_count::Int, threshold::Int,
                               singleton_counts::Dict{Int,Int}, count_batch::F,
                               candidate_batch_size::Int; progress::Bool=false, itemset_io=nothing) where F
    levels = LevelStats[]
    transaction_count == 0 && return build_external_summary(transaction_count, threshold,
        levels, 0, UInt128(0), UInt64(0), UInt64(0))
    singleton_total = length(singleton_counts)
    l1_path = create_level_file(directory, 1, "level-1")
    l1_count = 0
    itemset_count = 0
    support_sum = UInt128(0)
    fingerprint_sum = UInt64(0)
    fingerprint_xor = UInt64(0)

    open(l1_path, "w") do output
        for item in sort!(collect(keys(singleton_counts)))
            support = singleton_counts[item]
            if support >= threshold
                itemset = (item,)
                write_itemset(output, itemset)
                isnothing(itemset_io) || println(itemset_io, item, " #SUP: ", support)
                l1_count += 1
                itemset_count += 1
                support_sum += UInt128(support)
                h = external_itemset_hash(itemset, support)
                fingerprint_sum += h
                fingerprint_xor ⊻= h
            end
        end
    end
    push!(levels, LevelStats(1, singleton_total, l1_count))
    empty!(singleton_counts)
    previous = DiskLevel(l1_path, 1, l1_count)
    itemset_size = 2

    while previous.count > 0
        current_path = create_level_file(directory, itemset_size, "level-$itemset_size")
        current_count = Ref(0)
        progress && println(stderr, "[external] Starting L$itemset_size from $(previous.count) itemsets")
        progress && flush(stderr)
        batch_number = Ref(0)
        counted_candidates = Ref(0)
        generation_phase = open(current_path, "w") do output
            foreach_disk_candidate_batch(previous, candidate_batch_size;
                                         progress, level=itemset_size) do candidates
                batch_number[] += 1
                counted_candidates[] += length(candidates)
                supports = count_batch(candidates, itemset_size)
                if progress && (batch_number[] == 1 || batch_number[] % 10 == 0)
                    println(stderr, "[external] L$itemset_size support-count batch $(batch_number[]) | candidates=$(counted_candidates[]) | frequent=$(current_count[])")
                    flush(stderr)
                end
                length(supports) == length(candidates) || error("counter returned wrong support count")
                for id in eachindex(candidates)
                    support = supports[id]
                    if support >= threshold
                        itemset = candidates[id]
                        write_itemset(output, itemset)
                        isnothing(itemset_io) || println(itemset_io, join(itemset, ' '), " #SUP: ", support)
                        current_count[] += 1
                        itemset_count += 1
                        support_sum += UInt128(support)
                        h = external_itemset_hash(itemset, support)
                        fingerprint_sum += h
                        fingerprint_xor ⊻= h
                    end
                end
            end
        end
        if generation_phase.total == 0
            rm(current_path; force=true)
            break
        end
        push!(levels, LevelStats(itemset_size, generation_phase.total, current_count[]))
        progress && println(stderr, "[external] Finished L$itemset_size | candidates=$(generation_phase.total) | frequent=$(current_count[]) | elapsed=$(round(generation_phase.generation_ns / 1e9; digits=2))s")
        progress && flush(stderr)
        old_level_path = previous.path
        previous = DiskLevel(current_path, itemset_size, current_count[])
        rm(old_level_path; force=true)
        itemset_size += 1
    end
    return build_external_summary(transaction_count, threshold, levels, itemset_count,
                                  support_sum, fingerprint_sum, fingerprint_xor)
end

function with_external_directory(callback::F, workdir) where F
    directory = isnothing(workdir) ? mktempdir(prefix="fim-apriori-") : mktempdir(workdir; prefix="fim-apriori-")
    try
        return callback(directory)
    finally
        rm(directory; recursive=true, force=true)
    end
end

"""Baseline Apriori benchmark path: input transactions stay in RAM; levels/results stay on disk."""
function mine_external(transactions; min_support=nothing, min_count=nothing,
                       counter=:hash_tree, candidate_batch_size::Int=10_000,
                       workdir=nothing, normalized::Bool=false, progress::Bool=false, itemset_io=nothing)
    candidate_batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    mode = counter_mode(counter)
    database = if normalized
        transactions isa Vector{Vector{Int}} ||
            throw(ArgumentError("normalized=true requires Vector{Vector{Int}} input"))
        transactions
    else
        normalize_transactions(transactions)
    end
    threshold = support_threshold(length(database), min_support, min_count)
    singleton_counts = Dict{Int,Int}()
    for transaction in database, item in transaction
        singleton_counts[item] = get(singleton_counts, item, 0) + 1
    end
    return with_external_directory(workdir) do directory
        count_batch = (candidates, _) -> count_candidates(candidates, database, mode)
        external_mine_levels!(directory, length(database), threshold, singleton_counts,
                              count_batch, candidate_batch_size; progress, itemset_io)
    end
end

function mine_bitset_external(path::AbstractString; min_support=nothing, min_count=nothing,
                              candidate_batch_size::Int=10_000, workdir=nothing,
                              progress::Bool=false, itemset_io=nothing)
    candidate_batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    transaction_count, singleton_counts = file_singleton_counts(path)
    threshold = support_threshold(transaction_count, min_support, min_count)
    return with_external_directory(workdir) do directory
        # Build the L1 disk file first, then construct vertical bitmaps from its IDs.
        l1_items = sort!([item for (item, count) in singleton_counts if count >= threshold])
        columns = Dict(item => column for (column, item) in enumerate(l1_items))
        bitmaps = zeros(UInt64, cld(transaction_count, 64), length(l1_items))
        observed = foreach_spmf_transaction(path) do tid, transaction
            word_index = ((tid - 1) >>> 6) + 1
            bit_index = (tid - 1) & 63
            bit = one(UInt64) << bit_index
            for item in transaction
                column = get(columns, item, 0)
                column == 0 || (bitmaps[word_index, column] |= bit)
            end
        end
        observed == transaction_count || throw(ArgumentError("input file changed between passes"))
        count_batch = (candidates, _) -> begin
            counts = Vector{Int}(undef, length(candidates))
            for id in eachindex(candidates)
                counts[id] = bitset_support(bitmaps, columns, candidates[id])
            end
            counts
        end
        external_mine_levels!(directory, transaction_count, threshold, singleton_counts,
                              count_batch, candidate_batch_size; progress, itemset_io)
    end
end

function mine_streaming_external(path::AbstractString; min_support=nothing, min_count=nothing,
                                 candidate_batch_size::Int=10_000, workdir=nothing,
                                 progress::Bool=false, itemset_io=nothing)
    candidate_batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    transaction_count, singleton_counts = file_singleton_counts(path)
    threshold = support_threshold(transaction_count, min_support, min_count)
    return with_external_directory(workdir) do directory
        count_batch = (candidates, _) -> begin
            tree = HashTree(candidates)
            workspace = HashQueryWorkspace(tree)
            counts = zeros(Int, length(candidates))
            observed = foreach_spmf_transaction(path) do _, transaction
                for id in matching_ids!(workspace, tree, transaction)
                    counts[id] += 1
                end
            end
            observed == transaction_count || throw(ArgumentError("input file changed between passes"))
            counts
        end
        external_mine_levels!(directory, transaction_count, threshold, singleton_counts,
                              count_batch, candidate_batch_size; progress, itemset_io)
    end
end
''',
    "src/apriori_optimized.jl": r'''# Hai biến thể Apriori tối ưu được thiết kế dựa trên mine() hiện tại.
# 1) mine_bitset: lưu vertical TID bitmap và đếm support bằng AND + popcount.
# 2) mine_streaming: không giữ toàn bộ database trong RAM; đọc file theo từng lượt.

"""Duyệt file SPMF theo từng transaction, không materialize toàn bộ database.

Callback nhận `(tid, transaction)`. Quy tắc bỏ qua dòng trắng/metadata, xử lý BOM,
parse Int, sort và loại item trùng được giữ giống `read_spmf`.
"""
function foreach_spmf_transaction(callback::F, path::AbstractString) where {F}
    transaction_count = 0
    open(path, "r") do source
        for (line_number, line) in enumerate(eachline(source))
            if line_number == 1 && startswith(line, '\ufeff')
                line = chop(line; head=1, tail=0)
            end
            text = strip(line)
            (isempty(text) || startswith(text, ('#', '%', '@'))) && continue

            items = Int[]
            for token in split(text)
                item = tryparse(Int, token)
                (isnothing(item) || item < 0) &&
                    throw(ArgumentError("line $line_number: expected nonnegative integer items"))
                push!(items, item)
            end
            unique!(sort!(items))
            transaction_count += 1
            callback(transaction_count, items)
        end
    end
    return transaction_count
end

"""First file pass: count N and singleton support without retaining transactions."""
function file_singleton_counts(path::AbstractString)
    singletons = Dict{Int,Int}()
    transaction_count = foreach_spmf_transaction(path) do _, transaction
        for item in transaction
            singletons[item] = get(singletons, item, 0) + 1
        end
    end
    return transaction_count, singletons
end

"""Create L1 and fill the common MiningResult containers."""
function initial_frequent_level!(answer, levels, singletons, threshold)
    previous = NTuple{1,Int}[(item,) for (item, count) in singletons if count >= threshold]
    sort!(previous)
    for itemset in previous
        answer[itemset] = singletons[itemset[1]]
    end
    push!(levels, LevelStats(1, length(singletons), length(previous)))
    return previous
end

"""Count an itemset by intersecting its singleton TID bitmaps.

`bitmaps` has one UInt64 column per frequent singleton and one row per 64 TIDs.
The candidate is mapped to bitmap columns once, outside the word loop.
"""
@inline function bitset_support(bitmaps::Matrix{UInt64}, columns::Dict{Int,Int},
                                candidate::NTuple{K,Int}) where {K}
    candidate_columns = ntuple(index -> columns[candidate[index]], Val(K))
    support = 0
    @inbounds for word_index in axes(bitmaps, 1)
        intersection = bitmaps[word_index, candidate_columns[1]]
        for position in 2:K
            intersection &= bitmaps[word_index, candidate_columns[position]]
            iszero(intersection) && break
        end
        support += count_ones(intersection)
    end
    return support
end

"""Build a UInt64 vertical bitmap matrix for items in L1."""
function build_singleton_bitmaps(path::AbstractString, transaction_count::Int,
                                 previous::Vector{NTuple{1,Int}})
    columns = Dict{Int,Int}()
    for (column, itemset) in enumerate(previous)
        columns[itemset[1]] = column
    end

    word_count = cld(transaction_count, 64)
    bitmaps = zeros(UInt64, word_count, length(previous))
    observed = foreach_spmf_transaction(path) do tid, transaction
        word_index = ((tid - 1) >>> 6) + 1
        bit_index = (tid - 1) & 63
        bit = one(UInt64) << bit_index
        for item in transaction
            column = get(columns, item, 0)
            if column != 0
                bitmaps[word_index, column] |= bit
            end
        end
    end
    observed == transaction_count ||
        throw(ArgumentError("input file changed between Apriori passes (transaction count differs)"))
    return bitmaps, columns
end

"""Apriori using vertical TID bitmaps; optimized for fast support counting.

The input is a path to an SPMF transaction file. Only items frequent at level 1
receive bitmaps. Candidate support is computed by word-wise intersections and
`count_ones`, avoiding a transaction-by-candidate hash-tree traversal.
"""
function mine_bitset(path::AbstractString; min_support=nothing, min_count=nothing,
                      candidate_batch_size::Int=10_000)
    candidate_batch_size >= 1 ||
        throw(ArgumentError("candidate_batch_size must be >= 1"))
    mining_started = time_ns()
    timings = PhaseTiming[]

    started = time_ns()
    transaction_count, singletons = file_singleton_counts(path)
    record_phase!(timings, :singleton_count, 1, time_ns() - started, length(singletons), 0)

    started = time_ns()
    threshold = support_threshold(transaction_count, min_support, min_count)
    record_phase!(timings, :threshold, 0, time_ns() - started, 0, 0)

    answer = Dict{Tuple{Vararg{Int}},Int}()
    levels = LevelStats[]
    if iszero(transaction_count)
        return MiningResult(answer, transaction_count, threshold, levels,
                            timings, time_ns() - mining_started)
    end
    previous = initial_frequent_level!(answer, levels, singletons, threshold)
    isempty(previous) && return MiningResult(answer, transaction_count, threshold, levels,
                                               timings, time_ns() - mining_started)

    # L1's support counts are now copied to the result; release the temporary map.
    singletons = nothing  # let the singleton dictionary become collectible
    # With one frequent singleton, C2 is empty; retain the original phase record
    # but skip the unnecessary second file pass that would build a one-column bitmap.
    if length(previous) < 2
        level_started = time_ns()
        started = time_ns()
        apriori_gen(previous)
        record_phase!(timings, :generate_candidates, 2, time_ns() - started, 0, 0)
        record_phase!(timings, :level_total, 2, time_ns() - level_started, 0, 0)
        return MiningResult(answer, transaction_count, threshold, levels,
                            timings, time_ns() - mining_started)
    end

    started = time_ns()
    bitmaps, columns = build_singleton_bitmaps(path, transaction_count, previous)
    record_phase!(timings, :build_counter, 1, time_ns() - started,
                  length(previous), 0)

    itemset_size = 2
    while !isempty(previous)
        level_started = time_ns()
        candidate_type = typeof((first(previous)..., 0))
        current = candidate_type[]
        generation_phase_index = Ref(0)
        candidate_stats = foreach_candidate_batch(previous, candidate_batch_size, NoTrace()) do candidates
            if iszero(generation_phase_index[])
                push!(timings, PhaseTiming(:generate_candidates, itemset_size, UInt64(0), 0, 0))
                generation_phase_index[] = length(timings)
            end
            count_started = time_ns()
            counts = Vector{Int}(undef, length(candidates))
            for candidate_id in eachindex(candidates)
                counts[candidate_id] = bitset_support(bitmaps, columns, candidates[candidate_id])
            end
            record_phase!(timings, :count_support, itemset_size,
                          time_ns() - count_started, length(candidates), 0)
            filter_started = time_ns()
            frequent_in_batch = 0
            for candidate_id in eachindex(candidates)
                support = counts[candidate_id]
                if support >= threshold
                    candidate = candidates[candidate_id]
                    push!(current, candidate)
                    answer[candidate] = support
                    frequent_in_batch += 1
                end
            end
            record_phase!(timings, :filter, itemset_size, time_ns() - filter_started,
                          length(candidates), frequent_in_batch)
        end
        candidate_total = candidate_stats.total
        if iszero(generation_phase_index[])
            record_phase!(timings, :generate_candidates, itemset_size,
                          candidate_stats.generation_ns, candidate_total, 0)
        else
            timings[generation_phase_index[]] =
                PhaseTiming(:generate_candidates, itemset_size, candidate_stats.generation_ns,
                            candidate_total, 0)
        end
        if iszero(candidate_total)
            record_phase!(timings, :level_total, itemset_size,
                          time_ns() - level_started, 0, 0)
            break
        end
        push!(levels, LevelStats(itemset_size, candidate_total, length(current)))
        record_phase!(timings, :level_total, itemset_size, time_ns() - level_started,
                      candidate_total, length(current))
        previous = current
        itemset_size += 1
    end

    return MiningResult(answer, transaction_count, threshold, levels, timings,
                        time_ns() - mining_started)
end

"""Apriori with a disk-backed transaction source and bounded transaction memory.

The file is scanned once to count singletons and once per candidate batch for
support counting. At most one normalized transaction is resident while support
is counted. This exchanges additional sequential disk reads for a much smaller
database-memory footprint. Candidate processing is bounded by
`candidate_batch_size`: each batch gets its own hash-tree and counters, then is
discarded. The previous frequent level and final output still reside in RAM.
"""
function mine_streaming(path::AbstractString; min_support=nothing, min_count=nothing,
                         candidate_batch_size::Int=10_000)
    candidate_batch_size >= 1 ||
        throw(ArgumentError("candidate_batch_size must be >= 1"))
    mining_started = time_ns()
    timings = PhaseTiming[]

    started = time_ns()
    transaction_count, singletons = file_singleton_counts(path)
    record_phase!(timings, :singleton_count, 1, time_ns() - started, length(singletons), 0)

    started = time_ns()
    threshold = support_threshold(transaction_count, min_support, min_count)
    record_phase!(timings, :threshold, 0, time_ns() - started, 0, 0)

    answer = Dict{Tuple{Vararg{Int}},Int}()
    levels = LevelStats[]
    if iszero(transaction_count)
        return MiningResult(answer, transaction_count, threshold, levels,
                            timings, time_ns() - mining_started)
    end
    previous = initial_frequent_level!(answer, levels, singletons, threshold)
    isempty(previous) && return MiningResult(answer, transaction_count, threshold, levels,
                                               timings, time_ns() - mining_started)
    singletons = nothing  # let the singleton dictionary become collectible

    itemset_size = 2
    while !isempty(previous)
        level_started = time_ns()
        candidate_type = typeof((first(previous)..., 0))
        current = candidate_type[]
        generation_phase_index = Ref(0)
        candidate_stats = foreach_candidate_batch(previous, candidate_batch_size, NoTrace()) do candidates
            if iszero(generation_phase_index[])
                push!(timings, PhaseTiming(:generate_candidates, itemset_size, UInt64(0), 0, 0))
                generation_phase_index[] = length(timings)
            end
            build_started = time_ns()
            tree = HashTree(candidates)
            workspace = HashQueryWorkspace(tree)
            record_phase!(timings, :build_counter, itemset_size,
                          time_ns() - build_started, length(candidates), 0)

            count_started = time_ns()
            counts = zeros(Int, length(candidates))
            observed = foreach_spmf_transaction(path) do _, transaction
                for candidate_id in matching_ids!(workspace, tree, transaction)
                    counts[candidate_id] += 1
                end
            end
            observed == transaction_count ||
                throw(ArgumentError("input file changed between Apriori passes (transaction count differs)"))
            record_phase!(timings, :count_support, itemset_size,
                          time_ns() - count_started, length(candidates), 0)

            filter_started = time_ns()
            frequent_in_batch = 0
            for candidate_id in eachindex(candidates)
                support = counts[candidate_id]
                if support >= threshold
                    candidate = candidates[candidate_id]
                    push!(current, candidate)
                    answer[candidate] = support
                    frequent_in_batch += 1
                end
            end
            record_phase!(timings, :filter, itemset_size, time_ns() - filter_started,
                          length(candidates), frequent_in_batch)
        end
        candidate_total = candidate_stats.total
        if iszero(generation_phase_index[])
            record_phase!(timings, :generate_candidates, itemset_size,
                          candidate_stats.generation_ns, candidate_total, 0)
        else
            timings[generation_phase_index[]] =
                PhaseTiming(:generate_candidates, itemset_size, candidate_stats.generation_ns,
                            candidate_total, 0)
        end
        if iszero(candidate_total)
            record_phase!(timings, :level_total, itemset_size,
                          time_ns() - level_started, 0, 0)
            break
        end
        push!(levels, LevelStats(itemset_size, candidate_total, length(current)))
        record_phase!(timings, :level_total, itemset_size, time_ns() - level_started,
                      candidate_total, length(current))
        previous = current
        itemset_size += 1
    end

    return MiningResult(answer, transaction_count, threshold, levels, timings,
                        time_ns() - mining_started)
end
''',
    "test/apriori_optimized.jl": r'''@testset "Optimized Apriori variants" begin
    @testset "External output preserves support across bitmap words and candidate batches" begin
        mktempdir() do scratch
            path = joinpath(scratch, "word-boundaries.txt")
            for n in (63, 64, 65, 127, 128, 129)
                rows = [i in (1, 63, 64, 65, 127, 128, 129) ? [1, 2, 3] : [1, 3] for i in 1:n]
                write(path, join((join(row, ' ') for row in rows), "\n") * "\n")
                expected = mine(rows; min_count=2, counter=:scan).itemsets
                for (algorithm, input) in ((mine_external, rows), (mine_bitset_external, path),
                                           (mine_streaming_external, path))
                    buffer = IOBuffer()
                    summary = algorithm(input; min_count=2, candidate_batch_size=1, itemset_io=buffer)
                    lines = split(chomp(String(take!(buffer))), '\n')
                    reference = [join(key, ' ') * " #SUP: " * string(value) for (key, value) in expected]
                    @test sort(lines) == sort(reference)
                    @test summary.itemsets == length(lines)
                end
            end
        end
    end
    root = normpath(joinpath(@__DIR__, ".."))
    datasets = joinpath(root, "data", "test-kit", "datasets")
    for filename in sort(readdir(datasets))
        endswith(filename, ".txt") || continue
        path = joinpath(datasets, filename)
        transactions = read_spmf(path)
        expected = mine(transactions; min_count=2, counter=:hash_tree)
        bitset = mine_bitset(path; min_count=2)
        streaming = mine_streaming(path; min_count=2)

        @test bitset.itemsets == expected.itemsets
        @test streaming.itemsets == expected.itemsets
        @test bitset.transaction_count == expected.transaction_count
        @test streaming.transaction_count == expected.transaction_count
        @test bitset.min_support_count == expected.min_support_count
        @test streaming.min_support_count == expected.min_support_count
        @test [(x.size, x.candidates, x.frequent) for x in bitset.levels] ==
              [(x.size, x.candidates, x.frequent) for x in expected.levels]
        @test [(x.size, x.candidates, x.frequent) for x in streaming.levels] ==
              [(x.size, x.candidates, x.frequent) for x in expected.levels]
    end

    mktempdir() do scratch
        path = joinpath(scratch, "transactions.txt")
        write(path, "@RELATION=test\n3 1 3 2\n1 2\n2 3\n3 1\n")
        transactions = read_spmf(path)
        for threshold in 1:4
            expected = mine(transactions; min_count=threshold)
            @test mine_bitset(path; min_count=threshold).itemsets == expected.itemsets
            @test mine_streaming(path; min_count=threshold).itemsets == expected.itemsets
        end
        @test mine_bitset(path; min_support="0.5").itemsets ==
              mine(transactions; min_support="0.5").itemsets
        @test mine_streaming(path; min_support="0.5").itemsets ==
              mine(transactions; min_support="0.5").itemsets

        fractional_path = joinpath(scratch, "fractional.txt")
        write(fractional_path, "1\n1\n1\n1\n2\n2\n3\n")
        fractional_rows = read_spmf(fractional_path)
        fractional_expected = mine(fractional_rows; min_support="0.43")
        @test fractional_expected.min_support_count == 4
        @test mine_bitset(fractional_path; min_support="0.43").itemsets ==
              fractional_expected.itemsets
        @test mine_streaming(fractional_path; min_support="0.43").itemsets ==
              fractional_expected.itemsets

        empty_path = joinpath(scratch, "empty.txt")
        write(empty_path, "# empty database\n\n@META=test\n")
        expected_empty = mine(Vector{Int}[]; min_count=1)
        @test mine_bitset(empty_path; min_count=1).itemsets == expected_empty.itemsets
        @test mine_streaming(empty_path; min_count=1).itemsets == expected_empty.itemsets
        @test isempty(mine_bitset(empty_path; min_count=1).levels)
        @test isempty(mine_streaming(empty_path; min_count=1).levels)
    end

    # Bounded candidate batches must preserve exact itemset/support results and
    # per-level candidate/frequent counts for every engine.
    mktempdir() do scratch
        path = joinpath(scratch, "batching.txt")
        transactions = [
            [1, 2, 3, 4], [1, 2, 3], [1, 2, 4], [1, 3, 4],
            [2, 3, 4], [1, 2, 3, 4], [1, 2], [3, 4],
        ]
        open(path, "w") do output
            foreach(row -> println(output, join(row, ' ')), transactions)
        end
        expected = mine(transactions; min_count=2, candidate_batch_size=10_000)
        for batch_size in (1, 2, 3, 5)
            baseline = mine(transactions; min_count=2, candidate_batch_size=batch_size)
            bitset = mine_bitset(path; min_count=2, candidate_batch_size=batch_size)
            streaming = mine_streaming(path; min_count=2, candidate_batch_size=batch_size)
            @test baseline.itemsets == expected.itemsets
            @test bitset.itemsets == expected.itemsets
            @test streaming.itemsets == expected.itemsets
            @test [(x.size, x.candidates, x.frequent) for x in baseline.levels] ==
                  [(x.size, x.candidates, x.frequent) for x in expected.levels]
            @test [(x.size, x.candidates, x.frequent) for x in bitset.levels] ==
                  [(x.size, x.candidates, x.frequent) for x in expected.levels]
            @test [(x.size, x.candidates, x.frequent) for x in streaming.levels] ==
                  [(x.size, x.candidates, x.frequent) for x in expected.levels]
        end
    end


    # Bloom prefilter must never reject an itemset inserted from a disk level.
    mktempdir() do bloom_scratch
        bloom_path = joinpath(bloom_scratch, "bloom-level.bin")
        open(bloom_path, "w") do output
            for item in 1:2_000
                write(output, Int64(item))
                write(output, Int64(item + 10_000))
            end
        end
        bloom_level = FrequentItemsetMining.DiskLevel(bloom_path, 2, 2_000)
        tiny = FrequentItemsetMining.build_disk_membership_bloom(bloom_level; max_bytes=8)
        absent = (10_001, 20_001)
        @test FrequentItemsetMining.bloom_might_contain(tiny, absent)
        open(bloom_path) do io
            @test !FrequentItemsetMining.disk_level_contains(io, bloom_level, absent)
        end
        @test isnothing(FrequentItemsetMining.build_packed_disk_membership(bloom_level; max_bytes=0))
        for (key, keytype) in (((1, 1 << 31), UInt64),
                               ((1, 2, 3, 1 << 31), UInt128),
                               ((1, 2, 1 << 42), nothing))
            boundary_path = joinpath(bloom_scratch, "packed-boundary-$(length(key)).bin")
            open(boundary_path, "w") do io
                foreach(item -> write(io, Int64(item)), key)
            end
            level = FrequentItemsetMining.DiskLevel(boundary_path, length(key), 1)
            index = FrequentItemsetMining.build_packed_disk_membership(level)
            if isnothing(keytype)
                @test isnothing(index)
            else
                @test eltype(index.keys) == keytype
                @test FrequentItemsetMining.disk_membership_contains(index, key)
            end
        end
        bloom = FrequentItemsetMining.build_disk_membership_bloom(bloom_level)
        @test all(FrequentItemsetMining.bloom_might_contain(bloom, (item, item + 10_000))
                  for item in 1:2_000)
        packed = FrequentItemsetMining.build_packed_disk_membership(bloom_level)
        @test packed !== nothing
        @test all(FrequentItemsetMining.disk_membership_contains(
                      packed, (item, item + 10_000)) for item in 1:2_000)
        @test !FrequentItemsetMining.disk_membership_contains(packed, (9_999, 12_001))

        # Exercise the UInt128 packed-key path (3 * 22 bits = 66 bits).
        wide_path = joinpath(bloom_scratch, "wide-level.bin")
        wide_items = [(1, 2_000_000, 4_000_000),
                      (1, 3_000_000, 4_000_000),
                      (2, 3_000_000, 4_000_000)]
        open(wide_path, "w") do output
            for itemset in wide_items, item in itemset
                write(output, Int64(item))
            end
        end
        wide_level = FrequentItemsetMining.DiskLevel(wide_path, 3, length(wide_items))
        wide_packed = FrequentItemsetMining.build_packed_disk_membership(wide_level)
        @test wide_packed !== nothing
        @test all(FrequentItemsetMining.disk_membership_contains(wide_packed, itemset)
                  for itemset in wide_items)
        @test !FrequentItemsetMining.disk_membership_contains(
            wide_packed, (1, 2_500_000, 4_000_000))
    end

    # Disk-backed benchmark APIs must match the regular in-memory API while
    # avoiding a materialized Dict of all frequent itemsets.
    mktempdir() do scratch
        path = joinpath(scratch, "external-memory.txt")
        transactions = [
            [0, 1, 2, 3], [0, 1, 2], [0, 1, 3], [0, 2, 3],
            [1, 2, 3], [0, 1, 2, 3], [0, 1], [2, 3],
        ]
        open(path, "w") do output
            foreach(row -> println(output, join(row, ' ')), transactions)
        end
        expected = mine(transactions; min_count=2)
        expected_hashes = UInt64[]
        mix_for_test = value -> begin
            value = UInt64(value)
            value ⊻= value >> 30
            value *= 0xbf58476d1ce4e5b9
            value ⊻= value >> 27
            value *= 0x94d049bb133111eb
            value ⊻= value >> 31
            value
        end
        for (itemset, support) in expected.itemsets
            h = mix_for_test(UInt64(length(itemset)) ⊻ (UInt64(support) << 1))
            for item in itemset
                h = mix_for_test(h ⊻ UInt64(item))
            end
            push!(expected_hashes, mix_for_test(h ⊻ UInt64(support)))
        end
        expected_fingerprint_sum = foldl(+, expected_hashes; init=UInt64(0))
        expected_fingerprint_xor = foldl(⊻, expected_hashes; init=UInt64(0))
        expected_support_sum = sum(UInt128(value) for value in values(expected.itemsets); init=UInt128(0))
        expected_candidate_total = sum(level.candidates for level in expected.levels; init=0)

        for batch_size in (1, 2, 5, 10_000)
            external_results = (
                mine_external(transactions; min_count=2, candidate_batch_size=batch_size, workdir=scratch),
                mine_external(transactions; min_count=2, counter=:scan, candidate_batch_size=batch_size, workdir=scratch),
                mine_bitset_external(path; min_count=2, candidate_batch_size=batch_size, workdir=scratch),
                mine_streaming_external(path; min_count=2, candidate_batch_size=batch_size, workdir=scratch),
            )
            for result in external_results
                @test result.transaction_count == expected.transaction_count
                @test result.min_count == expected.min_support_count
                @test result.itemsets == length(expected.itemsets)
                @test result.support_sum == expected_support_sum
                @test result.candidate_total == expected_candidate_total
                @test result.level_count == length(expected.levels)
                @test result.fingerprint_sum == expected_fingerprint_sum
                @test result.fingerprint_xor == expected_fingerprint_xor
            end
        end
        @test readdir(scratch) == ["external-memory.txt"]

        # Regression: C3=(1,2,4) must be pruned because its subset (2,4)
        # is absent from L2, even though the two join parents are frequent.
        # The old external loop checked the wrong positions and retained C3.
        prune_transactions = [[1, 2], [1, 4], [1, 2], [1, 4]]
        prune_path = joinpath(scratch, "prune-regression.txt")
        open(prune_path, "w") do output
            foreach(row -> println(output, join(row, ' ')), prune_transactions)
        end
        prune_expected = mine(prune_transactions; min_count=2)
        @test [(level.size, level.candidates) for level in prune_expected.levels] == [(1, 3), (2, 3)]
        @test sum(level.candidates for level in prune_expected.levels; init=0) == 6
        @test length(prune_expected.levels) == 2
        prune_results = (
            mine_external(prune_transactions; min_count=2, candidate_batch_size=2, workdir=scratch),
            mine_bitset_external(prune_path; min_count=2, candidate_batch_size=2, workdir=scratch),
            mine_streaming_external(prune_path; min_count=2, candidate_batch_size=2, workdir=scratch),
        )
        for result in prune_results
            @test result.itemsets == length(prune_expected.itemsets)
            @test result.candidate_total == 6
            @test result.level_count == 2
        end

        empty_path = joinpath(scratch, "empty.txt")
        write(empty_path, "# empty\n\n")
        empty_result = mine_streaming_external(empty_path; min_count=1, workdir=scratch)
        @test empty_result.itemsets == 0
        @test empty_result.level_count == 0
    end

    @test_throws ArgumentError mine([[1, 2]]; min_count=1, candidate_batch_size=0)
    @test_throws ArgumentError mine_bitset(joinpath(root, "data", "toy", "paper.txt");
                                           min_count=1, candidate_batch_size=0)
    @test_throws ArgumentError mine_streaming(joinpath(root, "data", "toy", "paper.txt");
                                              min_count=1, candidate_batch_size=0)

    rng = MersenneTwister(20261121)
    mktempdir() do scratch
        for trial in 1:20
            path = joinpath(scratch, "random-$trial.txt")
            transactions = Vector{Int}[]
            for _ in 1:10
                row = Int[item for item in 0:6 if rand(rng) < 0.55]
                isempty(row) && push!(row, rand(rng, 0:6))
                push!(transactions, row)
            end
            open(path, "w") do output
                for row in transactions
                    println(output, join(row, ' '))
                end
            end
            expected_rows = read_spmf(path)
            threshold = rand(rng, 1:10)
            expected = mine(expected_rows; min_count=threshold, counter=:hash_tree)
            @test mine_bitset(path; min_count=threshold).itemsets == expected.itemsets
            @test mine_streaming(path; min_count=threshold).itemsets == expected.itemsets
        end
    end
end
''',
    "test/benchmark_optimized_sampling.jl": r'''module BenchmarkOptimizedSamplingTests
using Test
using SHA
include(joinpath(@__DIR__, "..", "bin", "benchmark-optimized.jl"))

@testset "Optimized benchmark APRIORI-V1 transaction sampling" begin
    mktempdir() do scratch
        source = joinpath(scratch, "transactions.txt")
        # Distinct TIDs make it possible to verify exact selected source rows.
        write(source, "@RELATION=sha-sampler\n\n" * join(1:29, "\n") * "\n")
        output = joinpath(scratch, "run", "results.csv")
        samples, total, manifest = create_transaction_samples(
            source, output, UInt64(42); percentages=(10, 25, 50, 75, 100))

        @test total == 29
        @test [sample.transactions for sample in samples] == [2, 7, 14, 21, 29]
        @test isfile(manifest)
        manifest_text = read(manifest, String)
        @test occursin("protocol,source_sha256,sample_pct,transactions,seed,sample_sha256,selected_tid_sha256,sample_file", manifest_text)
        @test occursin("apriori-v1", manifest_text)

        # Independent expression of the protocol: SHA256("apriori-v1|42|i"),
        # lexical digest order, then source-order output of each selected prefix.
        expected_rank = sort(collect(1:29); by=i ->
            (bytes2hex(sha256("apriori-v1|42|$(i)")), i))
        previous = Int[]
        for (sample, percentage) in zip(samples, (10, 25, 50, 75, 100))
            count = fld(29 * percentage, 100)
            expected_ids = sort(expected_rank[1:count])
            actual_ids = parse.(Int, readlines(sample.path))

            @test actual_ids == expected_ids
            @test issubset(previous, actual_ids)
            @test sample.transactions == count
            actual_sample_sha = open(sample.path, "r") do io
                bytes2hex(sha256(io))
            end
            @test sample.sample_sha256 == actual_sample_sha
            @test sample.selected_tid_sha256 == bytes2hex(sha256(join(expected_ids, "\n") * "\n"))
            previous = actual_ids
        end

        # Re-running with the same input and seed must reproduce byte-identical samples.
        second, second_total, _ = create_transaction_samples(
            source, joinpath(scratch, "again", "results.csv"), UInt64(42);
            percentages=(10, 25, 50, 75, 100))
        @test second_total == total
        @test [s.sample_sha256 for s in second] == [s.sample_sha256 for s in samples]
        @test [s.selected_tid_sha256 for s in second] == [s.selected_tid_sha256 for s in samples]
    end

    mktempdir() do scratch
        source = joinpath(scratch, "normalized.txt")
        # Item order is normalized, duplicate items inside a basket are removed,
        # metadata/blank lines are skipped, and duplicate transactions are retained.
        write(source, "@RELATION=normalization\n3 1 3 2\n\n5 4 5\n5 4 5\n2 2 2\n")
        samples, total, _ = create_transaction_samples(
            source, joinpath(scratch, "normalized.csv"), UInt64(42); percentages=(100,))
        @test total == 4
        @test readlines(only(samples).path) == ["1 2 3", "4 5", "4 5", "2"]
    end

    mktempdir() do scratch
        # Golden check for Chess: floor(3196 * 10 / 100) is exactly 319,
        # and the selected-TID hash is fixed by the repository protocol.
        source = joinpath(scratch, "chess-sized.txt")
        write(source, join(1:3196, "\n") * "\n")
        samples, total, _ = create_transaction_samples(
            source, joinpath(scratch, "chess.csv"), UInt64(42); percentages=(10, 100))
        @test total == 3196
        @test [sample.transactions for sample in samples] == [319, 3196]
        @test samples[1].selected_tid_sha256 ==
            "aaa8b4bf695fa5d7008ef34cc4b9a0af55320a7c6435dff1f4a5cad69e2a49f2"
    end

    # Verify the actual checked-in Chess source, not only synthetic rows.
    chess_input = normpath(joinpath(@__DIR__, "..", "data", "benchmarks", "chess.txt"))
    if isfile(chess_input)
        @test sha256_file(chess_input) ==
            "a12ea887df58a396709430af5bf0a9a32d1f6eba8e7c13dd41f28b98572c5db2"
        mktempdir() do scratch
            samples, total, _ = create_transaction_samples(
                chess_input, joinpath(scratch, "chess.csv"), UInt64(42);
                percentages=(10, 100))
            @test total == 3196
            @test [sample.transactions for sample in samples] == [319, 3196]
            @test samples[1].sample_sha256 ==
                "5d8b4aceb1319bf8ce43f8c6791a70387cc495e160fba3fe4076020d90809a6c"
            @test samples[1].selected_tid_sha256 ==
                "aaa8b4bf695fa5d7008ef34cc4b9a0af55320a7c6435dff1f4a5cad69e2a49f2"
            @test samples[2].sample_sha256 ==
                "341d0ff8676e053dd3b417d61f84c732736154f987ce56975a099b07460957f5"
        end
    end

    mktempdir() do scratch
        source = joinpath(scratch, "small.txt")
        write(source, "1\n2\n3\n4\n5\n6\n7\n8\n9\n")
        @test_throws ErrorException create_transaction_samples(
            source, joinpath(scratch, "small.csv"), UInt64(42); percentages=(10,))
    end
end

end # module BenchmarkOptimizedSamplingTests
''',
    "test/market_basket.jl": r'''using Test
using DelimitedFiles: readdlm
include(joinpath(@__DIR__, "..", "apps", "market-basket", "main.jl"))

@testset "Default Retail application matches the notebook" begin
    mktempdir() do dir
        out = IOBuffer()
        @test MarketBasket.main(String[]; output_root=dir, out=out) == 0
        @test occursin("88162 giỏ | 55 tập mục phổ biến | 45 luật", String(take!(out)))
        run = only(readdir(dir; join=true))
        @test basename(run) == "retail-1"
        reference = joinpath(@__DIR__, "..", "apps", "market-basket", "results", "retail-rules.csv")
        all_rules = joinpath(run, "retail-rules.csv")
        @test isfile(all_rules)
        if isfile(all_rules)
            @test readlines(all_rules) == readlines(reference)
        end
        expected, _ = readdlm(reference, ',', String; header=true)
        top, _ = readdlm(joinpath(run, "top10-rules.csv"), ',', String; header=true)
        @test size(top, 1) == 10
        @test top[:, 2:4] == expected[1:10, 1:3]
        @test all(isapprox.(parse.(Float64, top[:, 5:7]), parse.(Float64, expected[1:10, 4:6]); atol=5e-9))
    end
end

@testset "Market basket rules" begin
    mined = MarketBasket.FIM.mine([[1,2],[1,2],[1],[1],[3]]; min_count=1)
    ranked = MarketBasket.rank_rules(mined, 1//2)
    @test ranked.total == 2
    @test [(r.lhs, r.rhs) for r in ranked.rules] == [((2,), (1,)), ((1,), (2,))]
    @test [(r.count, r.support, r.confidence, r.lift) for r in ranked.rules] ==
          [(2, 2//5, 1, 5//4), (2, 2//5, 1//2, 5//4)]
    @test MarketBasket.rank_rules(mined, MarketBasket.confidence_ratio("0.50000000000000001")).total == 1

    triple = MarketBasket.FIM.mine([[1,2,3],[1,2,3],[4]]; min_count=2)
    tied = MarketBasket.rank_rules(triple, 1//1)
    @test tied.total == 12
    @test length(tied.rules) == 10
    @test (tied.rules[2].lhs, tied.rules[2].rhs) == ((1,), (2,3))
    @test issorted([(r.lhs, r.rhs) for r in tied.rules])

    four = MarketBasket.FIM.mine([[1,2,3,4],[1,2,3,4],[5]]; min_count=2)
    larger = MarketBasket.rank_rules(four, 1//1)
    @test larger.total == 50
    @test any(r -> r.lhs == (1,) && r.rhs == (2,3,4), larger.rules)
end

@testset "Market basket application" begin
    mktempdir() do dir
        write(joinpath(dir, "baskets.txt"), "1 2\n1 2\n1\n1\n3\n")
        catalog = joinpath(dir, "products.csv")
        write(catalog, "item_id,StockCode,Description\r\n1,A,\"Cốc, to\"\r\n2,B,\"Tách \"\"A\"\"\"\r\n3,C,Khay\r\n")
        output = joinpath(dir, "runs")
        invoke(args) = MarketBasket.main(args; data_dir=dir, output_root=output,
                                         out=IOBuffer(), err=IOBuffer())
        @test invoke(["--minconf", "1.1"]) == 2
        @test invoke(["--minconf"]) == 2
        @test invoke(["--minsup", "0"]) == 2
        err = IOBuffer()
        @test MarketBasket.main(["--minconf", "nope"]; data_dir=joinpath(dir, "missing"),
                                output_root=output, err=err) == 2
        @test occursin("--minconf", String(take!(err)))
        @test !ispath(output)
        @test invoke(["--minsup", "0.4", "--minconf", "0.5"]) == 0
        first_run = only(readdir(output; join=true))
        path = joinpath(first_run, "top10-rules.csv")
        table, header = readdlm(path, ',', String; header=true)
        @test vec(header) == ["rank","antecedent","consequent","basket_count","support","confidence","lift"]
        @test table[1,2:3] == ["Tách \"A\"", "Cốc, to"]
        @test parse.(Float64, table[1,4:7]) == [2,0.4,1,1.25]
        @test table[2,2:3] == reverse(table[1,2:3])
        @test occursin("Tách \"A\"", read(joinpath(first_run, "report.txt"), String))
        original = read(path)
        @test invoke(["--minsup", "1"]) == 0
        @test read(path) == original
        empty_run = only(setdiff(readdir(output; join=true), [first_run]))
        @test length(readlines(joinpath(empty_run, "top10-rules.csv"))) == 1
        @test occursin("0", read(joinpath(empty_run, "report.txt"), String))
        write(catalog, "item_id,StockCode,Description\n1,A,Cup\n")
        @test invoke(String[]) == 2
        @test length(readdir(output)) == 2
    end
end
''',
    "test/spmf_test_kit.jl": r'''module SpmfTestKit

using Test, TOML
include(joinpath(@__DIR__, "..", "bin", "benchmark.jl"))  # patterns, filehash and the Julia miner

@testset "Apriori vs SPMF on the small test databases" begin
    kit = joinpath(@__DIR__, "..", "data", "test-kit")
    evidence = TOML.parsefile(joinpath(kit, "spmf-2.68.toml"))
    cases = evidence["cases"]
    datasets = readdir(joinpath(kit, "datasets"))
    @test length(cases) >= 5
    @test Set(name * ".txt" for name in keys(cases)) == Set(datasets)

    jar = get(ENV, "SPMF_JAR", "")
    if !isempty(jar)
        @test filehash(jar) == evidence["jar_sha256"]
    end

    mktempdir() do scratch
        @testset "$name" for name in sort!(collect(keys(cases)))
            case = cases[name]
            input = joinpath(kit, "datasets", name * ".txt")
            @test filehash(input) == case["input_sha256"]
            rows = read_spmf(input)
            minsup = case["minsup"]
            n = length(rows)

            # Each SPMF capture includes its header, even when zero itemsets qualify.
            capture = joinpath(scratch, "$(name)-capture.txt")
            write(capture, case["output"])
            reference = patterns(capture, 1, n)
            @test patterns(joinpath(kit, "expected", name * ".txt"), 1, n) == reference

            if !isempty(jar)
                live = joinpath(scratch, name * ".txt")
                stats = read(`java -jar $jar run Apriori $input $live $minsup`, String)
                reported = match(r"Frequent itemsets count\s*:\s*(\d+)", stats)
                @test !isnothing(reported) && parse(Int, reported[1]) == length(reference)
                @test patterns(live, 1, n) == reference
            end

            @testset "$counter" for counter in (:scan, :hash_tree)
                result = mine(rows; min_support=minsup, counter=counter)
                output = joinpath(scratch, "$(name)-$(counter).txt")
                write_spmf(result.itemsets, output)
                @test patterns(output, 1, n) == reference
            end
        end
    end
end

end
''',
    "bin/benchmark-optimized.jl": r'''#!/usr/bin/env julia
# So sánh baseline Apriori hiện tại với mine_bitset và mine_streaming.
# Mỗi engine chạy trong process con riêng để đo peak RSS độc lập.
# Benchmark được lặp trên các tập con lồng nhau gồm 10%, 25%, 50%, 75%, 100% giao tác.

using FrequentItemsetMining
using SHA

const PROJECT_ROOT = dirname(@__DIR__)
const SCRIPT_PATH = abspath(@__FILE__)
const SAMPLE_PERCENTAGES = (10, 25, 50, 75, 100)
const HELP_TEXT = """
So sánh 3 engine Apriori trên 5 quy mô dữ liệu: 10%, 25%, 50%, 75%, 100% giao tác.

Cách chạy từ thư mục gốc repository:
  julia --project=. bin/benchmark-optimized.jl --input data/benchmarks/mushrooms.txt --minsup 0.10
  julia --project=. bin/benchmark-optimized.jl --input data/benchmarks/retail.txt --min-count 500 --repeats 5 --seed 42

Tham số:
  --input PATH       File transaction SPMF (bắt buộc)
  --minsup RATIO     Support ratio dạng thập phân, ví dụ 0.05; được áp dụng lại
                     theo số giao tác của TỪNG tập con
  --min-count N      Support count tuyệt đối, giữ nguyên N ở mọi tập con
  --baseline MODE    hash-tree (mặc định) hoặc scan
  --warmup N         Số lượt warm-up mỗi engine (mặc định 1)
  --repeats N        Số lượt đo mỗi engine (mặc định 3)
  --seed N           Seed trong SHA-256 rank TID, tái lập được (mặc định 42)
  --candidate-batch-size N  Số ứng viên tối đa mỗi batch (mặc định 10000)
  --sample-percentages LIST  Tỷ lệ cần chạy, ví dụ 10 hoặc 10,25 (mặc định tất cả)
  --engines LIST     Engine cần chạy: baseline,bitset,streaming (mặc định tất cả)
  --output PATH      CSV đầu ra (mặc định output/apriori_optimization_results.csv)

Bộ nhớ:
  Benchmark dùng đường chạy external-memory: mỗi level frequent được ghi ra file
  tạm, chỉ giữ batch ứng viên và summary trong RAM. File tạm tự xóa sau mỗi lượt.
  --help             Hiện trợ giúp

Cách lấy mẫu (giao thức APRIORI-V1):
  Chuẩn hóa giao tác theo parser SPMF: bỏ dòng trống/metadata, sắp item,
  loại item trùng trong mỗi giao tác nhưng giữ giao tác trùng nội dung.
  Xếp hạng TID theo SHA256("apriori-v1|SEED|TID"), tăng dần; lấy đúng
  floor(N*p/100) TID đầu, rồi ghi theo thứ tự nguồn. Seed mặc định là 42.
  Các tập con lồng nhau; mọi engine dùng cùng file mẫu ở mỗi tỷ lệ.
  Mẫu, manifest và hash kiểm toán được lưu cạnh CSV.

Đo thời gian:
  median_ms đo end-to-end từng lượt, gồm parse file + khai phá. Tạo tập con không
  nằm trong thời gian đo. Peak RAM là peak working set/RSS của process con, gồm
  Julia runtime và warm-up; so sánh các engine trong cùng một tỷ lệ.

Worker nội bộ:
  Smoke test một engine:
  julia --project=. bin/benchmark-optimized.jl --input data/benchmarks/chess.txt --minsup 0.10 --sample-percentages 10 --engines baseline --warmup 0 --repeats 1 --output output/chess-external-smoke.csv

  --worker INPUT MIN_COUNT ENGINE BASELINE WARMUP REPEATS BATCH_SIZE
"""

"""SHA-256 của file, dùng để kiểm toán input và các file mẫu."""
function sha256_file(path::AbstractString)
    return open(path, "r") do source
        bytes2hex(sha256(source))
    end
end

"""Trả về TID xếp theo SHA256(apriori-v1|seed|TID), có tie-break theo TID."""
function rank_transaction_indices(total::Int, seed::UInt64)
    total >= 1 || throw(ArgumentError("total must be >= 1"))
    digests = [bytes2hex(sha256("apriori-v1|$(seed)|$(tid)")) for tid in 1:total]
    ranked = collect(1:total)
    sort!(ranked; by=tid -> (digests[tid], tid))
    return ranked
end

"""Hash TID đã chọn theo thứ tự nguồn; mỗi TID kết thúc bằng LF."""
function selected_tid_sha256(indices::AbstractVector{<:Integer})
    buffer = IOBuffer()
    for tid in indices
        print(buffer, tid, '\n')
    end
    return bytes2hex(sha256(take!(buffer)))
end

csv_field(value) = "\"" * replace(string(value), "\"" => "\"\"") * "\""

"""
Tạo các tập con lồng nhau theo giao thức APRIORI-V1.
Giao tác được parse/chuẩn hóa bằng read_spmf, xếp hạng bằng SHA-256,
chọn floor(N*p/100) TID và ghi theo thứ tự nguồn.
"""
function create_transaction_samples(input::String, output::String, seed::UInt64;
                                    percentages=SAMPLE_PERCENTAGES)
    transactions = read_spmf(input)
    total = length(transactions)
    total > 0 || error("Input không chứa transaction hợp lệ")
    isempty(percentages) && throw(ArgumentError("percentages cannot be empty"))
    all(pct -> 1 <= pct <= 100, percentages) ||
        throw(ArgumentError("sample percentages must be between 1 and 100"))
    length(unique(percentages)) == length(percentages) ||
        throw(ArgumentError("sample percentages must be unique"))

    ranked_indices = rank_transaction_indices(total, seed)
    source_sha = sha256_file(input)
    sample_directory = output * ".samples"
    mkpath(sample_directory)
    samples = NamedTuple[]

    manifest_path = joinpath(sample_directory, "manifest.csv")
    open(manifest_path, "w") do manifest
        println(manifest, "protocol,source_sha256,sample_pct,transactions,seed,sample_sha256,selected_tid_sha256,sample_file")
        for percentage in percentages
            sample_count = fld(total * percentage, 100)
            sample_count >= 1 || error(
                "The APRIORI-V1 floor rule selects zero transactions at $(percentage)% of N=$total")

            # Prefix of SHA-ranked TIDs, restored to source order before writing.
            selected_indices = sort!(ranked_indices[1:sample_count])
            sample_path = joinpath(sample_directory, "transactions_$(percentage)pct.txt")
            open(sample_path, "w") do target
                for transaction_index in selected_indices
                    print(target, join(transactions[transaction_index], ' '), '\n')
                end
            end

            sample_sha = sha256_file(sample_path)
            tid_sha = selected_tid_sha256(selected_indices)
            push!(samples, (; percentage, transactions=sample_count, path=sample_path,
                            sample_sha256=sample_sha, selected_tid_sha256=tid_sha))
            fields = ("apriori-v1", source_sha, percentage, sample_count, seed,
                      sample_sha, tid_sha, sample_path)
            println(manifest, join(csv_field.(fields), ','))
        end
    end
    return samples, total, manifest_path
end

"""Một lượt chạy; elapsed bao gồm đọc input và mine, không serialize output."""
function run_once(input::String, min_count::Int, engine::String, baseline::String,
                  candidate_batch_size::Int; itemset_io=nothing, progress=true)
    started = time_ns()
    result = if engine == "baseline"
        transactions = read_spmf(input)
        counter = baseline == "scan" ? :scan : :hash_tree
        mine_external(transactions; min_count=min_count, counter=counter, candidate_batch_size,
                      normalized=true, progress, itemset_io)
    elseif engine == "bitset"
        mine_bitset_external(input; min_count=min_count, candidate_batch_size, progress, itemset_io)
    elseif engine == "streaming"
        mine_streaming_external(input; min_count=min_count, candidate_batch_size, progress, itemset_io)
    else
        error("Unknown engine: $engine")
    end
    return time_ns() - started, result
end

"""Mix 64-bit xác định, không phụ thuộc seed hash ngẫu nhiên của từng process."""
@inline function mix64(value::UInt64)
    value ⊻= value >> 30
    value *= 0xbf58476d1ce4e5b9
    value ⊻= value >> 27
    value *= 0x94d049bb133111eb
    value ⊻= value >> 31
    return value
end

"""Fingerprint giao hoán của itemset->support, tính không cần sort/copy keys."""
summarize_result(result::ExternalMiningSummary) = result

function summarize_result(result::MiningResult)
    fingerprint_sum = UInt64(0)
    fingerprint_xor = UInt64(0)
    support_sum = UInt128(0)
    for (itemset, support) in result.itemsets
        hash_value = mix64(UInt64(length(itemset)) ⊻ (UInt64(support) << 1))
        for item in itemset
            hash_value = mix64(hash_value ⊻ UInt64(item))
        end
        hash_value = mix64(hash_value ⊻ UInt64(support))
        fingerprint_sum += hash_value
        fingerprint_xor ⊻= hash_value
        support_sum += UInt128(support)
    end
    candidate_total = 0
    for level in result.levels
        candidate_total += level.candidates
    end
    return (; transaction_count=result.transaction_count,
              min_count=result.min_support_count,
              itemsets=length(result.itemsets), support_sum,
              candidate_total, level_count=length(result.levels),
              fingerprint_sum, fingerprint_xor)
end

function median_value(values::Vector{Float64})
    ordered = sort(values)
    middle = length(ordered) ÷ 2
    return isodd(length(ordered)) ? ordered[middle + 1] :
           (ordered[middle] + ordered[middle + 1]) / 2
end

"""Process worker. Chỉ in summary nhỏ, không serialize toàn bộ itemsets."""
function worker(args)
    length(args) in (7, 8) || error("Expected 7 worker arguments and an optional verification output")
    input = abspath(args[1])
    min_count = parse(Int, args[2])
    engine, baseline = args[3], args[4]
    warmups, repeats = parse(Int, args[5]), parse(Int, args[6])
    candidate_batch_size = parse(Int, args[7])
    candidate_batch_size >= 1 || error("candidate_batch_size must be >= 1")
    warmups >= 0 || error("warmup must be >= 0")
    repeats >= 1 || error("repeats must be >= 1")

    for _ in 1:warmups
        _, warm_result = run_once(input, min_count, engine, baseline, candidate_batch_size)
        warm_result = nothing
        GC.gc(true)
    end

    times_ms = Float64[]
    last_summary = nothing
    for _ in 1:repeats
        elapsed_ns, result = run_once(input, min_count, engine, baseline, candidate_batch_size)
        push!(times_ms, elapsed_ns / 1e6)
        last_summary = summarize_result(result)
        result = nothing
        GC.gc(true)
    end

    label = engine == "baseline" ? "baseline-$baseline" : engine
    # Capture the high-water mark before serializing an optional verification run.
    println("SAMPLES|", join(times_ms, '|'))
    println("PEAK_BYTES|", Sys.maxrss())
    println(join(("RESULT", label,
        string(round(median_value(times_ms); digits=6)),
        string(round(minimum(times_ms); digits=6)),
        string(round(maximum(times_ms); digits=6)),
        last_summary.transaction_count, last_summary.min_count,
        last_summary.itemsets, last_summary.support_sum,
        last_summary.candidate_total, last_summary.level_count,
        last_summary.fingerprint_sum, last_summary.fingerprint_xor), '|'))
    flush(stdout)
    if length(args) == 8
        open(args[8], "w") do io
            run_once(input, min_count, engine, baseline, candidate_batch_size; itemset_io=io, progress=false)
        end
    end
    return 0
end

struct WinMemoryCounters
    size::UInt32
    page_faults::UInt32
    peak_working_set::UInt
    remaining::NTuple{7,UInt}
end

function windows_peak_working_set(handle)
    info = Ref(WinMemoryCounters(sizeof(WinMemoryCounters), 0, 0, ntuple(_ -> UInt(0), 7)))
    ok = ccall((:GetProcessMemoryInfo, "psapi"), stdcall, Int32,
        (Ptr{Cvoid}, Ref{WinMemoryCounters}, UInt32), handle, info, sizeof(WinMemoryCounters))
    return ok == 0 ? 0 : Int(info[].peak_working_set)
end

"""Run worker và đo peak RSS/working set của process con."""
function measured_process(command::Cmd, log_path::String)
    rss_path = log_path * ".rss"
    measured = if Sys.iswindows()
        command
    elseif Sys.islinux() && isfile("/usr/bin/time")
        `/usr/bin/time -o $rss_path -f %M $command`
    else
        error("Peak RAM measurement được hỗ trợ trên Windows và Linux (GNU /usr/bin/time)")
    end

    open(log_path, "w") do log
        started = time_ns()
        process = run(pipeline(measured; stdout=log, stderr=stderr); wait=false)
        peak_bytes = 0
        try
            if Sys.iswindows()
                handle = ccall((:OpenProcess, "kernel32"), stdcall, Ptr{Cvoid},
                    (UInt32, Int32, UInt32), 0x0410, 0, getpid(process))
                handle == C_NULL && error("Cannot open worker process for RAM measurement")
                try
                    while process_running(process)
                        peak_bytes = max(peak_bytes, windows_peak_working_set(handle))
                        sleep(0.01)
                    end
                    peak_bytes = max(peak_bytes, windows_peak_working_set(handle))
                finally
                    ccall((:CloseHandle, "kernel32"), stdcall, Int32, (Ptr{Cvoid},), handle)
                end
            end
            wait(process)
        finally
            if process_running(process)
                kill(process)
                wait(process)
            end
        end
        process_wall_ms = (time_ns() - started) / 1e6
        success(process) || error("Worker failed ($(process.exitcode)); inspect $log_path")
        if Sys.islinux()
            peak_bytes = 1024 * parse(Int, strip(read(rss_path, String)))
        end
        peak_bytes > 0 || error("Peak RAM measurement unavailable; inspect $log_path")
        return (; process_wall_ms, peak_ram_mib=peak_bytes / 2^20)
    end
end

function parse_worker_result(log_path::String)
    log_text = read(log_path, String)
    matches = filter(line -> startswith(line, "RESULT|"), split(log_text, '\n'))
    isempty(matches) && error("Worker did not emit RESULT; inspect $log_path")
    fields = split(last(matches), '|')
    length(fields) == 13 || error("Malformed worker RESULT; inspect $log_path")
    return (; label=fields[2], median_ms=parse(Float64, fields[3]),
        minimum_ms=parse(Float64, fields[4]), maximum_ms=parse(Float64, fields[5]),
        transaction_count=parse(Int, fields[6]), min_count=parse(Int, fields[7]),
        itemsets=parse(Int, fields[8]), support_sum=parse(UInt128, fields[9]),
        candidate_total=parse(Int, fields[10]), level_count=parse(Int, fields[11]),
        fingerprint_sum=parse(UInt64, fields[12]), fingerprint_xor=parse(UInt64, fields[13]))
end

function write_markdown_report(path::String, rows; input::String, original_transactions::Int,
                               minsup, fixed_min_count, warmups::Int, repeats::Int,
                               baseline::String, seed::UInt64, manifest_path::String,
                               candidate_batch_size::Int, sample_percentages,
                               input_sha256::String)
    open(path, "w") do report
        println(report, "# Báo cáo so sánh hiệu năng Apriori theo quy mô tập giao tác")
        println(report)
        println(report, "- **Input gốc:** `", input, "`")
        println(report, "- **Input SHA-256:** `", input_sha256, "`")
        println(report, "- **Số giao tác gốc:** ", original_transactions)
        println(report, "- **Tỷ lệ lấy mẫu:** ", join((string(pct, "%") for pct in sample_percentages), ", "))
        println(report, "- **Seed:** ", seed, " (có thể tái lập)")
        if !isnothing(minsup)
            println(report, "- **Minsup:** ", minsup, " (quy đổi thành min_count riêng theo từng tập con)")
        else
            println(report, "- **Min-count:** ", fixed_min_count, " (giữ nguyên ở mọi tập con được chọn)")
        end
        println(report, "- **Baseline:** `", baseline, "`")
        println(report, "- **Warm-up / repeats:** ", warmups, " / ", repeats)
        println(report, "- **Candidate batch size:** ", candidate_batch_size)
        println(report, "- **Julia / OS / kiến trúc:** ", first(rows).julia_version,
                " / ", first(rows).os, " / ", first(rows).architecture)
        println(report, "- **Manifest và các file mẫu:** `", manifest_path, "`")
        println(report)

        for percentage in sample_percentages
            subset_rows = filter(row -> row.sample_pct == percentage, rows)
            isempty(subset_rows) && continue
            reference = only(filter(row -> startswith(row.engine, "baseline-"), subset_rows))
            println(report, "## Tập con ", percentage, "%")
            println(report)
            println(report, "- **Số giao tác:** ", reference.transactions)
            println(report, "- **min_count áp dụng:** ", reference.min_count)
            println(report, "- **Sample SHA-256:** `", reference.sample_sha256, "`")
            println(report, "- **Selected TID SHA-256:** `", reference.selected_tid_sha256, "`")
            println(report)
            println(report, "| Engine | Median (ms) | Peak RAM (MiB) | Speedup | RAM giảm (%) | Khớp baseline |")
            println(report, "|---|---:|---:|---:|---:|---|")
            for row in subset_rows
                println(report, "| ", row.engine, " | ", string(round(row.median_ms; digits=3)),
                    " | ", string(round(row.peak_ram_mib; digits=2)),
                    " | ", string(round(row.speedup_vs_baseline; digits=2), "×"),
                    " | ", string(round(row.ram_reduction_pct; digits=2)),
                    " | ", row.same_as_baseline ? "PASS" : "MISMATCH", " |")
            end
            println(report)
            for row in subset_rows
                startswith(row.engine, "baseline-") && continue
                time_change = 100 * (reference.median_ms - row.median_ms) / reference.median_ms
                ram_change = 100 * (reference.peak_ram_mib - row.peak_ram_mib) /
                             reference.peak_ram_mib
                time_text = time_change >= 0 ?
                    "nhanh hơn baseline " * string(round(time_change; digits=2), "%") :
                    "chậm hơn baseline " * string(round(-time_change; digits=2), "%")
                ram_text = ram_change >= 0 ?
                    "giảm peak RAM " * string(round(ram_change; digits=2), "%") :
                    "tăng peak RAM " * string(round(-ram_change; digits=2), "%")
                println(report, "- **", row.engine, "**: ", time_text, "; ", ram_text,
                        ". Fingerprint: ", row.same_as_baseline ? "PASS" : "MISMATCH", ".")
            end
            println(report)
        end

        println(report, "## Quy trình lấy mẫu và giới hạn đo")
        println(report, "Giao thức APRIORI-V1: parse SPMF theo thứ tự nguồn; bỏ dòng trống và metadata; ")
        println(report, "sắp item và bỏ item trùng trong mỗi giao tác, nhưng giữ giao tác trùng nội dung. ")
        println(report, "Xếp hạng TID theo SHA-256 của chuỗi `apriori-v1|seed|tid` tăng dần, tie-break bằng TID; ")
        println(report, "chọn đúng `floor(N*p/100)` TID đầu, rồi ghi lại theo thứ tự nguồn. ")
        println(report, "Mẫu 100% giữ toàn bộ giao tác theo thứ tự nguồn sau chuẩn hóa. ")
        println(report, "Mỗi engine ở cùng tỷ lệ nhận cùng một file mẫu. Việc tạo file mẫu nằm ngoài thời gian ")
        println(report, "và peak RAM của process engine. `median_ms` gồm đọc/parse file mẫu và khai phá. ")
        println(report, "`peak_ram_mib` là peak RSS/working set của process con, gồm Julia runtime và warm-up. ")
        println(report, "Với `--minsup`, support count được tính lại theo số transaction của từng tập con; ")
        println(report, "với `--min-count`, ngưỡng đếm tuyệt đối được giữ nguyên ở cả năm quy mô. ")
        println(report, "Manifest lưu SHA-256 của input, sample và danh sách TID được chọn để kiểm toán. ")
        println(report, "Fingerprint là kiểm tra nhất quán giữa các engine; test oracle nên đối chiếu ")
        println(report, "mapping itemset → support đầy đủ.")
    end
    return path
end

function parse_options(args)
    options = Dict{String,String}()
    allowed = Set(("--input", "--minsup", "--min-count", "--baseline",
                   "--warmup", "--repeats", "--seed", "--candidate-batch-size",
                   "--sample-percentages", "--engines", "--output"))
    index = 1
    while index <= length(args)
        key = args[index]
        key in allowed || error("Unknown option: $key\n\n$HELP_TEXT")
        index < length(args) || error("Missing value for $key")
        haskey(options, key) && error("Option repeated: $key")
        options[key] = args[index + 1]
        index += 2
    end
    return options
end

function write_csv(path::String, rows)
    isempty(rows) && error("No benchmark rows to write")
    names = keys(first(rows))
    escape_csv(value) = "\"" * replace(string(value), "\"" => "\"\"") * "\""
    mkpath(dirname(path))
    open(path, "w") do output
        println(output, join(names, ','))
        for row in rows
            println(output, join((escape_csv(value) for value in values(row)), ','))
        end
    end
    return path
end

function benchmark(args)
    isempty(args) && error(HELP_TEXT)
    "--help" in args && (println(HELP_TEXT); return 0)
    options = parse_options(args)
    haskey(options, "--input") || error("--input is required\n\n$HELP_TEXT")
    haskey(options, "--minsup") ⊻ haskey(options, "--min-count") ||
        error("Provide exactly one of --minsup or --min-count")

    input = abspath(options["--input"])
    isfile(input) || error("Input file does not exist: $input")
    baseline = get(options, "--baseline", "hash-tree")
    baseline in ("hash-tree", "scan") || error("--baseline must be hash-tree or scan")
    warmups = parse(Int, get(options, "--warmup", "1"))
    repeats = parse(Int, get(options, "--repeats", "3"))
    candidate_batch_size = parse(Int, get(options, "--candidate-batch-size", "10000"))
    candidate_batch_size >= 1 || error("--candidate-batch-size must be >= 1")
    seed_value = parse(Int64, get(options, "--seed", "42"))
    seed_value >= 0 || error("--seed must be >= 0")
    seed = UInt64(seed_value)
    warmups >= 0 || error("--warmup must be >= 0")
    repeats >= 1 || error("--repeats must be >= 1")

    output = abspath(get(options, "--output", joinpath(PROJECT_ROOT, "output",
                                                       "apriori_optimization_results.csv")))
    sample_percentages = parse.(Int, split(get(options, "--sample-percentages",
        join(SAMPLE_PERCENTAGES, ',')), ','))
    isempty(sample_percentages) && error("--sample-percentages cannot be empty")
    length(unique(sample_percentages)) == length(sample_percentages) ||
        error("--sample-percentages contains duplicates")
    all(pct -> pct in SAMPLE_PERCENTAGES, sample_percentages) ||
        error("--sample-percentages values must be selected from $(join(SAMPLE_PERCENTAGES, ", "))")
    requested_engines = split(get(options, "--engines", "baseline,bitset,streaming"), ',')
    isempty(requested_engines) && error("--engines cannot be empty")
    length(unique(requested_engines)) == length(requested_engines) || error("--engines contains duplicates")
    all(engine -> engine in ("baseline", "bitset", "streaming"), requested_engines) ||
        error("--engines accepts baseline, bitset, streaming")
    "baseline" in requested_engines || error("--engines must include baseline as reference")

    samples, original_transactions, manifest_path =
        create_transaction_samples(input, output, seed; percentages=sample_percentages)
    input_sha256 = sha256_file(input)
    minsup = get(options, "--minsup", nothing)
    fixed_min_count = isnothing(minsup) ? parse(Int, options["--min-count"]) : nothing

    julia = joinpath(Sys.BINDIR, Base.julia_exename())
    metrics_by_sample = Dict{Int,Vector{NamedTuple}}()

    println("Input gốc: $input")
    println("Input SHA-256: $input_sha256")
    println("Giao tác gốc: $original_transactions | seed: $seed")
    println("Sampler: APRIORI-V1 SHA-256 rank + floor(N*p/100), source order")
    println("Tỷ lệ: $(join(sample_percentages, ", "))%")
    println("Engine: $(join(requested_engines, ", "))")
    engine_lookup = Dict("baseline" => ("baseline", "baseline-$baseline"),
                         "bitset" => ("bitset", "bitset"),
                         "streaming" => ("streaming", "streaming"))
    engine_specs = [engine_lookup[name] for name in requested_engines]
    println("Số cấu hình: $(length(samples) * length(engine_specs))")
    flush(stdout)

    for sample in samples
        min_count = isnothing(minsup) ?
            FrequentItemsetMining.support_threshold(sample.transactions, nothing, fixed_min_count) :
            FrequentItemsetMining.support_threshold(sample.transactions, minsup, nothing)
        println("\n--- $(sample.percentage)%: $(sample.transactions) transactions | min_count=$min_count ---")
        sample_metrics = NamedTuple[]

        for (engine, label) in engine_specs
            log_directory = output * ".logs"
            log_path = joinpath(log_directory, string(sample.percentage) * "pct_" *
                                replace(label, '-' => '_') * ".log")
            mkpath(log_directory)
            command = `$julia --startup-file=no --threads=1 --project=$PROJECT_ROOT $SCRIPT_PATH --worker $(sample.path) $min_count $engine $baseline $warmups $repeats $candidate_batch_size`
            println("  Starting $label (warmup=$warmups, repeats=$repeats)...")
            flush(stdout)
            process_metrics = measured_process(command, log_path)
            worker_metrics = parse_worker_result(log_path)
            worker_metrics.label == label || error("Engine label mismatch in $log_path")
            worker_metrics.transaction_count == sample.transactions ||
                error("Transaction count mismatch in $log_path")
            worker_metrics.min_count == min_count || error("min_count mismatch in $log_path")
            push!(sample_metrics, merge(worker_metrics, process_metrics))
            println("  $label | median=$(round(worker_metrics.median_ms; digits=3)) ms | " *
                    "peak=$(round(process_metrics.peak_ram_mib; digits=2)) MiB | " *
                    "$(worker_metrics.itemsets) itemsets")
            flush(stdout)
        end
        metrics_by_sample[sample.percentage] = sample_metrics
    end

    rows = NamedTuple[]
    all_match = true
    for sample in samples
        sample_metrics = metrics_by_sample[sample.percentage]
        reference = only(filter(metric -> startswith(metric.label, "baseline-"), sample_metrics))
        for metric in sample_metrics
            same = metric.transaction_count == reference.transaction_count &&
                   metric.min_count == reference.min_count &&
                   metric.itemsets == reference.itemsets &&
                   metric.support_sum == reference.support_sum &&
                   metric.candidate_total == reference.candidate_total &&
                   metric.level_count == reference.level_count &&
                   metric.fingerprint_sum == reference.fingerprint_sum &&
                   metric.fingerprint_xor == reference.fingerprint_xor
            all_match &= same
            speedup = reference.median_ms / metric.median_ms
            memory_reduction = 100 * (reference.peak_ram_mib - metric.peak_ram_mib) /
                               reference.peak_ram_mib
            push!(rows, (; protocol="apriori-v1", sample_pct=sample.percentage,
                sample_input=sample.path, original_transactions, engine=metric.label,
                input, input_sha256, minsup_ratio=isnothing(minsup) ? "" : string(minsup),
                sample_sha256=sample.sample_sha256,
                selected_tid_sha256=sample.selected_tid_sha256,
                transactions=metric.transaction_count, seed, min_count=metric.min_count,
                warmups, repeats, candidate_batch_size, median_ms=metric.median_ms,
                minimum_ms=metric.minimum_ms, maximum_ms=metric.maximum_ms,
                process_wall_ms=metric.process_wall_ms, peak_ram_mib=metric.peak_ram_mib,
                frequent_itemsets=metric.itemsets, support_sum=metric.support_sum,
                candidates_total=metric.candidate_total, counted_levels=metric.level_count,
                fingerprint_sum=metric.fingerprint_sum, fingerprint_xor=metric.fingerprint_xor,
                same_as_baseline=same, speedup_vs_baseline=speedup,
                ram_reduction_pct=memory_reduction, julia_version=string(VERSION),
                os=string(Sys.KERNEL), architecture=string(Sys.ARCH)))
        end
    end

    write_csv(output, rows)
    report_path = splitext(output)[1] * ".md"
    write_markdown_report(report_path, rows; input, original_transactions, minsup,
        fixed_min_count, warmups, repeats, baseline, seed, manifest_path,
        candidate_batch_size, sample_percentages, input_sha256)
    println("\nCSV: $output")
    println("Markdown report: $report_path")
    println("Sample files + manifest: $(output).samples")
    println("Logs: $(output).logs")
    if !all_match
        error("Kết quả các engine không đồng nhất ở ít nhất một tỷ lệ. CSV đã được lưu; kiểm tra logs.")
    end
    println("ĐỐI CHIẾU: PASS — fingerprint và thống kê khớp baseline ở các tỷ lệ đã chọn.")
    return 0
end

if abspath(PROGRAM_FILE) == SCRIPT_PATH
    if !isempty(ARGS) && ARGS[1] == "--worker"
        exit(worker(ARGS[2:end]))
    elseif length(ARGS) == 5 && ARGS[1] == "--export"
        input, count, engine, target = ARGS[2:end]
        open(target, "w") do io
            run_once(input, parse(Int, count), engine, "hash-tree", 10_000; itemset_io=io, progress=false)
        end
    else
        exit(benchmark(ARGS))
    end
end
''',
    "apps/market-basket/main.jl": r'''module MarketBasket

import FrequentItemsetMining as FIM
using DelimitedFiles: readdlm
using Printf: @sprintf

include("rules.jl")

const ROOT = normpath(joinpath(@__DIR__, "..", ".."))
include(joinpath(ROOT, "src", "AssociationRules.jl"))
const HELP = """
Phân tích giỏ hàng Retail của SPMF bằng Apriori
    julia --project=. apps/market-basket/main.jl [--minsup 0.02] [--minconf 0.50]

    --minsup    Tỷ lệ giỏ tối thiểu chứa cả hai vế, trong (0, 1].
    --minconf   Tỷ lệ tối thiểu giỏ có vế phải trong các giỏ có vế trái, trong [0, 1].
    --help      Hiển thị hướng dẫn.

Mỗi lần chạy tạo retail-rules.csv, top10-rules.csv và report.txt trong output/market-basket/retail-N/.
"""

function options(args)
    values = Dict{String,String}()
    for i in 1:2:length(args)
        flag = args[i]
        flag in ("--minsup", "--minconf") || throw(ArgumentError("tham số lạ $flag"))
        haskey(values, flag) && throw(ArgumentError("tham số lặp $flag"))
        i < length(args) || throw(ArgumentError("thiếu giá trị cho $flag"))
        values[flag] = args[i+1]
    end
    return get(values, "--minsup", "0.02"), get(values, "--minconf", "0.50")
end

function product_names(path)
    data, header = readdlm(path, ',', String; header=true)
    vec(header) == ["item_id", "StockCode", "Description"] ||
        throw(ArgumentError("products.csv cần các cột item_id,StockCode,Description"))
    names = Dict{Int,String}()
    for row in eachrow(data)
        id = tryparse(Int, row[1])
        (isnothing(id) || id < 0 || haskey(names, id) || isempty(strip(row[3]))) &&
            throw(ArgumentError("ID hoặc tên sản phẩm không hợp lệ ở dòng có item_id=$(row[1])"))
        names[id] = row[3]
    end
    return names
end

function csv_cell(text)
    occursin(r"""[,"\r\n]""", text) || return text
    return "\"" * replace(text, "\"" => "\"\"") * "\""
end

score(x) = @sprintf("%.8f", Float64(x))
percent(x) = @sprintf("%.2f%%", 100 * Float64(x))

rule_sides(r::BasketRule) = (r.lhs, r.rhs)
rule_sides(r::AssociationRules.Rule) = (r.antecedent, r.consequent)
side_label(items, ::Nothing) = join(items, ' ')
side_label(items, names) = join((names[id] for id in items), " + ")

function write_results(csv, report, result, ranked, names, minsup, minconf)
    println(csv, "rank,antecedent,consequent,basket_count,support,confidence,lift")
    println(report, isnothing(names) ? "Phân tích giỏ hàng Retail (SPMF)" : "Phân tích giỏ hàng UCI Online Retail")
    println(report, "Số giỏ: ", result.transaction_count)
    println(report, "Minsup: $minsup (ít nhất $(result.min_support_count) giỏ) | Minconf: $minconf")
    println(report, "Tập mục phổ biến: ", length(result.itemsets), " | Luật đạt ngưỡng: ", ranked.total)
    println(report, "Số luật hiển thị: ", length(ranked.rules))
    isempty(ranked.rules) && println(report, "\nKhông có luật đạt hai ngưỡng đã chọn.")
    for (rank, rule) in enumerate(ranked.rules)
        left, right = rule_sides(rule)
        lhs = side_label(left, names)
        rhs = side_label(right, names)
        println(csv, join((rank, csv_cell(lhs), csv_cell(rhs), rule.count,
                           score(rule.support), score(rule.confidence), score(rule.lift)), ','))
        println(report, "\n$rank. $lhs → $rhs")
        println(report, "   Có $(rule.count) giỏ chứa tất cả các món trên (support $(percent(rule.support))).")
        println(report, "   Trong các giỏ có đủ món ở vế trái, $(percent(rule.confidence)) cũng có đủ món ở vế phải.")
        lift_text = @sprintf("%.2f", Float64(rule.lift))
        println(report, "   Tỷ lệ này bằng $lift_text lần tỷ lệ gặp vế phải trong toàn bộ giỏ (lift).")
        if rule.lift > 1
            println(report, "   Có thể thử gợi ý $rhs khi giỏ đã có $lhs.")
        else
            println(report, "   Tỷ lệ mua kèm bằng hoặc thấp hơn mức chung. Ưu tiên thử gợi ý khác.")
        end
    end
end

function main(args=ARGS; data_dir=nothing,
              output_root=joinpath(ROOT, "output", "market-basket"), out=stdout, err=stderr)
    if args == ["--help"]
        print(out, HELP)
        return 0
    end
    try
        minsup, minconf = options(args)
        cutoff = confidence_ratio(minconf)
        input = isnothing(data_dir) ? joinpath(ROOT, "data", "benchmarks", "retail.txt") : joinpath(data_dir, "baskets.txt")
        baskets = FIM.read_spmf(input)
        result = FIM.mine(baskets; min_support=minsup, counter=:hash_tree)
        if isnothing(data_dir)
            counts = Dict{Vector{Int},Int}(collect(items) => n for (items, n) in result.itemsets)
            all_rules = AssociationRules.generate_rules(counts, result.transaction_count; minconf=Float64(cutoff))
            ranked = (total=length(all_rules), rules=first(all_rules, 10))
            names = nothing
            stem = "retail"
        else
            # Explicitly supplied baskets and product catalog use the historical named-product format.
            names = product_names(joinpath(data_dir, "products.csv"))
            for basket in baskets, id in basket
                haskey(names, id) || throw(ArgumentError("products.csv thiếu tên cho item_id=$id"))
            end
            all_rules = nothing
            ranked = rank_rules(result, cutoff)
            stem = "online-retail"
        end
        csv, report = IOBuffer(), IOBuffer()
        write_results(csv, report, result, ranked, names, minsup, minconf)
        dir = FIM.numbered_run_directory(output_root, stem)
        csv_path, report_path = joinpath(dir, "top10-rules.csv"), joinpath(dir, "report.txt")
        rules_path = joinpath(dir, "retail-rules.csv")
        try
            write(csv_path, take!(csv))
            write(report_path, take!(report))
            isnothing(all_rules) || AssociationRules.write_rules_csv(rules_path, all_rules)
        catch
            rm(csv_path; force=true)
            rm(report_path; force=true)
            isnothing(all_rules) || rm(rules_path; force=true)
            isempty(readdir(dir)) && rm(dir)
            rethrow()
        end
        println(out, result.transaction_count, " giỏ | ", length(result.itemsets),
                " tập mục phổ biến | ", ranked.total, " luật đạt ngưỡng.")
        println(out, "Top ", length(ranked.rules), ": ", csv_path)
        isnothing(all_rules) || println(out, "Toàn bộ luật: ", rules_path)
        println(out, "Báo cáo: ", report_path)
        return 0
    catch exc
        if exc isa ArgumentError || exc isa SystemError || exc isa Base.IOError
            println(err, "market-basket: ", sprint(showerror, exc))
            return 2
        end
        rethrow()
    end
end

end # module MarketBasket

if abspath(PROGRAM_FILE) == @__FILE__
    exit(MarketBasket.main())
end
''',
    "apps/market-basket/rules.jl": r'''"""A directed rule, with exact scores for threshold checks and ranking."""
struct BasketRule
    lhs::Tuple{Vararg{Int}}
    rhs::Tuple{Vararg{Int}}
    count::Int
    support::Rational{Int}
    confidence::Rational{Int}
    lift::Rational{Int128}
end

rank_key(r::BasketRule) = (-r.lift, -r.confidence, -r.count, r.lhs, r.rhs)

function confidence_ratio(min_confidence)
    cutoff = try
        FIM.support_ratio(min_confidence)
    catch exc
        exc isa ArgumentError || rethrow()
        throw(ArgumentError("--minconf cần là số thập phân từ 0 đến 1"))
    end
    0 <= cutoff <= 1 || throw(ArgumentError("--minconf phải nằm trong [0, 1]"))
    return cutoff
end

"""Count all qualifying rules and keep the ten highest-ranked ones."""
function rank_rules(result::FIM.MiningResult, cutoff::Rational)
    0 <= cutoff <= 1 || throw(ArgumentError("--minconf phải nằm trong [0, 1]"))
    best = BasketRule[]
    total = 0
    counts, n = result.itemsets, result.transaction_count
    for (items, together) in counts
        length(items) < 2 && continue
        consequents = [(item,) for item in items]
        while !isempty(consequents)
            passed = empty(consequents)
            for rhs in consequents
                lhs = Tuple(item for item in items if !(item in rhs))
                confidence = together // counts[lhs]
                confidence < cutoff && continue
                push!(passed, rhs)
                total += 1
                lift = (Int128(together) * n) // (Int128(counts[lhs]) * counts[rhs])
                rule = BasketRule(lhs, rhs, together, together // n, confidence, lift)
                position = searchsortedfirst(best, rule; by=rank_key)
                if position <= 10
                    insert!(best, position, rule)
                    length(best) > 10 && pop!(best)
                end
            end
            # Growing the consequent can only decrease confidence for this union.
            # Join only consequents that passed, leaving at least one item on the left.
            (length(passed) < 2 || length(first(passed)) + 1 == length(items)) && break
            consequents = FIM.apriori_gen(passed)
        end
    end
    return (; total, rules=best)
end
''',
}
for relative, content in source_files.items():
    path = project / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(content)
(project / "output").mkdir(exist_ok=True)
print(f"Đã chuẩn bị {len(source_files)} tệp mã nguồn và cấu hình.")

In [ ]:
# @title Giải nén dữ liệu và kiểm tra SHA-256
import base64
import gzip

datasets = {
    "data/test-kit/datasets/01_spmf_example.txt": ("d5335622670e7451638c6bad97f62d5a8361c896b5e53cd202dd21fb264c3ee4", "H4sIAAAAAAACCjNUMFYw4TICkqZchgoQ2giJDQDpvI1YIAAAAA=="),
    "data/toy/dense.txt": ("30100610f1db5193c115fca6e75d6d6d61491e1407a65bd5d9c6f0501d847a3c", (
        "H4sIAAAAAAAC/zNUMFIwVjDhMkSmwSSIDRKBiAMA7cqdBCgAAAA="
    )),
    "data/toy/disjoint.txt": ("396b894cfacac3b1ac042ee4ded025cd70a5377f6503261bf0ef35c43915a12b", (
        "H4sIAAAAAAAC/zNUMOIyVjDhMlUw4zJXsOACAPDPDGIQAAAA"
    )),
    "data/toy/duplicates.txt": ("69225ac39fe2dd489e44bc9de88655511f1896851a37104bc2a62a65c09b94a8", (
        "H4sIAAAAAAAC/zNWMARCIy4gVjDmMgKShlzGQJahgjEXAN/7Zg4cAAAA"
    )),
    "data/toy/paper.txt": ("7f7c31e47cdf01736274e381b15203a0d3cabb393af90cb87fd2c12c1d2f70e4", (
        "H4sIAAAAAAAC/zNUMFYw4TICkqZchgoQ2kjBlAsAJdyf5xgAAAA="
    )),
    "data/toy/sparse.txt": ("e24e462a05985a4855554f4ae5fe0f3223c327c7ac86b948bc2febb4cc722d70", (
        "H4sIAAAAAAAC/zNQMDSAAS4DBXMuczQBZK6JEZelJbIAANfLT0s+AAAA"
    )),
    "data/benchmarks/retail.txt": ("..."
    )),
    "data/test-kit/datasets/02_hcmus_fim_slide.txt": ("d37c212244f0f8df527fc8681650155614dd70393250c613f512bb7778a4337b", (
        "H4sIAAAAAAACCjNUMFIwUTDlMlIwBpKGUB6INobTMHkTLgAjXjSQLgAAAA=="
    )),
    "data/test-kit/datasets/03_threshold_boundary.txt": ("7ec653d2d1090b53613b0c3efca8d578ed11079ec45499d27a53a31f4d7e7b39", (
        "H4sIAAAAAAACCjNUMOIyVDAGYiMgCcIA0HEdcxIAAAA="
    )),
    "data/test-kit/datasets/04_dense_transactions.txt": ("029465521f2c0dd7c3fbc2cc8464bdcfc008f8768f6d86d051f9aae88fdf293f", (
        "H4sIAAAAAAACCjNTMFewULDkMsNBAwAis2VYIAAAAA=="
    )),
    "data/test-kit/datasets/05_sparse_no_frequent.txt": ("46ef41713f14bafff3d35786975211b64bc2bc0f24524238eb5623df80125fe2", (
        "H4sIAAAAAAACCjNUMOIyVjDhMlUw4zJXsOCyVDA04AIA0ozQ6BUAAAA="
    )),
    "data/test-kit/datasets/06_repeated_baskets.txt": ("2953b20b137d29de1032d93051736a2e3318fa7c21dc7297350eb52e3d7e9d5e", (
        "H4sIAAAAAAACCjNUMFIw5jJEJblANADBUkMeGgAAAA=="
    )),
    "data/test-kit/datasets/07_fractional_ceiling.txt": ("23f63c743f76a44dfcd3922f463dfb35b22884aed976985fc4a50a81fe099106", (
        "H4sIAAAAAAACCjNUMOIyBGEuIyA05jLhAgBhstGdEgAAAA=="
    )),
    "data/test-kit/datasets/08_large_item_ids.txt": ("6c2a7ebcd3a593bf04c477a33183970e52a28ffcbe37aa05f25ff0ec4db530d8", (
        "H4sIAAAAAAACCjM0MFWwNDDiMgTShgYGhlxADoRhCJGBcAA0z4vFJwAAAA=="
    )),
    "data/test-kit/expected/01_spmf_example.txt": ("cc49732ea5fbf540729fc7f33072f8bfc2adb55875fa093741704b08d81ac643", (
        "H4sIAAAAAAACCjNUUA4ODbBSMOYygrJMuIzhLFM4y1ABJm8EZBvDdRkqmMLFjZDEjRRMkcwzRVKPUGUE5pmimItqGro+hCwAj2v2l7gAAAA="
    )),
    "data/test-kit/expected/02_hcmus_fim_slide.txt": ("136c12bc0470a22d0a7815bed2e88fb07b4620f8222e076057240a691b4eeee8", (
        "H4sIAAAAAAACCjNSUA4ODbBSMOMyhbJMuYwUEGwAgU659CAAAAA="
    )),
    "data/test-kit/expected/03_threshold_boundary.txt": ("ffaa9ad794acd6edbcb5164bf7cc41ebc58c97f1000b3371f5fbce9357b0850b", (
        "H4sIAAAAAAACCjNUUA4ODbBSMOYygrOM4SxDBZioEZBtDGcbIbEBM2Y99EIAAAA="
    )),
    "data/test-kit/expected/04_dense_transactions.txt": ("73e1b0d7998f353560563377f31ad8e7453d3985698a5ae68551d31d0d3408cd", (
        "H4sIAAAAAAACCjNTUA4ODbBSMOEyh7Ms4CxLOMtMwRyJbYHEtkQywQKJbYlkHqo5Fig8SxRzUU1D14fgAwC99ruWuAAAAA=="
    )),
    "data/test-kit/expected/05_sparse_no_frequent.txt": ("e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", (
        "H4sIAAAAAAACCgMAAAAAAAAAAAA="
    )),
    "data/test-kit/expected/06_repeated_baskets.txt": ("f47ff7d1e6f6e5e18707dea91a372d732c0ced4ca78f1224cba3845cfdaaa9ad", (
        "H4sIAAAAAAACCjNUUA4ODbBSMOEygrJMuYzhYoYKRkhsmLgxUC2qGoQMAJCiSWxQAAAA"
    )),
    "data/test-kit/expected/07_fractional_ceiling.txt": ("b0a75d0ffdb64e68315c9c7205bba732aec4d47823a5a0e9b9bc6965c42b299a", (
        "H4sIAAAAAAACCjNSUA4ODbBSMOECAAw8zfAKAAAA"
    )),
    "data/test-kit/expected/08_large_item_ids.txt": ("7b785bdd749572e17a09faa02bdfbc7f584afeabc8a37691ab0311e3d275789e", (
        "H4sIAAAAAAACCjM0MFVQDg4NsFIw5rI0MIKzDQ0MDJE4pgoISSMwH0mBEVgnigAAJ/QF9FcAAAA="
    )),
    "data/test-kit/spmf-2.68.toml": ("bf564fdf62d48073290dedf633690eaeaf977f9369dbc8ad05e06c5e35dc442e", (
        "H4sIAAAAAAACCsVWbW/bRgz+rl8huB+yYYhzR95rgQAthgYY0KFF02IYhkG4F16szpY9SQ62fz86aW0p8NBv6TceSR2PDx+SelHfvv/1pt7ux91+rFPYjfue"
        "cr3t6nFFdduxdqhDl/nY07DarvPA2noTurbQMC4/D+z6w9ubet12VFOX2+5u+HFZDbtNae6pH1q2X9cLWBq3qD6H+zDT2p9QXwKCXlRhfbft23G1ORhe7/qW"
        "T4uq33dNDiOxEgSYSykvQfM9fTOsAmjzcEtKkJ0n9F54QEFBKK1zUEEm6yM6n5N3ibVRGCsilWiQDJs0ATsvquqPFAYalkI2Dw+nf8Jmt6Y/q03bDfvdIYhY"
        "KnZ8AGQSOmtEbQD4WrJKS4MumRiyt8VA1sGhkcl5w5E0pswp5AyyRDAqIZFaVF+Qv64vLi6qVze/vH3z8ff3b64XNz39vadurNuRNgONw6J6dfvu04ef2fZQ"
        "stt3Nx9/e/3hTb0ax93w8upqt2rX7W5Hl2W777uW+sv79o76Zdpurg5ZXS0qWb+4/fT+ZY0VfJFUhUdJT3R64okT+eQj6693AMsnH3n0edRPT6e7Hk8nG9TT"
        "mI+eJ+sBm2ONoFmlzX5oSrtphnWbn5TJnSsT2gQSQKkiistFgy3JGSeNFlJrI1XOVqBnIopkJBYtIUZrrQsK0cZnLtPXxPWxTGYCvZ7Dgc2xNZu43Xc59P/O"
        "EdFnELGUjMYMWQovomaaYhRMyZKCy9o6ylIK6ykprbzPYIPGgLKobMlG9N+duFjhGYrC/9DyCYVUk6kbqBn70A0hjTyOhilmcinOYCbAK6M19y8kwYRJWGLi"
        "4eOUUTGnkoRwxVnjisnOZKFl8SGQcyUX8FieGTNzbFV7lNxR8hOdn3i6iXzSm9pOZDeR/Uw//8LNTvMoTz1P51mlNA/k0HOpum1TvqD07bGsDBUlLTeyVDGU"
        "UjAjk9p4y9WTkcuVICZRQGlQgI4iD3HMxQkJuhA8c6VmGZumpx3xyuN2DsNfHGKerzmTL3iNEUSUaDP4TFIgZI/MQIsmACFKV8JhBuZkwVvUglNmPXezz5q+"
        "UzdPlw9MZh5OlgzOlsJ0+ZwsM/ws8+SxpcO6ScSP6e6eMAbPIIjFYLIKizVBqVxSRg9QlGFaRNQRwDkViDe78U6XpIIWwclCwnspzDMjCOe7xTXr0N9RcwjV"
        "tHn49h4wCYKlyNkG7TEWoZKyNhwIg55/ajQEcKWkSGhDELpwfxTBayHz0hDZPXPeXpzmvxSTHwYh5MwwUcCD4vQlPNwyczg8/T+3b3rlCgsAAA=="
    )),
    "data/benchmarks/chess.txt": ("a12ea887df58a396709430af5bf0a9a32d1f6eba8e7c13dd41f28b98572c5db2", (
        "H4sIAAAAAAACCt196ZnsOq7k/7GiTBAXkKL/js30e327z6lKSQQQEZk1FsQHEMS+lK/2ZV/za32V8lXaV7GvMr/K+qrlq7aval91ftX11cpX619tfLXzqx9f"
        "vX71/tXHVz+/7Piy+mX9y8aXnV/j+Br1a/SvMb7G+TWPr1m/Zv/6P+W/UFUNNRRQfzCwHntQMw/lo6q8E6qxGFgAEtgFYgGAsmeooaCq6qB+ikUX/CsAlMGg"
        "Wgqq6qgK/6ui+1clSNUQU+V7q3dDcYS96IQ9RdULsag6I1LxRmQTqhHtVdXZq6qzV5XrB25SVd4MxZFAtGKqJHULlsCm0xZN94Wb7gvnoBrqrcDaoum+cNP9"
        "q3+gui6+6hyXs+ugVFS1B6pcjnR7DwO3daBALHJQTQcV0habUA3gnHWdI933TONnUFXeDJVgoJCqBvAtBG+F+1cF4Fu8zN0uSiwMgSrxQsXSFSo+GKog36ro"
        "xKLoxOKtVLUtqoaCKhKUSgLXLlTX/asMlPmEnQNVDh3Wfijy1sdCaKau+1gZKM5blQ/6WE0BtcBQRUdVAVNVuT5nBsriEvgNKld/2UxzLl39ha9uK9c7y0Al"
        "dGAl6cC75O2CFUWGDoqhLZpOW0TrL0tXFAl84QaTQOFblTgDz+3+trS6PUW9iH9CdR1V5KYzF9RUU1XAVFUdAzMlVCcD30cV819xyt332gKa+cFClfgXxuWJ"
        "A1BoqtCOdBe16ZOgXiaZVFj/KxcysqrotYhY10I4dK81RGMpKqoY4r4eodhe5x8MFDi4XTdsw4D69lanDiqpcf1khT9W0X2sovtYCBFkjisxoDZFkEMWEWvP"
        "OnadyfptUGkGNpS4C4bLGFAlC9UAVA3FIGCFQpnPjNDm2MLCXnQSWHRi8UnGsemMY9MZERVUeZ3UUlFVdVTx7FXT2aum0+w5qAaAYnbQb0JVndde8d6tahSV"
        "AeV3OavOkf4NUMVt8Cs398Oceu26qdcYA4uOgcen6EC/Fa463+I3QBW3Fa54g8+ceu3X5R503qLr0nRdl2N6xUBShN85EX5/ZqBqlrdzN9X0dNBTdAwMzfKS"
        "o8auCw82FVPTKaaKV0woqhL/qoJnebtubLhz5kNxYlF0b1V0b4VWt0KqKj4fqBob7oqxYUfBe+lajJawxQj3WuMXQyU6jJauw0jYJ7iwHUaCyevOnyc/dVBJ"
        "1VSE+oLW97N0fT8L2/cjmIfuunnorpgcdvTiICb0SpSDBWBHukDad1vbZ4qBqtnrfQY2RR16wbKqqonyrpsoJwl70/0rCQNDb1V1VqRyHVzyIMwgjKNW2FsV"
        "Vh2ab68qeMDs46lCt/1Ap5R5YtF0YhEtzUG1BXbM+1YCJRPlKqoCDJw6qmY2zYQdXr+JevhYMbI+j4NNR5UTqurequreqiLrqJUjFgl1sXT/KuHHJAvRC3Nc"
        "xGVFTHdkznRH5kx3ZC7IQDZU8sicRCzKO6lqLAksOgksOgmMUjV0UOFy2QdAEVZlm25/9QdAFdFbfS8rDR2UT1ssnb1aOnu1dEbEC0VYa/4mqshrzU231tx0"
        "a80lXzjBwFiSyXQbEiVU0Rio+sLzOmqUQIWo6qKokQT1w+ADdp6ZbueZudMxfKoAK89MuF7NdDvPTLfz7N0yyNl5FlQYSKgHBlKp+ibtGg6OXwxV8LvwTLd2"
        "z3Tr1baoGlSo61bfQYaagFV4Es1UdBJI8wWXzhdcOtu4hE7T0nkXHwxVRHKRc2SWzjYurM8kyGl1XaZOAgUY4jDdZIVH2qsOKvyxEiZfkj9jQBV8k5aEqsKC"
        "YpaWTtaJFiHUUEBdpcBP1jUYCVRYLIpOLMo7qWosCSw6CSw6CUScfyND/Wxyk+hADtTdxdyTeBJLxcGho+rnfKNEtXOgvskFm6zwzyrvhGosESw6ESw6EcSd"
        "9sRBdZ1u7zph7zph7zphF7xV072VE6rhJXDooBhUbZ8gPHW3UU/FaU9ygPBzXuTUnWE9dWdYT90Z1s94K2bYM7dne0L571N3LvIUnos8deciT9Ykm4aDKrKI"
        "WJ7LlKfuXOQv+VpLJ+7OGszJGinnQ921Tp3E4WsVB4eOKlVemgHl2Sd4Cm97nviDm2Et6HqskwhVRHKB1oJF94dxJ0YY7sX/B1h7lrhz/QtVFaHrqghdldpX"
        "Yd3c45D4F13nX3Sdf9G5/oWkYFF1UGFhLzoJLDoJ/CTr2HSqvelUe9Op9qbTgU2nmJpOMTWdYnIysHFzdTSo+7diUfVC2JviABenPkz2ObvOt+g636LrfIuu"
        "M41dZxq7TrN3nQ7sOsWUp6qwqjDUYxwn6zwBFmo5DzycrFMIqh6FXw4FYuDUUTXVDCywrarMPN2pS/1IOi9IDGw6qhqmhLAwic6dfyXsJ1m6JMnSZYoXPvnY"
        "SFQJ2owyE2amG2Yz3TCb6YbZTDfMZrqxL9MNs72JKvIwm+mG2Uw3zGa6YTa0WEydtpi6Lzy5exJP3aLJk7qGR2BEGFCMDlzTdeCargPXhB24JuzANUVD2Ge8"
        "FqdX1XS9qqbrHzVd/6jpmjo/AIrQP2rC/lHT9Y8aPU0CXYhHgwrn9U1XQvCvglL5ggyogs9KmyJRd4j+1dKp9gUL5wblgtnLcG6wzvX93DUwWNfmfk45DuS1"
        "Oe9bLexbIaEKEqoAoF6+FYeBKKgWgZrECT2atug6bfGzwYP2hbvuC3fdF+5v+8IuCTSdvZJY4fJOqMZiII2qpRML2gbhwb4IIxD2rhP2rheLv6BOnSOt2tM5"
        "dEPKQzQyx2Rg1zEwI4Gmk0DTSaDpJNB0Emg6CZRcHd75V4a8e+2BSt5SfhNUw0AtHQNjkYghz8tuQWHu83qgGFQtMFTRiUXRMfD+X6kYuB4kMHYHIaRuYxcD"
        "nG/VRUaEBHVN1VBAVQ7UoYOCikVhiYVQ2BGaveqgksd0JG9VIr5F59urUwflus8LMsM9caScoARxVEksfsldPgoxUOJcqKgq0ctHNJ8zlk2wramAj4SaohDf"
        "YqEI/swXNMSXvFV5J1WNJRZFJxY0Bi6ssONviIYkcMEU01BQxWAgJx1junSM6dIxpsuRGD5HAlJMXWTwSVCZUASCVUUsJGIRkj+mS/74qcIy8Dl3IXAFGVAP"
        "kbcGiu11nkQorG4quj9cdCJIs1lLZ7OW0I58KlkLnOu8hzrweWnT5aVNl5c2XarTdKlOU5QrolCSGIF6Dt1059DRUOmk4Knr+zl1fT+nru/n1DXjnLoOmZNS"
        "WeKdizTdWTvTnbXb8y1OcLmHfkHPdDfZTHeTzXRXqkx3Ogr0VkX3VoX1VtBKPl8JVh1UYa5TNd0qQdOtEjTdKkHTrRI03SpB060SNN0qQdOtEjTdKkHTrRI0"
        "3XY1E648M+FyMNMt7DLdFi3TLYEy3Qoj060wMt3+E6Pv1PC0DQxd9+PQtesPXQfz0PUJDl2b29C1uQ1d79nQtU4NXa1i6Fo8hq5xZegyF0MXzA122IOf5s31"
        "dBp43VQSavmWSt9ATTVV5SOg0AykUbWcElgci/UJUAyqFhiq6BhYdAzcmeblM3A5lmjR1G3XqduuU7ddp267Tt12nQ6UQEXapgy8ne6tmunqY5nOZTKdy4Q4"
        "+ZGBWhSXyXTemem8M8lbFdZbLee/IjiCpnMEnYvGVVCbYrF0X3g5vzDBvTWdeysRi5I5LSIwjV1nGrvONHadaew609hFptE7VpGzwl1nr7RQQ2Qau840dp1p"
        "hDPQecmJY4W7zjR2nb1y9awQst8nNfvtgyo6qnZ62vlULR1VC3MFhpAQPKkJQYEE9k8XC/61YUziJ8JA02kL02kLUwg7/2YUJuhhi4W36cx0p8QwSbqthgt0"
        "uXvoyt1Dl9IfugLW0BWwhq4GPdjVgwhVphOLtA4cujTx0EUiQ+cHDp13O3Tx1dD5gUNn8IfOjRlS7xbScGH0prMF7torOqoKi6qFfSt8Z4LpOsGM3t+2wFQV"
        "HQMLa6c5SFt0nbboOm3Rddqi67RF133hrvvCXfeveC1TpmuZMl1zkemai0zXhmO6NhzTteGYrmHFdP0Wpuu3MF1nguk6E0zXLmC6wvqtsFdiQ3v9BVDpCmr9"
        "n7vkdn2XHJe32IcqOqhHqtozFNAPDInFwifpvjHwG1QDU0WD+qaYaF/4xfBB1Y3aVPpQynh6LELyVisXm1AlIu1/rn+quoHXCrL4CQlc+CJ+JXas1F8AhbP4"
        "Eqq6qL8NArXAVAnfCuEy7UjgQtorjgRqoYYCKszAAjONkrfqurfqurfq3LbHoYNS6cCuU0wZg28638J0voXpNLvpNLvpNLvpNHvakR66SGSIEheqlp+q65mq"
        "mjYSfDLBdKkzEFR6Q3bVFPEFb9U/660QVaWqhhoKKAYDF0VbSN6q6BhYWAxcOrEgVACrrjBc6YXhoYNSCXvXCXt/s7CfOnV76sRCRdXQuTFAqAq4Ib+ZIqmA"
        "G/J+qDBVRQd1RdX8560aF8o2oSKnaz1UVR0UXCy2qGJDYaniiAWCgT2tLV5B/bs1QYL18rF67g+DoBoAaigYWHVQWAb+bJqCQk0K1KYELp0ELp0ELp0EIqhq"
        "qLf6t2rvOTM8HFbkkaqZ0hYSqqqOKuxbmRoKrZgK1zRuQqF1YNF94aL7wkX3hYvuX/kl8N7lbLmbLB4JbAmXs/rsFZSqMANLEGoooO5juYbMJvAk0BtgNXx8"
        "sAnViPFBQ8Zy90qw4f2Y/X2Z3XkpxW8bA7fqWwqqpaCqDmozPsAxcOgYKKGq6qhiSGDO51y6uNtLlUosluJf2TNUQAcWnQQWrg7cj+WgYlF0iqkEJfCegROT"
        "pfsPVed2Uj9Xl/sTip8n5kMFqJo6qHSakw/FYGCIKkKEGoAqACiyz/74VlCXEwiVs8Lf3mrFcyQoqFfC3nMB6gommbquj6Tr+kg6v4/kTFFVIl+46yqAXVcB"
        "7LoKYNdVALuuAvgPFNsKhyWw6CRwG6rr/lVXC/u/krfe7UUx5+KMLEqKKsEUWVGFOxRQ9Z1QTPesiwLv+gYoSeDdQwFCAST1XVSZu6NYCDUUUMneW9P1SUve"
        "qryTqsZ6q8KSwOWUwNAXtlttYT4j0hL2KgQVU7d8CVzbA/JkKDRVhULVhFHVFUbkxQ48sgj2jT98YtRFJ9jGgs/9mC7NBGWg6RiIFguJH1N/pC6YLlPXuUyZ"
        "lGCQKoEf03V+TNf5MV3kx5Rc2Biiiu8yVR1VAQZONVW4qOdkTfIKoYYC6iqWO9P5b9xbFd1bFTxVpoPap6qxJLDoJFDIQNzU9ckaWx8fTVV0wHvooH6WsE5W"
        "gxYf6m7+4CQ2g6k4OHRU/Zw/OHVzlCd1ENXHwKKT9vJOqhpLAotOAotOAtFvJRmYO1kTh+OjqcKNAZ66McATPwbIk8Cmk8CmE4uNJkuaz3lVwWLG3V6oAhsK"
        "OHVDAadiecwOAxdmwDsPBY1Qs1TBoBaYgUVHVYHN/XMl0OExLeQ2g1PX045Wt3fdiEms4qlCn7pe/ZPaq89O316VGzmZ4s5P355EqAIVd1zzLSfX3nWp4i5K"
        "FV/VsJjqAgvlKQIyxULiS3edK9g/2j+DthSfupbiU9dSfOpaik9dS/H5huK6vb+O7+9eTpY2TVfaRDNwcvt8H6ES7d8x02iKcBjXvXxSu5cj9kpCVahRuivs"
        "FSBGNWGMaroY1XQxquliVNPFqKaLUU0YoyIaVQnuRTJGNV2MarrA0XQhVtoQD92mi6E7zjJ0o5RDN0o52M36AgZ2HVUvGajq9B26Tt+h6x4drO7Rrnurrnur"
        "rn+rCANNJ4GmEwvTvZVwy0B/OmfHPLtFgCoZqOwCmfbDkTbd8moQVUVHVfFRBRhRdspFF0l7ckBUQlUJQkmkveqgkuN5BM0UM1hoqOkQdi3UEDMQP7VpyJOU"
        "ErEob4CaIudC8lZRKNLkeoiBCybspBlv063FMd1aHNMtkDHdUhfTbVox3aIQ023vMN1KDdOt1DDdlgE0VNpjokENMQPxbaqmW2iAhuK/VdG9VckEqAIJ7CJh"
        "rzooaqe56dq/TddRbLp+TjSUSrObTrObTt3yoaLhQRdJYFVDqSSw6ySwiySQ2hxjyNaEtGYfuoTqYMdXAih4a8Kpg3IyUBXKDYXBr4j61V6bLwpqY032b6Nq"
        "b/6g0i8PBqgqOgYWt1gwoLYZ2OESODahjLWphkIVkYGSt7Kt5R18qqoOKszAohOL8maqDKZuG3KBUd6IzN1j13wooGYnUFUAVJFvyjJ0YNPpwBxUw9z/5VNV"
        "dVTx1C0Uqr2TKotDVZ3HxNxYeA/VPo6qpYBK+IGVckSez8Cqpirwr4pOAj8Daqn/VdMJe9NJYNO9VXuO8BvoiHxVQK3dvAX+UG4WqgCgOuKtDh0UlKqiowqR"
        "D/z5hSfYCv9v26gk+Y2FOnRQBcPAqYOKrlWrHKhDB7XHwK6jipzlZEA1H1R9g1gsyhfucN/CkQ+clHXSECg0VSW7zJdcQe2iuvC3Bkuyaey6EnQXlaAZDFw6"
        "Bi4wA53/6sWaJBLWmXqsopN2P1VVRxXjrbZFsOvUBbPc3XXl7q4rd3d+uftMMbDoxKLoxKLoxOLdX/ipqtR1JdSuK6FqqVJVALuuAth1FcC+nWhvOiimEak6"
        "I1J1mr3hoQJiUT5ILJpOLF6Verqu1NN1pZ6uK/V0dKlnm4HMUk/XlXo6u9Tj+8ILqW4lRREV1LfdT5J8looqUxRFGAxcOgZuC3vXvVU+PNj+V59BlfCtGljY"
        "+++TwKZ7q6ZjoJOqqhP2xjWN5ADVFCVUBgNDX1hIVeV+4QZ3pLlQJe4HLp3LyaGqc/sFwnVN003xmm4I1XSjtabrGzVd36jp2rNM13SWOJ9D/ldd96+67l91"
        "9r/C1PAtlWNyZc5Ml40xXTbGdHkL/wEnCVTSYzKdc2Y65+zdVC2fb3ESW7/5UAyqlo6qBaaq6BiIGytDi8W/b29IsG5G2CQfq+s+VudK+1BAVR1UmIFF91ZF"
        "p5kO2IaLU7cL4tQtaDh1eztO3YKGk5InvmFg0+nAptOBTaeYmk4xNZ1z0dT/auqsMHkK9dRNoZ66KdRfQpVzspHp3QKhhFRB3Zilc6QXctjrJHYXSbIJXZdN"
        "6LpsQtdlE7oim3A3wnbq5spO3bDXqZsUkfzhrvvDXfeHO/djqRIXXZe46IrExWt10bl/eJChTh1UTi6K7mM5lSBLLpZOLpbusZZaCzadFmw6Ldh00u6Eajrj"
        "2CjqQhLkq6CuhgJO3VTFqZuqoDkyQihJQuaqIVYi7F0ngV0ngXyvXQXFeKumY6ATqr0/IbN0ceN6Q9OK6Vpx7M1ULR1VnFYc07XioKmaOrGYYLGIGXz+Wy1k"
        "n/TJaohVaYuu0xZdpy26QlsAUu2mS7WbLtVuuSwTU+Fmwh7TRcOmixv5P2vBdPvQXYgcxCXZWKgFpqpgaptDt0p16JaODtElOxdVwn9lCmEvMKpOnbY4dV/4"
        "1H3hU/eFT90XPnVf+NR9YVXgPXQ+59CFPUPnMQ1p5mLSr7pjN8Kb7lK46W5qm+6mtukO25jusI3pDtuY7tKC6S4tmO7SgukuLZjuUIAE6lYC+f+KurDQdAsL"
        "TbeoxnSLaqCm0SLqFgi1KKbRdFbY3FYYuxPnO1VDB5W6dxSF6rov3HVfuOu+cNd94a77wl3xhQGXgSSPRePgQkIxVv2YbquQ6bYKmW59jOnWx5hu0YrphvNM"
        "N5xnuuE8042xma532XQNxabr8jVdh5vpOtxM13ZmurYz07Uyma6VyXStTKZr+jFdG4npuiBM1wVhun4B01XWTVfuDkEtIlUq32LoXM6hU0xAqPLVL6gquMGy"
        "/x0/+APqj9V+/9KBO1DTCzWSVD0qpr+perGxkIaV4eCheyw/VUARbK/q3dtyUYRyUXRyUXRyUSBy4fvEgatv4U+cxtr/WcupBqEc3H6sq+J62XBwp2dA+QcD"
        "/4TCjrH9UE1XUPktDeXSvSh7G+NWEuqSqvnjrZaDga+iObDF+vOtzhdOE8qTKd/F4kVan4b1MydY0JP/f3CwcqDK6z/cOb7gBdSgQ6Xk4shwUAhF/MRCfbEB"
        "hXaaSFBRsVjxP3zVjEN4q6tkO+GtvpdsAzb/8HEwiBVi4b9d3O700JL2MeCiRS0JHOpAQh2fAuVnYH7XBR+q6KAOHQMzCoOj3hsYyivtCKp2PLSZ2BbyGkoQ"
        "ZmWib1NAHW6onyFx84TEEqgUVZcMvBGLTaiLgA5S4CzX7sU9WQi5+IZ1R9fC0RV2mwyvcRFfK6nczadxXya2Xt4cbk/K3S599wzUsQ81NqEsD/WQrruhau4x"
        "8EK5X0EVPNQgvFWFMfDQMXAbqu8Je6Zstl2ymMDKLZ0qJwOrM3K0OFX3RqQRoZqOqpdQEwNVkcIOZCAU6pEqIQNNx0AOVYdO2N8BNRVQ7VO+cNX9q6r7V1FH"
        "ejntVYk70stpRErckV668OD/QRXPW5UsVItBDR2Uz2dPdMlsUlWcrSv2Yy0yHwr6rw78v4pBHTqotAS2VEdTgUI536rlJHA63goCJTQiftNYEpq96oxI1RmR"
        "aIpkARI/ZS9F4oW6pmqAoDrmrbzaonJ1IEcsDp1YRBnIh/rWWEzC6ntmWNMvjcTCk9WCrdkT2JqNh2osBiJa9lsOKupd/KeM6oqGFxgqGiD87ISIVUX6XTRc"
        "PVA1BVX8UFMB1XRQocbiTShvXS40XTF10xWxAkwCylsVcTbI4JMkPgYu3RdGjBK5oBp4agn1VlVHVQ3aq81RonTgvd9Olw4Q9tvp0qV1XJNgwLdYwWCu6xpx"
        "OqePpF2OBky0FW4wBh4ZBiKhLpKPHwOFa5q6mmVLOtLlXgJpDMz+q4P7r9KOdN9I0+GoUvXSxahyQt3NHaKb6YhY5Z6FbKiwYIQcGQZUeXgsUj+i4BdXrm7a"
        "FUFaQ+e4H2CfwKG5sfWJUQN657N5bNHQ2zOoPHVT5Uio9GMdOg4eOg6GEk1d1+nbde23+9Yxt7orzEB/82P1WKwacQZ7SLdzNgDgMuC76qII7UihzCnDWejM"
        "lLjaSTYKPl3X1Amn6sxSVXRUxVtVI2+V7oqlQ0XeSkVVyVK1GToiqjAuqJYt+NxAEYR9vF/YMb2+WXV76BTTodMW6LxqARfnuq7Xt7N7felWuOqssDcr+Cay"
        "vJopnOxMY1GNY4askiWr0uxIDVZH/pTBSYGq4CbmTaiCgRoKqjZj781WS46+4BVHFmarQdf1ZndRb3bxyEXz7JcmQ1lkWeeELbjahCp4DxfeQZqGajqoiil8"
        "n4jiUgKqfi6UgaHgq/fQxWg3Fme1al4wvGQVKNbVhqspXHdKK3EG2rNx606nbt3p1K07nbp1p7gJi575WhwRrDq5wK1W3deDK15N9Zp90IRA1w0j9FCD+ydR"
        "FbZZuN20DA6GPQxIEm3TFDM3CdMCrYBg4JZZ44ZvBFRVHVXOBqB2fSq1gHuNGuCOj6vXCDR8c7X1eeoWTE/gatrLjktXjLAJdZUZxGM97WJ+TK4GdjGXXJyF"
        "2I88RUuLB/salw/qwBz+GpSN9C8rqmPjOHXJjI/4oECTKuNpQ/eKD9x++8ODstL/SiyQzYkPwh7LyhgMKrSGtDmhaLsoBitMfYSayOOH31zBodtFwRT2KyjE"
        "fmQvVbhNAF5hX7A1pDtQBwWKOTN/BdVQUJcSOH8k3PlUwTYB+Bh4cBkYWjqQfysaA0FLB3aoWk6qtuv5NM3+0uXMr5MhQ1nEkWamwL1QOJ/9BipdS5V4TI9U"
        "TdjyqRhVh44q9Ftd/auFl8CrsncW6tljatthd0FCHZ5l4Ld5OsjpUkOeTNuCekoJQg5H2sOEAPCYo2WvmO0rQRPetLWrpbHBW3pzi6yXu0GB1xzRUNtU/VzG"
        "TKYKfrMPBDXfBjXpDDx0VPmhqEc+TXfk82+LlTSOtqsDUxbLhcW/XWoPswicP+w4Xer9WVCyDhpZC/laZOVOP/4KIusQknUoyJpkshZRO02dP42Gondq7Rnj"
        "QDG/PQc/mZs3L+Vi3UUJAQb6f9YOVVlvRtEnY8I+mVee01BAVQFUmoM4F2PmtHtxuLkT7GE4OAjBqhIshNFyKqdYB1oUaiigqgiK3lcX8jCoHvUnkXVEfzEf"
        "i90FifUHg+4Mvt8NBAVKOcU6w9p7oLZz4cPziWf2raZOLCBd0xvmkUZVDCoWgP8pF/WfgYSrj2WpSvEj1HRCra3mxH2qihuqEqDOXajGgoq91Yq/FRZqm6pG"
        "gXoRkEA+1rodnL+nC/GJXzi5NLp+uk1M3fQTqhHVYFcIYV5hmH95Q1gGTfdYBtCDVw2K+4akcaEMBtU+EaogGXjoqPoYqNsrft++8FXTm22cgpULu/MLl4Sw"
        "V92/qjphxzHw0DHQD1V/CDsearioWhl1m4U68FCWh3rw22MM/I9iKkiowwf1b1dagvWo2glQcGkvuo9V4pppOcOeNJTL4leRurjacLBjRvZFcHs2tT7toIKm"
        "f1oaapuqtieCJa4uXjIwAPVDLv68NbbzVjd/eDr+cNdFWF0XYXVdhNW3faZ0NNx1wVzXBXNdF8x1XYRFEoun1fRMKJ4EOqHQEVZXRFh3KXACWS65SEdzXRFi"
        "PXMQ7eB2ndfZVV6nEusqBX5VjWaUYcK5usd/DPcG81iJ97oKfoauxjl0Nc7BKo50HdRLBsaM8ZXneV6mSkY0X+zPgkOoqhfJpm/R40gkMJwZoJsNLLmP9dPu"
        "D1Za5gbq57IXwh/eNCT+1Z2Bsm1qcDkINYM9fIEK8byWQBBV74AaSAZODBS1SQENNQM9fLB/tXRQaKoON1X1N0HtfWFMm8z0kTXoHKR3//DJeoICNhotFpRX"
        "Bkn9U/ZQYNoMiUtwjhMLtTfH6Q30t6gaVKhDB8VhYNppIkABGdhYwv5WqL2FTeFiqmcFyyZUWls48oJM4zgpIUInqHYh1MbWtd9G1SmF2pnE/kiyQsbxl0Nl"
        "VNOHYE361zpo2h2IFTCQ+dU8TLvPqFl4fLRBib+plRijV2KcVEGKnCEoUA4tBgUKiQfWw+BCbUcJMagWCR6HLqIblDCLWvORQBWHYpqiNPhghT6uxcKbwr59"
        "YuLe4s/8qN7WPeIEVNVRVXVU3TOwhqg6AVAtBVV1VFUdVeF/dej+1aH7V4fuXx26f3Xo/tWh+1eH7l+RGejSFlWnA6tOB1adDqy6f1V1/6rq/lXV/au9+1/7"
        "WdX0vUrXRNY2VP0UqmoI6gRAtRRUfQtVz2fmJeFBCmqgoOaPk7Prx0mkPQYaDKoKoHyafekikYUND3yK6dCp20Onbp1QL8a9mRysIg66yWpEwaghDqIt1qGz"
        "WE6o58eCcrCKOOgmq6EEw2ceD53RT62uCULhLPE+VTT/4tD5Fw6qsv7FofMvHFDj/ohf2JWZry5x3t41u9nTFIWqOqgSZeDhg/q2p4mKlZGLQycXDA6ajoMA"
        "KHpKoepSClWXUqg+ixUO6F5BVR3UI1WWh6KHCFUXIlRdiFCFIULVhQhVGCLgPvGh+8ROqG8cZJMVVhihX9x13RBdV8zvumJ+15W9u67s3XUF4q4rEHddfbNz"
        "K4H9xbQySdg3oRASKIB6WltDLjp2USn1HgpdCeyi+mb4rQ7dWx3ZPZp8KFURoevS0l2XK+6KBO630TlqXnoLC5At9hnHhfUuBCmZrou9uy72vlFN6MCxq6I5"
        "FdYOWSYkK43Vr48553/W3/qi3xxz/kCsw4e1Xt2NRjszm1AIsw8UjKcF7kjBEGAdPizqa22kccPT+c13xwIC9UzV1tFj2zjO+nzMBwa1nRfchDrcUFUH9fNf"
        "AZd5PDkzHKgXawAKdJOCAGvbm+EIxjeyhEL4GsqEZAGwHg6KQ7ZOPsUkTCjXQfGIds9CHR8BtT08AtwDAII6MlBbx98jbzV4UNsVOqYEdp0EdpEE1pzb2eNr"
        "DANQM7AeIuJ29vgulOI/YzGDuwWZUElnusc3UWQk8IaqEwzlXexGxUp6uM5PjIJqzhVoNHlPPdfhe658kBBUTndQMV8QBJVeqlWIa5LzUDO4E4oJRU1rmS7X"
        "ZPRc08Xe08cEkMGggFS1LJTloYyaF9xY61YYu0gxDlpig1wBrQcteA83sdYtQ1WjM3DooFR+u0TdHnh1i4MCUtV0VL1DLJ6Xq9Jczt3aiCH3uHadd9v53u2p"
        "g4LEw+aOUcdv5eAp5OD1aQk4B8+tEwI09wwQ5Dv9MwkUvZgKx5qiVJMJU01ObzD2twJkFWbp24RldmgGLehjUF1PVkU/jGVIl7orXGowC1uWLEO6713nvmfq"
        "7OnK2WBVLZJQuQh8sCtngrfqoLcCMZBWSBjsy4SvqTJuCmPoFNPQhcXA4xL14SL7fw4WtB9U9b2eiP9M4NT46e1HqImBantQ5x3U8FN16Kg6glTdi8X0UPWt"
        "16jenXbclEA/VCFAnRGokqCq+RiIgjojZ9K7p4PqD6r+u2aDjXW+fqw81PkaqkqgeBx8hZUh6/AJxs5jGQwqQNVkURWAOnVU/Q3116hUTOMWD9aZsiPFRxbP"
        "jryCClAVeKwzZbIeFW77IRdVgrVHluHIUmGRHIzzNVT9aKjphwKJ+3xQ7khx38OiykXVyUXVyUXUQ1sAk7VpR1ZOLDzWcQF8wfvYsVzEjusJ6kBCRWNHLNQ2"
        "VU3HwKZjYOMy0BuNLHdO5vVRe4jbebwOU/vfqSYaVP0UqMKFagSov1O4/KxWJUDdUqXKat1QVRJUXf1heDRyXCRl4FjnNhZa3jtT3p++lsQSd50l7jpL3PmW"
        "GJHuNGHC2BwJYwbUKSSLk8W1VNUHAkXId5KhXOtXyFRxImJLxY4u1WQ6Z9Acuv2nL8P8V133hTtB2D3B9wyFI+brUriBajqoR5+JQxUCqiXe6nXwPTahzPlW"
        "E0/VoaPqCFI13kNVv6YK0XtxBVV0XzgMVdwMLH5hJ1B1fAoUlIF+quqPf3V/9CGRJ7nXFphWtxfH2l5jGb2tDk3WSlB16Kg68FR922sdsY0UqP7jsB4C6sBT"
        "ZTooF1U5BhoRquck8EXcvQVV3wYVLKRmGXjoGHjoGHh8BAOjUBai6tBBhd4KAPXQ4TF1beCxYDjUBn51vCrtCF5BFbzPuUlVozS3T10fveSt8H4g/a2qjoHV"
        "zUB+E/iEtkufbsWUaGG+a1WdunbpqWuXvlHtUwH1KBcWZ2BG2lsKqqWg6odSVSNZJhIDDcZAMlR4EiYH1T4IKicWuB56hs+JgGqUyYCpG0IgMLDqGIh9KwO/"
        "1dS91VRIYNNRlYPiKaaqU0xVp5iqTjFVnWKqKbEIpc5IOtDAbzV1byWBygU9heAxnb6h/JmYw3K2c2Kh7ts5CWQxOHjqOHgKOZhLyNzUldIDehM2oLdJVUYE"
        "mw+q6qACVWgQVY0zKY99rB2sKsQqQh5i3wsrG5aSeFORhcNaCYk/uFCSDNrSJRX4UIHOFRBVnwRlkS0U++p928PI22IOVOPs1sByEGUgSViPPLS7nSE83VR1"
        "uqnqfnGo4xy6RsFVjnbvNhj3g7aPLfvF19N0t0YBifWwRiE24RMaVeZMInT2JEIW6tBRdbipqrdiUcBNTV03HgCEev1We3+4CPUFurkYDnWCoQ4wVeg23K5o"
        "w30tF5xE5M3KC06/4A1UwSsMU/R2YqmyLTWoequhe6vx/reqiZ5poVhUHQOrgqqrvT/lYjJ/6vb+TPzen7rHwURekARlOqhXaa1NqKqDQuQFY1QloO4WhSWL"
        "MX5px/hn18l9l75ArK2Z+L0/yRkBQ84ImI4q/jajXwKVlPaF2EnGwMr9rKUQ9wDUduzYPDmFrX1G27EjCGslwtTQY71cPvVLoDwZ95FY8etcUTJ0+6yHbp/1"
        "EK1jJr/VN8008JufrzKrYKruljYMbM9gFsohgUSoc9dp6n9A8cewR85vX75/9bLq2PArtUai6uhXt5JaKpmqn4pp6hQTM9cpEYtHqJ/CnljFPKASiIBqCqiV"
        "+VcYzW7vh5JfCbS9uBGLxb9IaMKLhKa7SOiEMh1U5iKhZe8s0qAyc218sfjzvDwf6oKqzILpx+vouK3PIQbGTivbRbflsXUGe2fv+OYua9PdDjDh7QAU1ubV"
        "d9VG/zxZU0dWQAj5hwpMd6jgj6/leKxYat90RxFMdxTB6EcRXqncu3Q75ACDPTRcMqBOHRTkvpOHgyV9wmL6PIxfAuUXQT5UIWysc5qsqYtHJjQe4Q/OG7RC"
        "jIj0GWSpsPLVaKi4Gwyq5GqpTqp+IVS7goqs9Q8q3Mj+9qAvmIVyetOL3BVmwg40AwzQQQO6iTjGZbq+OtP11Zmur850zW78tzp1YoH7VxOcUpjglMIiNwua"
        "sDER6gpCMmgEsqjtliZs7eS/1il8LY7jTlC5ILepb994CqbQiM2xr1xcVnPsRQqNwcEzyMGYtAuhSuixoAFJJwQkhw4qEX0zc5CfDdWeC6pdF/r8ZqjztjwC"
        "N8QHPtdkt2uNDDwiMH1x6s/WMGZ1hAhFbHjz5JrGdrYTSpUhVft+V8lNy2VhnfvEQQ0dVYNCVf2vWPxLW5R/hmDPpxs7xbd14L+Bz39EcAcruuEgTFZomcI3"
        "qlqCKnvPY9Wb+8psqHsGeltW7fljQaDa1h/+8602m3Kil5wDUKnzyiSonv5X5Z1QjQWFZmDRMRD6r77lmfaNSEIHnjp166Xq0FGFhspsyXma/tqBminf4k8o"
        "yITAnmn8+VacEeLNL7wAB1M3GbicOvC8Z2A7cjc4/ws1NqEsD/V3ZSRA1uEhaxPLELfaNlhYdK9VhK9VhK9VAK911YdbEKNmNzd9+q2Dkb5G3P+ujQSi1MAG"
        "cg3Wq5LP+6AygtHwUBMtgwsn715r7MdaISinO9NvyOL9rXsslBSisKZvO1lAE/5MhLd97/NSMqa/YWsvUgBOxUDdd0u57/mxovV65X7SGFtEYcQyW9sMBNh9"
        "I/gYC8lCsrgDTImErEMBldBNO1AguXiZ4J/gaowXqnEKP4/3QA4dWSmov/TFhCWob6hqObkAQbUY1Lg3j48XaRCVs00GHjoGHpnCz4t/tXORZjrW+4ehmg/q"
        "/gsXmGZ66eFmoCyumKBQ4bc6dGJx7IvF2ITyaosnCRwJBg4KVa6jBThhDx0DgUOh/xUCanMopsDq7GEdGLXCJdq/UiJQaYPft32LWJ39+pYp3Lcojjr7hLXK"
        "bEJVPNTYo6rFGhUows55q0P3Vn6osA7cZqDXt1gK79aV2n99jWZQoY4UA8tGV06OKgBURCzqq0jk6XQ0UAIJUPsMRFN13BYrDuRbHddvtVhUCaEmOBIBncna"
        "h0ow8DkhCD1exZCL846DfCg8B3f1xV2EtTJxo889W06nMxGLrEzXuZuqTd0+d0Twzj1bmbjxoTocg0pkb6FQYarQiWIoFJYq01GFiPGXswBTuFANk05YziwT"
        "GaptBXM/O2Q4N3uBUAaGmjqq5gdR9ZT6OV9JYMXXUB+pWvF+MCpU7l8hAoQdT3pSbvmU6FjvtsF3QU0u1HJStdfe/g9UcBghClVAUFfd7TdY7U1YiFRx93eS"
        "rL97Os+tI0+BloE9M3LX3Y6uQ2exmjtZ3AmF6IsgtYtq3i6opqMqmWnKY00JVnNgvWz3pfm4j1Dh0Ru/cgIJhpeqdLq4v1qlgPPc+6YIegcsLDIn9di9F4Ua"
        "IKpiI1n7jQM7ZF000nfEav+AYBSdYBSdYBShYBSAYHhtMagcTYdy2sflF4wW52DRcbAIOVhSHMy+Vou81giV9NF2XwhlsIRTgUIJqcp94lhFOhM/gmr6XVFo"
        "33c9d4oxN205twOI+fSq38FI/+Eek4sw1onXgj0OVfFQGQbu+dOYx0J7nou4TmH/sTzD5f22mkCI9Xeg2gdB7XhNGSy/13SywscYVPsgqB2jn8Gar8zj1Qxs"
        "364zhe3j/ToKdL5kPUFl7mI3J1noLRs73xjqu0OgINoJEYFvyjt1Ww7W93QJ4WKulcG6aZ5/vOKWH8nCCXquCsNa0QThcqvCTShQK0b19EdwlgBN1qKNAJRj"
        "U8llfhB4YW9zhQ0Sy0dW+rEcZCEEo1wsN7oRjAneo0SAKk8G0uiLed4Bxd8BxFFNna2a7vqOxsbO3wK4krAJdWCWeozEemHn1mkmVS4oHFUtJxYTD3UooGYK"
        "6ltf/WCtGOw6qJf/Krb+wr8/ZOSW5SyfYsqME3saqEdunHg52sJdDNzLbO1TtfmFLWJEwm3hfiMyQ9oCB9VQUA5hX/HBBK+wL9js/Ej01TvHLcISGDL4MbEI"
        "GfzkuEXBQOHGLTJi4ZzCaTlhn0/XBF5hoWYg7tsHaGb4Z5EzbBu7b9GBa0tt952DbYQ17iCo/TXu5tyU3BPXGOBYU0jXxNFlGKyrGOEeakaeK7MbnCeFsaoP"
        "HAsuGWmsfclYnn3TT5IR2w3ugSrMixamO55hyNuV5jZbQqihgEpe5LTYWwWhIm+loWr8Dii+sB86YT90wn7oJPBwi0VVU5U5mzHprnsJMpB0NgMhgct1237X"
        "jYEysCt0ICAcMbd3xleCGLKm77X4qukbWQI12BVqEEOWx2ZlgkfJNz5o3zjWO2CgY09RlcE3JUGy0pJxqv4W6bKUOY4VuVJbLRioBto7G4aq/bV8t0mFGNSh"
        "oyoKNaRQJ2hzvN/LncLMIPpjObCUdFEznlOX8ZzgjGeKhf7PVaLTt4dbEdY3QElURqbrsitMcSrl6esHxyVykVRhzggFE04R7W46t8kcSrDqoPJUPWuLoYPC"
        "vxXNQ0u9VXAgYfoYOHRQYSss+VeH7l8JFdMRFHYh1NCJRQYqoph8/yr1Vhqo8Ow8ohYzdbWYyaqbZcaX/MLOGF/yK6abKVhEvngKyz4TWB9RzKYaejZ1qpIy"
        "Tut4Hw4bPuOufS22ctpMyXihCvIbp6jCzyybbhTb0KPYEybukOrIqYNCJCBNmIBkqFyE6172dNPKKgyHHkR4ueHVA0dQCCFQhBzuAJdup674nfTRPN3MTAfj"
        "A6AE3QM0q0/aE2G6lRQGXbAlIQsUP47QdFZ77lCMQflnzpLLB3rWcx+66Yeha50euh7ZwWque8lAVf/KCKXrWpCBUzTTMbBWZGy+VSygI4iFZVsvhi6lMFiu"
        "9M34SN0+qFZca2V8UEcQ6mpSpW0v9UhDlbdBtQhUfwMD78tLbduNSUjgZAn7UEAlW1ckDHSKxf0By1+DVZCCcbBea2Vl0MFBL1bJYZUnrKe73zXX6CF5rUP4"
        "WimjdXciez0FWSW4Iu/+GvekqHeJgzF1DsbUORgqBg6d2zQ23Kaig0JTVZBQL/XtuBULp7bYhCoAqOaEWnSqGsxg3a/yutS3Ia8piDVz1vERazkd6uLQTVOn"
        "m6ZON02dbppQ3XTeQ/1rc93OxeAnsdiEKmmo/wvR2z1JFjkFAA=="
    )),
    "apps/market-basket/results/retail-rules.csv": ("9ad66d16836992d0b2221d141521d341eac9e730b3bb645ea35006ef16b563b5", (
        "H4sIAAAAAAACCoWWUY5dNw5E/+8qsoCGIZKiSC1n4HSAAIGTmdj7n1PSm/4ad2DY3da7ry5ZrCrqX9++v399//X92/e3r39++/v93z/urz/49+8ff/3153/O"
        "B7/9ziNf39/++P2370/UW+xf5nizPfNtfBnuY9nK7daeFRyt5ZODOUbOuXO/5Zc1c2WNdA6X52NlLxgfti/MHjY8DKzIWcJJyzFmz+FlIxOcTF5X0yx7+NzP"
        "HL9cqM9wdg8KGzsqZruXcGaM6RY0UZ79quctRphAYkbt3pYr+DTV5y6hcN4d1m0C8W43IGNmuD9mtxIQD8jY/Bk9DRSImAcl9WKPiCxeLhTjHbXnqhVp8CtW"
        "gPk5vztnR+aE5TlHi5cAGCBa3rV33ylRyR6nEltzeVcaSPR2MCDVZ+1cc8ThJMqKGdnYnTZHPnwZfic4djvyaYPC07poYoQOM6Oth074vsebfVnF3M0RgBW8"
        "uKYkrM+BihGNjrVWdW13ARmj3nCFfOizHknGX0XZ3nb5yY7puRc/c7lqooeRjUB2ZM8GKhEgygkbY69t/rwKYsyXoYoavhOeXTJVQQ7ntOd7NXoWCAPo5gQi"
        "xdKtR8UcP/y0no5eCzlBBt6wAmrmHlUITT9p8PnAKSgQjkbMR8smCjsza1ttQwreTUVqCw2PuZE1Gl945VQ0DkmfMrTUCv0tJAqEAYV4plPo2MUk9wfXmCJf"
        "plhj1WQ+sKvxiqPGbrA8yhKu4+DA8+C/aIJBUlK86kHIceoJUoC6eT7IBxMQE8YhsRDk3uI6Etk7fVHoos7no62fEwRIzEKOkIP154UJEofMYY6jDgxfFzu1"
        "61QzgibCoLIkjoMzq9CdL8OlKaIplEThdDvH009XR0JJPAmHagt2p0KCRhVGKxg6YaFDzHHYidXBXFx/Pf3QfOa+7IXDQHgb1GArhiaWmbuvVtoUAbguDmmh"
        "lMM21c8dldk8bGo6KE2TIdnWGTnxIqExul2BPQRCJqAnATTey2fuY9TxuVGJduXxqBwJ/QeINzrTgjCC3v83c4AibywTCkTUpi6iyK54SA60F7XKJSmAvBUc"
        "udCABPGhQsQTNS/QdowD2cpPP9mMZq0XVqW0GEoOR6foy0+Ap9tzQXiJ7A7TRDe2HpRvqPBgQDjG9kTmFwPRiEOmyBhmP2L+tnRtRVQxGDIWPedVTiml92Zp"
        "SVZrCybZVSl/ivyxnxfDNfdNHmO45ID6ZQ+oFkY2J9PFEGi5pD9lPbuHaeKihmB2jQbeeUSDh/Emr0ainJ0uCwB6bzQBOZ0HhQcX48YWWMs08HWa+sQNZ/ey"
        "t+ADL2ypz+ENRlDGIjXCbuyI4U9UzHc7h5G+MMraPThTZk2OvFZq+cUrTH96udj0hTHX7Jq0JxRyIogatLS04LT9zs0ibkuEfhqqoaDD6UGJJoIaF7amLJQh"
        "iZfuKYEe67m29A0XfCOIQfTOlkK/5O1F0TrFIHMp4Y+AtUz9JNfgDeT63TOfoMDtqKX93wtt9f9DuZFFXlwzNfnWNE32Ia/UlKhbLmWy8ulW1liDwg0h9AZk"
        "flx5mvoss9jhZJzuPUvxKhw4R1hbzK/dJ7N0YfqH2xuEkE5oCVtSl4CyqVC3olTMPx8gr4wIfZV1j+rluxPFuuKwR0Jxw83roKS2OwuE5tkFz0syWqYyN0HL"
        "rUifkpd2SwGSqEkujuBdEFKYHNEFNc6OeqFw27w+IPEWOYRBFZNnKyB5HEglBI8dFLw+dUtAxtwLEXBoTuTKySrCH1lDHfuOpzQl31wzsHJDKREuDBZZhPYe"
        "Cqe+hyV8LfAaNtBcpHRNtHmupwQJ5kKjoWj3OiDaeIg6xZbFc13tdSrXg9o2vIHFS3gKhD0GJVvZw9IEg5uqn2uprkGXVZbtnreVoVfguVQ4HycSeQgwdFsw"
        "0kn5MjAP+kF+cMIcDh/jreeJ6UGP3LHggwmS3ke2SG9xFeUEGlpH3M96wgi88Gj3819w79sgpwwAAA=="
    )),
}
for relative, (expected_hash, encoded) in datasets.items():
    data = gzip.decompress(base64.b64decode(encoded))
    if hashlib.sha256(data).hexdigest() != expected_hash:
        raise ValueError(f"Dữ liệu không khớp SHA-256: {relative}")
    path = project / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
DATA = project / "data/benchmarks/retail.txt"
print(f"Đã kiểm tra và giải nén {len(datasets)} tệp dữ liệu.")

In [ ]:
# Seed cố định cho mọi phần ngẫu nhiên trong notebook.
# Apriori và bộ sinh luật không dùng số ngẫu nhiên; unit test dùng MersenneTwister(42).
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('seed =', SEED)

## Kiểm thử

Bộ kiểm thử của dự án kiểm tra Apriori, đối chiếu SPMF trên các CSDL nhỏ và kiểm tra bộ sinh luật của Retail.

In [ ]:
run_julia('test/runtests.jl')

## Ví dụ Apriori với 4 giỏ hàng

Trong ví dụ của Agrawal và Srikant, một tập mục phải xuất hiện trong ít nhất 2 giỏ. Cặp `{2, 5}` có mặt trong 3 trên 4 giỏ nên đạt ngưỡng. Cả 3 giỏ có mã 2 đều có mã 5, vì vậy luật `{2} ⇒ {5}` có confidence 100%.

Ta dùng ví dụ nhỏ này để đối chiếu hai bộ đếm `scan` và `hash-tree`, sau đó sinh các luật có confidence từ 60% trở lên.

In [ ]:
run_julia('bin/apriori.jl', '--input', 'data/toy/paper.txt', '--output', 'output/paper.txt', '--minsup', '0.5', '--counter', 'hash-tree', '--trace')
print('--- itemsets ---')
print((project / "output/paper.txt").read_text(encoding="utf-8"))

In [ ]:
# Hai bộ đếm phải cho cùng tập kết quả
run_julia("bin/apriori.jl", "--input", "data/toy/paper.txt", "--output", "output/paper-scan.txt",
          "--minsup", "0.5", "--counter", "scan", "--quiet")
hash_items = (project / "output/paper.txt").read_text(encoding="utf-8").splitlines()
scan_items = (project / "output/paper-scan.txt").read_text(encoding="utf-8").splitlines()
assert set(hash_items) == set(scan_items)
print(f"Scan và Hash-tree khớp {len(hash_items)} tập mục.")

In [ ]:
# Luật kết hợp trên ví dụ nhỏ
run_julia('bin/rules.jl', '--itemsets', 'output/paper.txt', '--data', 'data/toy/paper.txt', '--minconf', '0.6', '--top', '10')

## Dữ liệu Retail

Bộ Retail trong kho SPMF gồm 88.162 giỏ hàng của một cửa hàng bán lẻ ở Bỉ, với 16.470 mã mặt hàng. Mỗi dòng là một giỏ. Các số trên cùng một dòng là mã của những mặt hàng được mua chung.

In [ ]:
print("Dùng file:", DATA)
with DATA.open(encoding="ascii") as stream:
    print("Dòng đầu:", stream.readline().strip())
    print("Số giao dịch:", 1 + sum(1 for _ in stream))
print("SHA-256:", hashlib.sha256(DATA.read_bytes()).hexdigest())

## Tìm tập mục phổ biến

Với `MINSUP = 0.02`, một nhóm mặt hàng phải xuất hiện trong ít nhất 1.764 giỏ để đạt ngưỡng 2%. Apriori đếm từng mặt hàng trước, rồi ghép các mặt hàng đạt ngưỡng thành từng cặp. Nếu một cặp xuất hiện dưới 1.764 lần, mọi nhóm lớn hơn chứa cặp đó cũng sẽ dưới ngưỡng. Nhờ vậy, thuật toán có thể bỏ qua các nhóm này.

Hạ `MINSUP` để xét thêm những nhóm ít gặp hơn. Khi đó số nhóm cần đếm và thời gian chạy có thể tăng.

In [ ]:
MINSUP = 0.02      # support tối thiểu (tỷ lệ)
MINCONF = 0.5      # confidence tối thiểu của luật
ITEMSETS = project / "output/retail.txt"
RULES_CSV = project / "output/retail-rules.csv"

In [ ]:
run_julia('bin/apriori.jl', '--input', DATA, '--output', ITEMSETS,
          '--minsup', str(MINSUP), '--counter', 'hash-tree', '--trace')

In [ ]:
# Số itemset theo kích thước
from collections import Counter
sizes = Counter(len(line.split('#SUP:')[0].split())
                for line in open(ITEMSETS) if '#SUP:' in line)
for k in sorted(sizes): print(f'{k}-itemset: {sizes[k]:,}')
print('Tổng:', f'{sum(sizes.values()):,}')

## Sinh luật kết hợp

Từ mỗi tập mục phổ biến, ta tách các mặt hàng thành hai nhóm X và Y để lập luật `X ⇒ Y`. Với `n(X)` là số giỏ chứa X và `N` là tổng số giỏ, ba chỉ số được tính như sau.

- **Support** = `n(X ∪ Y) / N`, tỷ lệ giỏ chứa cả X và Y.
- **Confidence** = `n(X ∪ Y) / n(X)`, tỷ lệ giỏ có thêm Y trong số các giỏ chứa X.
- **Lift** = `confidence / (n(Y) / N)`, cho biết Y xuất hiện trong các giỏ có X thường xuyên gấp bao nhiêu lần so với toàn bộ giỏ.

Với `MINCONF = 0.5`, chỉ giữ luật có confidence từ 50% trở lên.

In [ ]:
run_julia('bin/rules.jl', '--itemsets', ITEMSETS, '--data', DATA,
          '--minconf', str(MINCONF), '--top', '10', '--out', RULES_CSV)

## 10 luật có lift cao nhất

Ở ngưỡng support 2% và confidence 50%, Apriori tìm được **55 tập mục phổ biến** và từ đó sinh được **45 luật**. Bảng và biểu đồ hiển thị 10 luật đứng đầu. Đường nét đứt đánh dấu lift bằng 1, khi tỷ lệ mua Y trong các giỏ có X bằng tỷ lệ mua Y trên toàn bộ giỏ.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rules = pd.read_csv(RULES_CSV, dtype={'antecedent': str, 'consequent': str})
print(f'Tổng số luật: {len(rules):,}')

# Nếu có file tên mặt hàng (ví dụ bộ Instacart trong SPMF), điền vào đây: {mã: 'tên'}
ITEM_NAMES = {}
def name(i): return ITEM_NAMES.get(int(i), f'#{i}')
def side(s): return '{' + ', '.join(name(i) for i in s.split()) + '}'

top = rules.head(10).copy()
top['luật'] = top['antecedent'].map(side) + ' ⇒ ' + top['consequent'].map(side)
top.index = range(1, len(top) + 1)
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
top[['luật', 'count', 'support', 'confidence', 'lift']].round(4)

In [ ]:
if len(top):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(top['luật'][::-1], top['lift'][::-1])
    ax.axvline(1.0, linestyle='--', linewidth=1)
    ax.set_xlabel('lift (đường nét đứt = 1: độc lập)')
    ax.set_title(f'Top-10 luật theo lift (minsup={MINSUP}, minconf={MINCONF})')
    plt.tight_layout(); plt.show()

## Diễn giải kết quả

Ở ngưỡng mặc định, luật đứng đầu là `{37} ⇒ {39, 40}`. Trong toàn bộ dữ liệu, **11,73%** số giỏ chứa cả hai mặt hàng 39 và 40. Khi chỉ xét 2.936 giỏ có mặt hàng 37, có 1.945 giỏ chứa cả 39 và 40, tương đương **66,25%**.

Đó là lý do luật này có lift **5,65** (66,25% chia cho 11,73%). Còn support **2,21%** cho biết tỷ lệ của 1.945 giỏ chứa đủ cả ba mã trong tổng số 88.162 giỏ.

In [ ]:
for i, r in top.iterrows():
    print(f"{i:>2}. Khách mua {side(r['antecedent'])} thì {r['confidence']:.0%} cũng mua {side(r['consequent'])}"
          f" (gấp {r['lift']:.2f} lần so với mua ngẫu nhiên; {r['support']:.2%} số giỏ chứa cả hai).")

Confidence có thể cao khi Y vốn phổ biến. Để chọn luật gợi ý mua kèm, cần xem thêm lift và số giỏ chứa cả hai vế. Lift lớn hơn 1 cho biết hai vế xuất hiện cùng nhau nhiều hơn mức kỳ vọng nếu mua độc lập. Lift nhỏ hơn 1 cho biết chúng ít đi cùng nhau hơn.

Các luật có thể giúp chọn mặt hàng để thử bán theo gói, đặt gần nhau trên kệ hoặc lên kế hoạch tồn kho. Retail dùng mã ẩn danh nên cần bảng đối chiếu mã hàng trước khi áp dụng cho sản phẩm cụ thể.

## Kiểm tra khả năng tái lập

Chạy lại bước sinh luật và so sánh mã SHA-256 của hai tệp CSV. Sau đó đối chiếu các tập mục và số lần xuất hiện do hai bộ đếm `scan` và `hash-tree` tìm được.

In [ ]:
second_csv = project / "output/retail-rules-rerun.csv"
scan_itemsets = project / "output/retail-scan.txt"
run_julia("bin/rules.jl", "--itemsets", ITEMSETS, "--data", DATA,
          "--minconf", MINCONF, "--top", "0", "--out", second_csv)
assert hashlib.sha256(RULES_CSV.read_bytes()).digest() == hashlib.sha256(second_csv.read_bytes()).digest()
run_julia("bin/apriori.jl", "--input", DATA, "--output", scan_itemsets,
          "--minsup", MINSUP, "--counter", "scan", "--quiet")
assert set(ITEMSETS.read_text().splitlines()) == set(scan_itemsets.read_text().splitlines())
print("Hai lần sinh luật cho cùng CSV. Hai bộ đếm khớp từng tập mục và số lần xuất hiện.")

## Tải bảng luật kết hợp (tùy chọn)

Kết quả được lưu trong `output/retail-rules.csv`.

In [ ]:
try:
    from google.colab import files
    files.download(str(RULES_CSV))
except ImportError:
    print('Không chạy trong Colab, file nằm ở', RULES_CSV)